# Behavioral Economics Decision Intelligence Platform

**An end-to-end machine-learning + behavioural-economics research pipeline built on real,
publicly archived experimental decision-making data.**

---

## What this notebook does

This notebook builds a complete decision-intelligence platform on top of **real published
experimental economics datasets** that are downloaded automatically from public research
repositories (GitHub, the Open Science Framework, and Zenodo). Nothing here is simulated:
every number reported is estimated from archived human choice data.

It has two modelling components, as well as causal, explanatory and optimisation layers:

* **Component A — Behavioural Pattern Prediction Engine.** Predicts *experimentally
  observed choice behaviour* from the decision environment (rewards, delays,
  probabilities, outcome distributions) and from a participant's previously observed
  choices. It also estimates *continuous* behavioural parameters (discount rates,
  present-bias, loss aversion, probability weighting) rather than forcing everything
  into binary labels.
* **Component B — Behavioural Intervention Optimiser.** Estimates
  `P(choice | decision environment, participant history)` and then searches the space of
  *permissible* decision-environment configurations for settings that optimise an
  explicitly defined objective (e.g. the cheapest incentive that reaches a target
  patience rate).

> **Scope and honesty statement.** The models here predict **experimentally observed
> behavioural decision patterns**. They do **not** diagnose psychological conditions, and
> they do **not** classify anybody's personality. Estimated quantities such as a discount
> rate or a loss-aversion coefficient are *observed behavioural tendencies under a
> specific experimental protocol*, not immutable traits. See the Ethics section.

---

## Research questions

1. Can ML predict experimentally observed behavioural decision patterns?
2. Which features of a decision environment are most predictive of choice?
3. **Does ML outperform classical behavioural-economic models** (expected utility,
   prospect theory, hyperbolic / quasi-hyperbolic discounting)?
4. Do participant-level behavioural tendencies improve prediction?
5. Which decision-environment changes produce the largest shifts in predicted choice?
6. Are those effects heterogeneous across participants and problems?
7. Can behavioural interventions be optimised computationally?
8. Does the model generalise to **independent experiments it never saw**?
9. How stable are the estimated behavioural patterns?
10. What are the ethical limits of modelling human decision-making this way?

---

## Datasets (all downloaded automatically; see the Dataset Discovery section)

| Track | Dataset | Phenomenon | Source |
|---|---|---|---|
| **RISK** | `choices13k` (Peterson et al., *Science* 2021) | risk preference, loss aversion, probability weighting, description–experience gap | GitHub |
| **ITC** | ITC Database (Pongratz & Schoemann, *Sci Data* 2026) | intertemporal choice, delay discounting, present bias | OSF |
| **RISK-IND** | CPC18 calibration set (Plonsky, Erev & Ert) | individual-level risky choice with feedback | Zenodo |

The pipeline is *degradation-tolerant*: each dataset is fetched with retries and
fallbacks, and if a source is genuinely unreachable the affected track is skipped with an
explicit report rather than being silently replaced by fabricated data.

---

## Methods

* **Behavioural economics** — expected value, CRRA/power expected utility, Cumulative
  Prospect Theory with rank-dependent probability weighting, exponential discounting,
  Mazur hyperbolic discounting, quasi-hyperbolic (β–δ) discounting. All fitted by
  maximum likelihood.
* **Machine learning** — logistic/linear baselines, random forests, gradient boosting,
  HistGradientBoosting, XGBoost, LightGBM, MLP; Optuna hyper-parameter search on a
  held-out validation split.
* **Validation** — participant-grouped and problem-grouped splitting,
  `StratifiedGroupKFold`, and **leave-one-study-out** cross-dataset generalisation.
* **Causal inference** — paired within-problem randomised contrasts, ATE with bootstrap
  CIs, IPW / doubly-robust estimation, and causal-forest style CATE.
* **Explainability** — SHAP global and local attributions, model-predicted
  counterfactuals (explicitly distinguished from causal effects).
* **Optimisation** — constrained grid, random and Bayesian (Optuna) search over the
  decision environment.

## How to run

Upload to Google Colab and run **Runtime ▸ Run all**. Everything (installs, downloads,
caching, training, reporting, the Gradio app) is automatic. Total runtime is roughly
20–40 minutes on a free CPU instance. Set `CONFIG.quick_mode = True` for a ~5 minute pass.
Google Drive mounting is optional — the notebook works without it.

By default the intertemporal track is capped at 400,000 trials, subsampled **by
participant** so nobody is partially observed. Set `CONFIG.max_itc_rows = None` to
train on all ~1.17 million trials (noticeably slower).

## 3. Behavioural economics background

Three families of empirical regularity motivate this project. Each one is *estimated from
data* later in the notebook rather than assumed.

**Intertemporal choice and present bias.** Choosing between a smaller-sooner (SS) and a
larger-later (LL) reward reveals a discount function. Exponential discounting,
$V = A e^{-kt}$, implies time-consistent preferences. Human choices are usually better
described by Mazur's hyperbolic form $V = A/(1+kt)$, or by the quasi-hyperbolic (β–δ)
form $V = A\beta^{\mathbb{1}[t>0]}e^{-kt}$, where $\beta < 1$ is a discrete extra penalty
on *any* delay. That $\beta$ is the cleanest operational definition of **present bias**,
and we estimate it per participant.

**Choice under risk.** Expected utility theory values a prospect as
$\sum_i p_i u(x_i)$. Cumulative Prospect Theory (Tversky & Kahneman, 1992) instead uses a
reference-dependent value function
$v(x) = x^{\alpha}$ for gains and $-\lambda(-x)^{\beta}$ for losses, with $\lambda>1$
encoding **loss aversion**, together with rank-dependent probability weighting
$w(p) = p^{\gamma}/(p^{\gamma}+(1-p)^{\gamma})^{1/\gamma}$, which overweights small
probabilities when $\gamma<1$. We fit $\alpha,\beta,\lambda,\gamma$ directly.

**The description–experience gap.** When people *read* a described gamble they behave as
if rare events are overweighted; when they *experience* the same gamble through repeated
sampling with feedback they behave as if rare events are underweighted. In `choices13k`
the same problem appears both with and without feedback, which gives a genuine
**paired, within-problem experimental contrast** — the causal backbone of this project.

Together these give the behavioural baselines that the ML models must beat.

## 4. Environment setup

Installs only what is missing, so re-runs are fast.

In [ ]:
# --- 4.1 Package installation -------------------------------------------------------
import importlib, subprocess, sys, os, warnings

REQUIRED = {
    "numpy": "numpy", "pandas": "pandas", "scipy": "scipy",
    "sklearn": "scikit-learn", "matplotlib": "matplotlib", "seaborn": "seaborn",
    "requests": "requests", "statsmodels": "statsmodels", "joblib": "joblib",
    "shap": "shap", "xgboost": "xgboost", "optuna": "optuna",
}
OPTIONAL = {"lightgbm": "lightgbm", "gradio": "gradio", "pyarrow": "pyarrow"}

def ensure(mapping, optional=False):
    missing = []
    for mod, pkg in mapping.items():
        try:
            importlib.import_module(mod)
        except ImportError:
            missing.append(pkg)
    if missing:
        print(("optional: " if optional else "") + "installing " + ", ".join(missing))
        cmd = [sys.executable, "-m", "pip", "install", "-q"] + missing
        rc = subprocess.call(cmd)
        if rc != 0 and not optional:
            print(f"!! pip returned {rc}; some required packages may be unavailable")
    else:
        print(("optional: " if optional else "") + "all present")

ensure(REQUIRED)
ensure(OPTIONAL, optional=True)
warnings.filterwarnings("ignore")
print("environment ready")

In [ ]:
# --- 4.2 Imports --------------------------------------------------------------------
from __future__ import annotations

import dataclasses, hashlib, json, logging, platform, random, shutil, textwrap, time, zipfile, io
from dataclasses import dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Callable, Iterable, Sequence

import numpy as np
import pandas as pd
import requests
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.optimize import minimize
from scipy.special import expit, logit

from sklearn.base import clone
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.ensemble import (GradientBoostingClassifier, GradientBoostingRegressor,
                              HistGradientBoostingClassifier, HistGradientBoostingRegressor,
                              RandomForestClassifier, RandomForestRegressor)
from sklearn.linear_model import (LinearRegression, LogisticRegression, Ridge, Lasso)
from sklearn.metrics import (accuracy_score, average_precision_score, brier_score_loss,
                             confusion_matrix, f1_score, log_loss, mean_absolute_error,
                             mean_squared_error, precision_score, r2_score, recall_score,
                             roc_auc_score, roc_curve, precision_recall_curve)
from sklearn.model_selection import (GroupKFold, StratifiedGroupKFold, GroupShuffleSplit)
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.calibration import calibration_curve

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def _try(name):
    try:
        return importlib.import_module(name)
    except Exception:
        return None

xgb = _try("xgboost")
lgb = _try("lightgbm")
shap = _try("shap")
sm = _try("statsmodels.api")

sns.set_theme(style="whitegrid", context="notebook")
matplotlib.rcParams["figure.dpi"] = 110
matplotlib.rcParams["savefig.dpi"] = 150
matplotlib.rcParams["figure.autolayout"] = True
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 60)

print(f"numpy {np.__version__} | pandas {pd.__version__} | "
      f"xgboost {'-' if xgb is None else xgb.__version__} | "
      f"lightgbm {'-' if lgb is None else lgb.__version__} | "
      f"shap {'-' if shap is None else shap.__version__}")

## 5. Configuration

All tunable behaviour lives in one dataclass. Seeds are fixed here and reused everywhere,
so a re-run reproduces the same splits, the same fits and the same reported numbers.

In [ ]:
# --- 5.1 Global configuration -------------------------------------------------------
@dataclass
class Config:
    random_seed: int = 42
    test_size: float = 0.15
    val_size: float = 0.15

    project_name: str = "behavioral_economics_project"
    use_google_drive: bool = True       # optional; falls back silently to local disk
    local_root: str = "/content/behavioral_economics_project"
    drive_root: str = "/content/drive/MyDrive/behavioral_economics_project"

    # --- dataset toggles (all sources are public; see the Dataset Discovery section)
    enable_choices13k: bool = True      # GitHub  – primary risky-choice track
    enable_itc_database: bool = True    # OSF     – intertemporal-choice track
    enable_cpc18: bool = True           # Zenodo  – individual-level risky choice (47 MB)

    # --- compute budget
    quick_mode: bool = False            # True => small trial counts, fast pass
    optuna_trials: int = 30
    cpt_restarts: int = 3
    n_bootstrap: int = 2000
    shap_sample: int = 2000
    # Participant-level cap on the ITC track. The full database is ~1.17M trials, which
    # pushes a free Colab CPU run past an hour. Set to None to use every trial.
    max_itc_rows: int | None = 400_000

    # --- networking
    http_timeout: int = 120
    http_retries: int = 4
    http_backoff: float = 2.0

    launch_gradio: bool = True

    def __post_init__(self):
        if self.quick_mode:
            self.optuna_trials = 8
            self.cpt_restarts = 1
            self.n_bootstrap = 400
            self.shap_sample = 500
            self.max_itc_rows = 120_000

CONFIG = Config()

RANDOM_SEED = CONFIG.random_seed
TEST_SIZE, VAL_SIZE = CONFIG.test_size, CONFIG.val_size

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)
RNG = np.random.default_rng(RANDOM_SEED)

print(json.dumps({k: str(v) for k, v in dataclasses.asdict(CONFIG).items()}, indent=2))

In [ ]:
# --- 5.2 Directories, optional Drive mount, logging ---------------------------------
IN_COLAB = "google.colab" in sys.modules

def _mount_drive() -> Path | None:
    if not (IN_COLAB and CONFIG.use_google_drive):
        return None
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        root = Path(CONFIG.drive_root)
        root.mkdir(parents=True, exist_ok=True)
        return root
    except Exception as exc:                                   # user declined, no Drive, ...
        print(f"Google Drive not mounted ({type(exc).__name__}); using local storage only.")
        return None

DRIVE_ROOT = _mount_drive()
ROOT = Path(CONFIG.local_root)
SUBDIRS = ["data/raw", "data/interim", "data/processed", "models", "outputs",
           "figures", "reports", "logs", "cache"]
for d in SUBDIRS:
    (ROOT / d).mkdir(parents=True, exist_ok=True)
if DRIVE_ROOT is not None:
    for d in SUBDIRS:
        (DRIVE_ROOT / d).mkdir(parents=True, exist_ok=True)

PATHS = {d.split("/")[-1] if "/" not in d else d.replace("/", "_"): ROOT / d for d in SUBDIRS}
RAW, INTERIM, PROCESSED = ROOT / "data/raw", ROOT / "data/interim", ROOT / "data/processed"
MODELS, OUTPUTS, FIGURES = ROOT / "models", ROOT / "outputs", ROOT / "figures"
REPORTS, LOGS, CACHE = ROOT / "reports", ROOT / "logs", ROOT / "cache"

# Drive acts as a persistent cache mirror for raw downloads only.
DRIVE_CACHE = (DRIVE_ROOT / "data/raw") if DRIVE_ROOT is not None else None

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-7s | %(message)s",
    datefmt="%H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout),
              logging.FileHandler(LOGS / "pipeline.log", mode="w")],
    force=True,
)
LOG = logging.getLogger("bedip")
LOG.info("root=%s  drive_cache=%s  colab=%s", ROOT, DRIVE_CACHE, IN_COLAB)

def savefig(name: str, fig=None, tight=True):
    fig = fig or plt.gcf()
    if tight:
        try: fig.tight_layout()
        except Exception: pass
    p = FIGURES / f"{name}.png"
    fig.savefig(p, bbox_inches="tight")
    return p

ENV_INFO = {
    "python": platform.python_version(),
    "platform": platform.platform(),
    "numpy": np.__version__, "pandas": pd.__version__,
    "scikit_learn": importlib.import_module("sklearn").__version__,
    "xgboost": getattr(xgb, "__version__", None),
    "lightgbm": getattr(lgb, "__version__", None),
    "shap": getattr(shap, "__version__", None),
    "optuna": optuna.__version__,
    "random_seed": RANDOM_SEED,
    "run_started_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
}
print(json.dumps(ENV_INFO, indent=2))

## 6. Dataset discovery

**This section was completed *before* any modelling.** Public repositories were searched
for experimental decision-making data covering intertemporal choice, risk and loss
aversion, and consumer / choice-architecture manipulations, with a hard requirement that
the data be **programmatically downloadable without credentials**.

The catalogue below records what was found — including the candidates that were
**rejected**, and why. Every URL is a real archived source; none are invented. The
`programmatic` column is the decisive one: several otherwise excellent datasets are
excluded purely because they cannot be ingested without a manual click-through or an
account, which would violate the automation requirement.

A note on what was deliberately *not* forced into the pipeline: there is no
well-archived, credential-free, row-level **retail pricing / anchoring** experiment that
meets the same bar. Rather than substitute a demographic survey with no decision data —
or worse, fabricate price data — the pricing module is implemented as a **behavioural
intervention optimiser** over decision variables that genuinely exist in the selected
data (reward magnitudes, delays, outcome distributions, feedback). This is stated again
in Limitations.

In [ ]:
# --- 6.1 Candidate dataset catalogue (researched, not assumed) ----------------------
CANDIDATES = [
    dict(
        dataset_id="choices13k",
        name="choices13k — human decisions on 13,006 risky choice problems",
        repository="GitHub (Princeton Computational Cognitive Science Lab)",
        url="https://github.com/jcpeterson/choices13k",
        authors="Peterson, Bourgin, Agrawal, Reichman & Griffiths",
        study="Peterson et al. (2021), Science 372(6547):1209-1214, doi:10.1126/science.abe2629",
        phenomenon="risk preference; loss aversion; probability weighting; description-experience gap",
        n_observations=14568, n_participants=14711,
        key_variables="Ha,pHa,La,Hb,pHb,Lb,LotShapeB,LotNumB,Amb,Corr,Feedback,Block,n,bRate",
        target="bRate = proportion of participants choosing option B",
        design="between-problem; each problem seen with and without outcome feedback",
        treatment_control="yes — Feedback (description vs experience), paired within problem",
        repeated_decisions="yes (aggregated to problem level)",
        participant_ids="no (aggregate choice rates per problem)",
        license="see repository; academic release accompanying a Science paper",
        download="direct raw file download from GitHub",
        api="GitHub raw endpoints",
        programmatic=True, ml_suitable=True, causal_suitable=True,
        limitations="aggregate rates only, so no individual-level profiling; MTurk sample; "
                    "known to carry extra decision noise vs lab data (Thomas et al. 2024)",
        include=True,
        rationale="Largest archived risky-choice dataset; exact outcome distributions are "
                  "published, enabling a genuine Cumulative Prospect Theory fit rather than "
                  "an approximation. Fully automatable. PRIMARY track.",
    ),
    dict(
        dataset_id="itc_database",
        name="ITC Database — choice and response-time data in intertemporal choice",
        repository="Open Science Framework",
        url="https://osf.io/3wsae/",
        authors="Pongratz & Schoemann",
        study="Pongratz & Schoemann (2026), Scientific Data 13:323, doi:10.1038/s41597-026-06947-4",
        phenomenon="intertemporal choice; delay discounting; present bias",
        n_observations=1_172_644, n_participants=11_852,
        key_variables="paper,subj_ident,sess_ident,trial_idx,ss_value,ss_time,ll_value,"
                      "ll_time,choice,rt,age,country,currency,procedure,incentivization,"
                      "presentation_of_information,time_pressure,online_study",
        target="choice (0 = smaller-sooner, 1 = larger-later)",
        design="100 archived studies pooled into one schema; trial-level binary choice",
        treatment_control="partial — study-level manipulations only (confounded with study)",
        repeated_decisions="yes — many trials per participant",
        participant_ids="yes — subj_ident unique across the whole database",
        license="CC BY-NC-SA 4.0",
        download="OSF REST API file listing, then direct file download",
        api="https://api.osf.io/v2/",
        programmatic=True, ml_suitable=True, causal_suitable=False,
        limitations="non-commercial licence; heterogeneous protocols across studies; "
                    "study-level moderators are observational, not randomised",
        include=True,
        rationale="Provides participant IDs and a study identifier, which is exactly what is "
                  "needed for grouped (leakage-free) splitting, per-participant behavioural "
                  "parameter estimation, and leave-one-study-out generalisation. PRIMARY "
                  "intertemporal track.",
    ),
    dict(
        dataset_id="cpc18",
        name="CPC18 calibration set — raw individual choices between gambles",
        repository="Zenodo",
        url="https://zenodo.org/records/845873",
        authors="Plonsky, Erev & Ert",
        study="Erev et al. (2017), Psychological Review 124(4):369-409, doi:10.1037/rev0000062",
        phenomenon="risk; ambiguity; decisions from experience; description-experience gap",
        n_observations=510_750, n_participants=686,
        key_variables="individual trial-level choices across 210 problems, 25 trials each, "
                      "no-feedback block then feedback blocks (schema resolved at runtime)",
        target="binary choice of option B",
        design="within-subject; block 1 without feedback, blocks 2-5 with feedback",
        treatment_control="yes — feedback manipulation within subject",
        repeated_decisions="yes", participant_ids="yes",
        license="CC BY 4.0",
        download="stable Zenodo file URL (47.5 MB CSV)",
        api="Zenodo REST API",
        programmatic=True, ml_suitable=True, causal_suitable=True,
        limitations="47 MB download; column naming is resolved by alias matching at runtime; "
                    "student sample recruited in Israel",
        include=True,
        rationale="An independent risky-choice experiment with a different population and "
                  "protocol from choices13k — used for genuine CROSS-DATASET generalisation "
                  "testing rather than for headline model fitting.",
    ),
    # ---------------- evaluated and NOT included ------------------------------------
    dict(
        dataset_id="cpc18_full", name="CPC18 full raw data (calibration + competition)",
        repository="Zenodo", url="https://zenodo.org/records/2571510",
        authors="Plonsky, Erev & Ert", study="Plonsky et al. (2019)",
        phenomenon="risk, ambiguity, experience", n_observations=694_500, n_participants=926,
        key_variables="as CPC18", target="binary choice", design="within-subject",
        treatment_control="yes", repeated_decisions="yes", participant_ids="yes",
        license="CC BY 4.0", download="Zenodo", api="Zenodo REST API",
        programmatic=True, ml_suitable=True, causal_suitable=True,
        limitations="superset of the calibration release; redundant once CPC18 is included",
        include=False,
        rationale="EXCLUDED: near-duplicate of the calibration set already selected; adding "
                  "it would leak overlapping problems across the cross-dataset test.",
    ),
    dict(
        dataset_id="cpc15", name="CPC15 raw data", repository="Zenodo",
        url="https://zenodo.org/records/321652", authors="Erev, Ert, Plonsky, Cohen & Cohen",
        study="Erev et al. (2017), Psychological Review", phenomenon="risky choice",
        n_observations=None, n_participants=None, key_variables="choices between gambles",
        target="binary choice", design="within-subject", treatment_control="yes",
        repeated_decisions="yes", participant_ids="yes", license="CC BY 4.0",
        download="Zenodo", api="Zenodo REST API",
        programmatic=True, ml_suitable=True, causal_suitable=True,
        limitations="fully contained within the CPC18 calibration release",
        include=False,
        rationale="EXCLUDED: subset of CPC18; including both would double-count the same "
                  "participants and inflate apparent generalisation.",
    ),
    dict(
        dataset_id="hurd_c13k", name="HURD risky-choice model library + bundled datasets",
        repository="GitHub", url="https://github.com/jcpeterson/hurd",
        authors="Peterson & Bourgin", study="Peterson et al. (2021), Science",
        phenomenon="risky choice", n_observations=None, n_participants=None,
        key_variables="copies of choices13k / CPC data", target="choice rate",
        design="n/a", treatment_control="n/a", repeated_decisions="n/a",
        participant_ids="no", license="see repository", download="git clone",
        api="GitHub", programmatic=True, ml_suitable=True, causal_suitable=False,
        limitations="redistribution of data already selected; adds a jax dependency",
        include=False,
        rationale="EXCLUDED: it is a modelling library that re-ships choices13k. We fetch "
                  "the canonical dataset directly and implement the theory ourselves.",
    ),
    dict(
        dataset_id="itc_amasino", name="Amount and time exert independent influences on ITC",
        repository="OSF", url="https://osf.io/2p3dj/", authors="Amasino, Sullivan, Kranton & Huettel",
        study="Amasino et al. (2019), Nature Human Behaviour 3:383-392",
        phenomenon="intertemporal choice", n_observations=13_793, n_participants=98,
        key_variables="ss_value, delay, choice, response time", target="binary choice",
        design="within-subject", treatment_control="no", repeated_decisions="yes",
        participant_ids="yes", license="see repository", download="OSF API",
        programmatic=True, ml_suitable=True, causal_suitable=False,
        limitations="single study, 98 participants",
        include=False,
        rationale="EXCLUDED as a separate source: it is already one of the 100 studies "
                  "pooled inside the ITC Database, which we ingest as a whole.",
    ),
    dict(
        dataset_id="zilker_pachur", name="Option complexity, framing, loss aversion and discounting",
        repository="OSF", url="https://osf.io/bj2tv/", authors="Zilker & Pachur",
        study="Zilker & Pachur (2020)", phenomenon="framing; loss aversion; discounting; ageing",
        n_observations=None, n_participants=None,
        key_variables="framing condition, gamble parameters, choice", target="binary choice",
        design="within-subject framing manipulation", treatment_control="yes",
        repeated_decisions="yes", participant_ids="yes", license="see repository",
        download="OSF API", programmatic=True, ml_suitable=True, causal_suitable=True,
        limitations="modest N; component tasks are already partially represented in the ITC Database",
        include=False,
        rationale="EXCLUDED from the modelling pipeline but noted as the closest available "
                  "framing-effect source; its intertemporal component is already inside the "
                  "ITC Database, so including it separately would duplicate participants.",
    ),
    dict(
        dataset_id="many_labs_1", name="Many Labs 1 replication project (anchoring, framing)",
        repository="OSF", url="https://osf.io/wx7ck/", authors="Klein et al.",
        study="Klein et al. (2014), Social Psychology 45(3):142-152",
        phenomenon="anchoring; risky-choice framing (Asian disease)",
        n_observations=None, n_participants=6344,
        key_variables="anchoring items, framing condition, one response per item",
        target="varies by item", design="randomised between-subject",
        treatment_control="yes", repeated_decisions="no (a handful of items per person)",
        participant_ids="yes", license="CC0 / open", download="OSF API",
        programmatic=True, ml_suitable=False, causal_suitable=True,
        limitations="very few decisions per participant; heterogeneous item formats; "
                    "no continuous decision-environment variables to optimise over",
        include=False,
        rationale="EXCLUDED: excellent for a pure anchoring/framing ATE, but it has no "
                  "repeated trial structure and no continuous intervention variables, so it "
                  "cannot support choice modelling, behavioural parameter estimation or the "
                  "optimiser. Forcing it in would be padding.",
    ),
    dict(
        dataset_id="kaggle_generic_consumer", name="Generic 'customer personality / marketing' CSVs",
        repository="Kaggle", url="https://www.kaggle.com/datasets", authors="various (often unattributed)",
        study="none", phenomenon="none (demographics + purchase totals)",
        n_observations=None, n_participants=None, key_variables="demographics, spend totals",
        target="n/a", design="observational, no experiment", treatment_control="no",
        repeated_decisions="no", participant_ids="sometimes", license="varies / unclear",
        download="Kaggle API (requires credentials)", api="requires an API token",
        programmatic=False, ml_suitable=False, causal_suitable=False,
        limitations="no experimental manipulation, no decision environment, unclear provenance",
        include=False,
        rationale="EXCLUDED on the brief's own criterion: generic demographic data with no "
                  "meaningful behavioural decision content, and it needs credentials.",
    ),
    dict(
        dataset_id="icpsr_behavioral", name="ICPSR behavioural-economics study archive",
        repository="ICPSR", url="https://www.icpsr.umich.edu/web/ICPSR/search/studies",
        authors="various", study="various", phenomenon="various",
        n_observations=None, n_participants=None, key_variables="varies",
        target="varies", design="varies", treatment_control="varies",
        repeated_decisions="varies", participant_ids="varies",
        license="restricted; institutional login", download="manual after authentication",
        api="no anonymous data API", programmatic=False, ml_suitable=True,
        causal_suitable=True, limitations="requires an authenticated account per study",
        include=False,
        rationale="EXCLUDED: cannot be ingested without credentials, which would break the "
                  "no-manual-download requirement. Listed for completeness.",
    ),
    dict(
        dataset_id="world_bank_findex", name="World Bank Global Findex",
        repository="World Bank", url="https://www.worldbank.org/en/publication/globalfindex",
        authors="World Bank", study="Global Findex Database", phenomenon="financial behaviour (survey)",
        n_observations=None, n_participants=None, key_variables="survey items on saving/borrowing",
        target="n/a", design="survey, not an experiment", treatment_control="no",
        repeated_decisions="no", participant_ids="yes", license="CC BY 4.0",
        download="direct file download", api="World Bank API",
        programmatic=True, ml_suitable=False, causal_suitable=False,
        limitations="self-reported survey responses, no incentivised choice task",
        include=False,
        rationale="EXCLUDED: survey attitudes are not experimentally observed choices; there "
                  "is no decision environment to model or optimise.",
    ),
]

CATALOG = pd.DataFrame(CANDIDATES)
CATALOG.to_csv(OUTPUTS / "dataset_catalog.csv", index=False)
print(f"{len(CATALOG)} candidates catalogued; {int(CATALOG.include.sum())} selected for the pipeline\n")
display(CATALOG.loc[:, ["dataset_id", "repository", "phenomenon", "participant_ids",
                        "programmatic", "ml_suitable", "causal_suitable", "include"]])

### 7. Final dataset selection and rationale

Three datasets are selected. They deliberately span two of the three phenomenon groups
requested in the brief, and they are **not** merged at the row level, because they measure
different decision problems with incompatible units. Instead each is modelled as its own
**track** under a shared behavioural schema, which is what makes the cross-dataset
generalisation test meaningful.

| Group | Track | Dataset | Why it earns its place |
|---|---|---|---|
| A — intertemporal / present bias | `ITC` | ITC Database | The only selected source with participant IDs *and* a study identifier, so it carries grouped splitting, per-participant β–δ estimation and leave-one-study-out validation. |
| B — risk / loss aversion | `RISK` | choices13k | Exact published outcome distributions make a real CPT fit possible; large enough for serious ML. |
| B′ — risk, independent replication | `RISK_IND` | CPC18 | A different lab, population and protocol — the out-of-domain test for the risk models. |

Group C (consumer pricing / anchoring) has **no** source meeting the automation bar, so
rather than inventing price data the optimiser operates on the real intervention variables
that do exist. This is a deliberate, documented substitution.

## 8. Automated data ingestion

A small ingestion layer handles: retries with exponential backoff, redirects, content
validation, checksum-based caching, optional mirroring to Google Drive, archive
extraction, and per-source fallbacks. A source that cannot be reached is **skipped with a
loud message** — never silently replaced.

In [ ]:
# --- 8.1 Download / cache layer -----------------------------------------------------
INGEST_LOG: list[dict] = []

class DownloadError(RuntimeError):
    pass

def _sha256(path: Path, limit: int | None = None) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while chunk := f.read(1 << 20):
            h.update(chunk)
            if limit and h.block_size and f.tell() > limit:
                break
    return h.hexdigest()

def http_get(url: str, *, stream: bool = True, timeout: int | None = None,
             headers: dict | None = None) -> requests.Response:
    """GET with retries, exponential backoff and redirect following."""
    timeout = timeout or CONFIG.http_timeout
    hdrs = {"User-Agent": "behavioral-econ-ml-pipeline/1.0 (research; contact via notebook)"}
    hdrs.update(headers or {})
    last = None
    for attempt in range(1, CONFIG.http_retries + 1):
        try:
            r = requests.get(url, stream=stream, timeout=timeout,
                             allow_redirects=True, headers=hdrs)
            if r.status_code == 200:
                return r
            last = DownloadError(f"HTTP {r.status_code} for {url}")
            if r.status_code in (400, 401, 403, 404):    # not worth retrying
                break
        except requests.RequestException as exc:
            last = exc
        if attempt < CONFIG.http_retries:
            wait = CONFIG.http_backoff ** attempt
            LOG.warning("attempt %d/%d failed (%s); retrying in %.0fs",
                        attempt, CONFIG.http_retries, last, wait)
            time.sleep(wait)
    raise DownloadError(f"could not fetch {url}: {last}")

def fetch(url: str, dest_name: str, *, min_bytes: int = 512,
          expect_text: bool = False) -> Path:
    """Download to data/raw with caching. Checks Drive mirror first, then the network."""
    dest = RAW / dest_name
    if dest.exists() and dest.stat().st_size >= min_bytes:
        LOG.info("cache hit  %s (%.1f MB)", dest_name, dest.stat().st_size / 1e6)
        return dest
    if DRIVE_CACHE is not None and (DRIVE_CACHE / dest_name).exists():
        src = DRIVE_CACHE / dest_name
        if src.stat().st_size >= min_bytes:
            shutil.copy2(src, dest)
            LOG.info("drive cache hit  %s (%.1f MB)", dest_name, dest.stat().st_size / 1e6)
            return dest

    LOG.info("downloading %s", url)
    t0 = time.time()
    r = http_get(url)
    tmp = dest.with_suffix(dest.suffix + ".part")
    total = 0
    with open(tmp, "wb") as f:
        for chunk in r.iter_content(1 << 20):
            if chunk:
                f.write(chunk); total += len(chunk)
    if total < min_bytes:
        tmp.unlink(missing_ok=True)
        raise DownloadError(f"{url} returned only {total} bytes")
    if expect_text:
        head = tmp.open("rb").read(400).lstrip()
        if head[:1] in (b"<",) and b"<html" in head.lower():
            tmp.unlink(missing_ok=True)
            raise DownloadError(f"{url} returned an HTML page, not data")
    tmp.replace(dest)
    INGEST_LOG.append(dict(url=url, file=dest_name, bytes=total,
                           seconds=round(time.time() - t0, 1),
                           retrieved_utc=datetime.now(timezone.utc).isoformat(timespec="seconds"),
                           sha256_prefix=_sha256(dest)[:16]))
    LOG.info("saved %s (%.1f MB in %.1fs)", dest_name, total / 1e6, time.time() - t0)
    if DRIVE_CACHE is not None:
        try:
            shutil.copy2(dest, DRIVE_CACHE / dest_name)
        except Exception as exc:
            LOG.warning("could not mirror to Drive: %s", exc)
    return dest

def fetch_with_fallbacks(sources: Sequence[tuple[str, str]], dest_name: str, **kw) -> Path:
    """sources = [(label, url), ...] tried in order."""
    errors = []
    for label, url in sources:
        try:
            return fetch(url, dest_name, **kw)
        except Exception as exc:
            LOG.warning("source '%s' failed: %s", label, exc)
            errors.append(f"{label}: {exc}")
    raise DownloadError("all sources failed -> " + " | ".join(errors))

def extract_archive(path: Path, into: Path) -> list[Path]:
    into.mkdir(parents=True, exist_ok=True)
    if zipfile.is_zipfile(path):
        with zipfile.ZipFile(path) as z:
            z.extractall(into)
        return sorted(p for p in into.rglob("*") if p.is_file())
    return [path]

print("ingestion layer ready")

In [ ]:
# --- 8.2 Loader: choices13k (GitHub) ------------------------------------------------
C13K_SELECTIONS = [
    ("github raw (master)",
     "https://raw.githubusercontent.com/jcpeterson/choices13k/master/c13k_selections.csv"),
    ("github raw (main)",
     "https://raw.githubusercontent.com/jcpeterson/choices13k/main/c13k_selections.csv"),
]
C13K_PROBLEMS = [
    ("github raw (master)",
     "https://raw.githubusercontent.com/jcpeterson/choices13k/master/c13k_problems.json"),
    ("github raw (main)",
     "https://raw.githubusercontent.com/jcpeterson/choices13k/main/c13k_problems.json"),
]

def load_choices13k() -> dict | None:
    """Returns {'selections': DataFrame, 'problems': dict} or None if unavailable.

    `c13k_problems.json` is keyed by the ROW INDEX of c13k_selections.csv (0..N-1), not by
    the `Problem` column. That alignment is verified below rather than assumed.
    """
    if not CONFIG.enable_choices13k:
        return None
    try:
        sel_p = fetch_with_fallbacks(C13K_SELECTIONS, "c13k_selections.csv",
                                     min_bytes=100_000, expect_text=True)
        prob_p = fetch_with_fallbacks(C13K_PROBLEMS, "c13k_problems.json",
                                      min_bytes=100_000, expect_text=True)
    except Exception as exc:
        LOG.error("choices13k unavailable: %s", exc)
        return None

    sel = pd.read_csv(sel_p)
    problems = json.loads(prob_p.read_text())

    required = {"Problem", "Feedback", "n", "Block", "Ha", "pHa", "La",
                "Hb", "pHb", "Lb", "LotShapeB", "LotNumB", "Amb", "Corr", "bRate"}
    missing = required - set(sel.columns)
    if missing:
        LOG.error("choices13k schema changed; missing %s", missing)
        return None

    # --- verify the JSON <-> CSV row alignment on a deterministic sample
    checked = bad = 0
    for i in range(0, len(sel), max(1, len(sel) // 60)):
        p = problems.get(str(i))
        if p is None:
            bad += 1; continue
        outs_a = [v for _, v in p["A"]]
        row = sel.iloc[i]
        checked += 1
        if abs(max(outs_a) - row["Ha"]) > 1e-6 or abs(min(outs_a) - row["La"]) > 1e-6:
            bad += 1
    LOG.info("choices13k alignment check: %d sampled, %d mismatches", checked, bad)
    if bad:
        LOG.error("row alignment failed; refusing to use exact distributions")
        return None
    return {"selections": sel, "problems": problems}

RAW_C13K = load_choices13k()
print("choices13k:", "OK" if RAW_C13K else "UNAVAILABLE",
      f"({len(RAW_C13K['selections'])} rows)" if RAW_C13K else "")

In [ ]:
# --- 8.3 Loader: ITC Database (Open Science Framework) ------------------------------
OSF_NODE = "3wsae"
OSF_API = f"https://api.osf.io/v2/nodes/{OSF_NODE}/files/osfstorage/"
OSF_ZIP = f"https://files.osf.io/v1/resources/{OSF_NODE}/providers/osfstorage/?zip="

def _osf_list_files(node_url: str, depth: int = 0) -> list[dict]:
    """Walk an OSF storage node via the public v2 API (handles pagination + folders)."""
    if depth > 2:
        return []
    out, url = [], node_url
    while url:
        r = http_get(url, stream=False)
        payload = r.json()
        for item in payload.get("data", []):
            attrs = item.get("attributes", {})
            if attrs.get("kind") == "file":
                out.append({
                    "name": attrs.get("name", ""),
                    "size": attrs.get("size") or 0,
                    "download": (item.get("links") or {}).get("download"),
                })
            elif attrs.get("kind") == "folder":
                sub = ((item.get("relationships", {}).get("files", {}) or {})
                       .get("links", {}).get("related", {}) or {}).get("href")
                if sub:
                    out.extend(_osf_list_files(sub, depth + 1))
        url = (payload.get("links") or {}).get("next")
    return out

def load_itc_database() -> pd.DataFrame | None:
    """Download the ITC Database CSV from OSF. Strategy: API listing -> node zip."""
    if not CONFIG.enable_itc_database:
        return None
    cached = RAW / "itc_database.csv"
    if cached.exists() and cached.stat().st_size > 1_000_000:
        LOG.info("ITC cache hit")
        return pd.read_csv(cached, low_memory=False)

    csv_path = None
    # --- strategy 1: the documented OSF REST API
    try:
        files = _osf_list_files(OSF_API)
        LOG.info("OSF listing returned %d files", len(files))
        csvs = [f for f in files if f["name"].lower().endswith(".csv") and f["download"]]
        if csvs:
            best = max(csvs, key=lambda f: f["size"])
            LOG.info("selected %s (%.1f MB)", best["name"], (best["size"] or 0) / 1e6)
            csv_path = fetch(best["download"], "itc_database.csv",
                             min_bytes=1_000_000, expect_text=True)
    except Exception as exc:
        LOG.warning("OSF API strategy failed: %s", exc)

    # --- strategy 2: whole-node zip
    if csv_path is None:
        try:
            z = fetch(OSF_ZIP, "itc_osf_node.zip", min_bytes=1_000_000)
            members = extract_archive(z, INTERIM / "itc_osf")
            cands = [p for p in members if p.suffix.lower() == ".csv" and p.stat().st_size > 1_000_000]
            if cands:
                best = max(cands, key=lambda p: p.stat().st_size)
                shutil.copy2(best, cached)
                csv_path = cached
        except Exception as exc:
            LOG.warning("OSF zip strategy failed: %s", exc)

    if csv_path is None:
        LOG.error("ITC Database unavailable — the intertemporal track will be skipped. "
                  "Data home: https://osf.io/3wsae/")
        return None

    df = pd.read_csv(csv_path, low_memory=False)
    needed = {"subj_ident", "ss_value", "ss_time", "ll_value", "ll_time", "choice"}
    if not needed.issubset(df.columns):
        LOG.error("ITC schema mismatch; found columns: %s", list(df.columns)[:40])
        return None
    LOG.info("ITC Database loaded: %s rows x %s cols", f"{len(df):,}", df.shape[1])
    return df

RAW_ITC = load_itc_database()
print("ITC Database:", "OK" if RAW_ITC is not None else "UNAVAILABLE",
      f"({len(RAW_ITC):,} rows)" if RAW_ITC is not None else "")

In [ ]:
# --- 8.4 Loader: CPC18 calibration set (Zenodo) -------------------------------------
CPC18_SOURCES = [
    ("zenodo record 845873",
     "https://zenodo.org/records/845873/files/All%20estimation%20raw%20data.csv?download=1"),
    ("zenodo legacy path",
     "https://zenodo.org/record/845873/files/All%20estimation%20raw%20data.csv?download=1"),
]

# Column names in the archive are resolved by alias rather than assumed, so a renamed
# release degrades to "skip this track" instead of a wrong-column silent failure.
CPC18_ALIASES = {
    "participant_id": ["subjid", "subject", "subj", "participant", "id", "sub"],
    "problem_id":     ["gameid", "game", "problem", "problemid", "prob"],
    "block":          ["block", "blk"],
    "trial":          ["trial", "trialnum", "t"],
    "feedback":       ["feedback", "fb"],
    "choice_b":       ["b", "choiceb", "choice", "bchoice", "chose_b", "choice_b"],
    "ha": ["ha"], "pha": ["pha"], "la": ["la"],
    "hb": ["hb"], "phb": ["phb"], "lb": ["lb"],
    "lotshape": ["lotshapeb", "lotshape"], "lotnum": ["lotnumb", "lotnum"],
    "amb": ["amb", "ambiguity"], "corr": ["corr", "correlation"],
    "age": ["age"], "gender": ["gender", "sex"], "location": ["location", "site"],
}

def _resolve_columns(df: pd.DataFrame, aliases: dict) -> dict:
    lower = {c.lower().replace(" ", "").replace("_", ""): c for c in df.columns}
    out = {}
    for canon, opts in aliases.items():
        for o in opts:
            key = o.lower().replace(" ", "").replace("_", "")
            if key in lower:
                out[canon] = lower[key]
                break
    return out

def load_cpc18() -> pd.DataFrame | None:
    if not CONFIG.enable_cpc18:
        return None
    try:
        p = fetch_with_fallbacks(CPC18_SOURCES, "cpc18_estimation_raw.csv",
                                 min_bytes=1_000_000, expect_text=True)
    except Exception as exc:
        LOG.error("CPC18 unavailable: %s", exc)
        return None
    try:
        df = pd.read_csv(p, low_memory=False)
    except Exception as exc:
        LOG.error("CPC18 unreadable: %s", exc)
        return None

    cols = _resolve_columns(df, CPC18_ALIASES)
    LOG.info("CPC18 raw header: %s", list(df.columns))
    LOG.info("CPC18 resolved: %s", cols)
    essential = {"participant_id", "problem_id", "choice_b", "ha", "pha", "la", "hb", "phb", "lb"}
    if not essential.issubset(cols):
        LOG.error("CPC18 column resolution incomplete (missing %s); "
                  "skipping this track rather than guessing.", essential - set(cols))
        return None

    out = df.rename(columns={v: k for k, v in cols.items()})
    out = out[[c for c in cols]].copy()
    out["choice_b"] = pd.to_numeric(out["choice_b"], errors="coerce")
    out = out.dropna(subset=["choice_b", "ha", "pha", "la", "hb", "phb", "lb"])
    if not out["choice_b"].dropna().isin([0, 1]).all():
        LOG.error("CPC18 choice column is not binary; skipping track.")
        return None
    LOG.info("CPC18 loaded: %s rows, %s participants", f"{len(out):,}",
             out["participant_id"].nunique())
    return out

RAW_CPC18 = load_cpc18()
print("CPC18:", "OK" if RAW_CPC18 is not None else "UNAVAILABLE",
      f"({len(RAW_CPC18):,} rows)" if RAW_CPC18 is not None else "")

In [ ]:
# --- 8.5 Ingestion summary ----------------------------------------------------------
AVAILABLE = {
    "choices13k": RAW_C13K is not None,
    "itc_database": RAW_ITC is not None,
    "cpc18": RAW_CPC18 is not None,
}
pd.DataFrame(INGEST_LOG).to_csv(OUTPUTS / "ingestion_log.csv", index=False)

print("\n" + "=" * 72)
print("INGESTION SUMMARY")
print("=" * 72)
for k, v in AVAILABLE.items():
    print(f"  {k:16s} {'AVAILABLE' if v else 'UNAVAILABLE -> dependent sections will be skipped'}")
if INGEST_LOG:
    display(pd.DataFrame(INGEST_LOG))

if not any(AVAILABLE.values()):
    raise SystemExit("No dataset could be retrieved. The notebook refuses to continue "
                     "with an empty or synthetic dataset. Check network access and rerun.")

## 9. Automated data validation

Every dataset passes through the same battery: existence, schema, dtypes, missingness,
duplicates, ID integrity, range/impossible-value checks, constant and near-zero-variance
columns, and outliers. **Nothing is silently dropped.** Every exclusion is counted,
reported, and justified — and where the source itself ships exclusion flags (the ITC
Database does), we honour the original authors' criteria rather than inventing our own.

In [ ]:
# --- 9.1 Generic validation engine --------------------------------------------------
QUALITY_ROWS: list[dict] = []
CLEANING_DECISIONS: list[dict] = []

def note_cleaning(dataset: str, action: str, n_rows: int, reason: str):
    CLEANING_DECISIONS.append(dict(dataset=dataset, action=action,
                                   rows_affected=int(n_rows), reason=reason))
    LOG.info("[%s] %s (%d rows): %s", dataset, action, n_rows, reason)

def validate_frame(df: pd.DataFrame, dataset: str, *,
                   id_cols: Sequence[str] = (),
                   range_rules: dict[str, tuple[float, float]] | None = None,
                   binary_cols: Sequence[str] = ()) -> pd.DataFrame:
    """Profiles a dataframe and appends per-column findings to the global report."""
    rows = []
    n = len(df)
    dup_all = int(df.duplicated().sum())
    for col in df.columns:
        s = df[col]
        miss = int(s.isna().sum())
        nun = int(s.nunique(dropna=True))
        rec = dict(dataset=dataset, column=col, dtype=str(s.dtype), n_rows=n,
                   n_missing=miss, pct_missing=round(100 * miss / max(n, 1), 3),
                   n_unique=nun, constant=(nun <= 1),
                   near_zero_variance=False, n_outliers_iqr=0,
                   range_violations=0, non_binary_values=0, notes="")
        if pd.api.types.is_numeric_dtype(s) and nun > 1:
            v = s.dropna().astype(float)
            if len(v) > 10:
                sd, mu = v.std(), abs(v.mean())
                rec["near_zero_variance"] = bool(sd < 1e-9 or (mu > 0 and sd / (mu + 1e-12) < 1e-4))
                q1, q3 = np.percentile(v, [25, 75]); iqr = q3 - q1
                if iqr > 0:
                    rec["n_outliers_iqr"] = int(((v < q1 - 3 * iqr) | (v > q3 + 3 * iqr)).sum())
        if range_rules and col in range_rules:
            lo, hi = range_rules[col]
            rec["range_violations"] = int(((s < lo) | (s > hi)).sum())
        if col in binary_cols:
            rec["non_binary_values"] = int((~s.dropna().isin([0, 1])).sum())
        if col in id_cols:
            rec["notes"] = f"ID column; {nun} unique"
        rows.append(rec)

    QUALITY_ROWS.extend(rows)
    rep = pd.DataFrame(rows)
    print(f"\n--- validation: {dataset} ---")
    print(f"  shape                     {df.shape}")
    print(f"  fully duplicated rows     {dup_all}")
    print(f"  columns with missing      {(rep.n_missing > 0).sum()}")
    print(f"  constant columns          {list(rep.loc[rep.constant, 'column'])}")
    print(f"  near-zero-variance cols   {list(rep.loc[rep.near_zero_variance, 'column'])}")
    bad = rep[(rep.range_violations > 0) | (rep.non_binary_values > 0)]
    if len(bad):
        print("  RANGE / DOMAIN VIOLATIONS:")
        print(bad[["column", "range_violations", "non_binary_values"]].to_string(index=False))
    else:
        print("  range / domain checks     all passed")
    for c in id_cols:
        if c in df.columns:
            print(f"  {c:24s}  {df[c].nunique()} unique, {int(df[c].isna().sum())} missing")
    return rep

In [ ]:
# --- 9.2 Validate choices13k --------------------------------------------------------
if AVAILABLE["choices13k"]:
    c13 = RAW_C13K["selections"].copy()
    validate_frame(
        c13, "choices13k",
        id_cols=["Problem"],
        range_rules={"pHa": (0, 1), "pHb": (0, 1), "bRate": (0, 1),
                     "n": (1, 1000), "Block": (1, 5), "LotNumB": (1, 20)},
    )
    # Integrity: the aggregate rate must be representable as k successes out of n trials.
    k_est = c13["bRate"] * c13["n"]
    resid = (k_est - k_est.round()).abs()
    print(f"  bRate*n integer-consistency: max deviation {resid.max():.4f} "
          f"({int((resid > 1e-6).sum())} rows non-integral -> rounded, documented)")
    if (resid > 1e-6).any():
        note_cleaning("choices13k", "round bRate*n to nearest integer",
                      int((resid > 1e-6).sum()),
                      "aggregate rates are stored as floats; binomial counts must be integral")
    print(f"  problems seen with AND without feedback: "
          f"{int((c13.groupby('Problem')['Feedback'].nunique() == 2).sum())}")

In [ ]:
# --- 9.3 Validate ITC Database ------------------------------------------------------
if AVAILABLE["itc_database"]:
    itc = RAW_ITC.copy()
    for c in ["ss_value", "ss_time", "ll_value", "ll_time", "choice", "rt", "age"]:
        if c in itc.columns:
            itc[c] = pd.to_numeric(itc[c], errors="coerce")
    validate_frame(
        itc, "itc_database",
        id_cols=[c for c in ["paper", "subj_ident", "sess_ident"] if c in itc.columns],
        range_rules={"ss_time": (0, 40_000), "ll_time": (0, 40_000), "age": (0, 120)},
        binary_cols=["choice"],
    )

    n0 = len(itc)
    # (a) honour the ORIGINAL authors' exclusion flags rather than inventing our own
    for flag, label in [("trial_excl", "trial"), ("subj_excl", "subject")]:
        if flag in itc.columns:
            mask = itc[flag].astype(str).str.lower().isin(["true", "1", "1.0", "yes"])
            if mask.any():
                note_cleaning("itc_database", f"drop rows flagged {flag} by source authors",
                              int(mask.sum()),
                              f"the ITC Database ships curator-defined {label} exclusion flags; "
                              "we defer to the original criteria for comparability")
                itc = itc.loc[~mask]

    # (b) structural validity of the intertemporal choice itself
    need = ["ss_value", "ss_time", "ll_value", "ll_time", "choice"]
    m_na = itc[need].isna().any(axis=1)
    note_cleaning("itc_database", "drop rows missing a core decision variable",
                  int(m_na.sum()), "cannot define an intertemporal trade-off without all four "
                                   "amounts/delays plus the observed choice")
    itc = itc.loc[~m_na]

    m_bad = ~(
        (itc.ll_value > itc.ss_value) & (itc.ll_time > itc.ss_time)
        & (itc.ss_value > 0) & (itc.ss_time >= 0)
    )
    note_cleaning("itc_database", "drop trials violating the SS/LL definition",
                  int(m_bad.sum()),
                  "a standard ITC trial requires ll_value > ss_value and ll_time > ss_time "
                  "with a positive sooner amount; violations are protocol variants "
                  "(e.g. loss framing) that the discounting models below do not cover")
    itc = itc.loc[~m_bad]

    # (c) implausible delays. The pooled database contains a handful of trials with
    #     delays of tens of thousands of days (the raw run showed a maximum of ~527,000
    #     days, i.e. ~1,444 years). These are almost certainly unit or data-entry errors,
    #     and because the discount functions below are exponential in delay they would
    #     dominate the likelihood. They are removed explicitly, never silently.
    MAX_PLAUSIBLE_DELAY_DAYS = 365 * 50
    m_delay = (itc.ll_time - itc.ss_time) > MAX_PLAUSIBLE_DELAY_DAYS
    if m_delay.any():
        LOG.info("  implausible delays observed: max %.0f days",
                 float((itc.ll_time - itc.ss_time).max()))
    note_cleaning("itc_database",
                  f"drop trials with delay > {MAX_PLAUSIBLE_DELAY_DAYS} days (50 years)",
                  int(m_delay.sum()),
                  "delays of this size are unit/data-entry errors; they would dominate the "
                  "likelihood of any exponential or hyperbolic discount function")
    itc = itc.loc[~m_delay]

    # (d) fully duplicated rows are reported but RETAINED: they are legitimate repeats
    #     within a participant, and participant-grouped splitting keeps them inside a
    #     single split, so they cannot leak across the train/test boundary.
    n_dup = int(itc.duplicated().sum())
    note_cleaning("itc_database", "fully duplicated rows RETAINED (reported only)", n_dup,
                  "identical rows sit within a participant, and participant-grouped "
                  "splitting confines them to one split, so they cannot leak")

    m_ch = ~itc["choice"].isin([0, 1])
    note_cleaning("itc_database", "drop non-binary choices", int(m_ch.sum()),
                  "target must be binary SS=0 / LL=1")
    itc = itc.loc[~m_ch]

    if CONFIG.max_itc_rows and len(itc) > CONFIG.max_itc_rows:
        keep = (itc.groupby("subj_ident", observed=True).ngroup()
                % max(1, len(itc) // CONFIG.max_itc_rows)) == 0
        note_cleaning("itc_database", f"participant-level subsample to ~{CONFIG.max_itc_rows:,} rows",
                      int((~keep).sum()),
                      "CONFIG.max_itc_rows caps runtime; subsampling is done by PARTICIPANT, never "
                      "by trial, so no participant is ever partially observed. Set "
                      "CONFIG.max_itc_rows = None to use every trial.")
        itc = itc.loc[keep]

    print(f"\n  retained {len(itc):,} / {n0:,} rows "
          f"({itc.subj_ident.nunique():,} participants, "
          f"{itc['paper'].nunique() if 'paper' in itc else '?'} studies)")
else:
    itc = None

In [ ]:
# --- 9.4 Validate CPC18 + write the data-quality report -----------------------------
if AVAILABLE["cpc18"]:
    cpc = RAW_CPC18.copy()
    validate_frame(cpc, "cpc18",
                   id_cols=[c for c in ["participant_id", "problem_id"] if c in cpc.columns],
                   range_rules={"pha": (0, 1), "phb": (0, 1)},
                   binary_cols=["choice_b"])
else:
    cpc = None

QUALITY = pd.DataFrame(QUALITY_ROWS)
QUALITY.to_csv(OUTPUTS / "data_quality_report.csv", index=False)
CLEAN_DF = pd.DataFrame(CLEANING_DECISIONS)
CLEAN_DF.to_csv(OUTPUTS / "cleaning_decisions.csv", index=False)

print("\n" + "=" * 72)
print("CLEANING DECISION LOG (every exclusion, with its justification)")
print("=" * 72)
if len(CLEAN_DF):
    display(CLEAN_DF)
else:
    print("  no rows were excluded")
print(f"\ndata_quality_report.csv: {len(QUALITY)} column-level records written")

## 10. Exploratory data analysis

EDA is deliberately targeted at the behavioural questions rather than being a generic
column dump: what do choice rates look like, how do they move with the decision
environment, and is the classic signature of each phenomenon actually present in this
data?

In [ ]:
# --- 10.1 choices13k exploration ----------------------------------------------------
if AVAILABLE["choices13k"]:
    fig, ax = plt.subplots(2, 3, figsize=(16, 8))
    a = ax.ravel()
    a[0].hist(c13.bRate, bins=40, color="#4C72B0", edgecolor="white")
    a[0].set(title="Choice rate for option B", xlabel="bRate", ylabel="problems")

    a[1].hist(c13.n, bins=30, color="#55A868", edgecolor="white")
    a[1].set(title="Participants per problem", xlabel="n")

    ev_a = c13.pHa * c13.Ha + (1 - c13.pHa) * c13.La
    ev_b = c13.pHb * c13.Hb + (1 - c13.pHb) * c13.Lb
    a[2].scatter(ev_b - ev_a, c13.bRate, s=3, alpha=.15, color="#C44E52")
    a[2].set(title="Choice rate vs expected-value advantage of B",
             xlabel="EV(B) - EV(A)", ylabel="bRate")
    a[2].axhline(.5, ls="--", c="k", lw=.8); a[2].axvline(0, ls="--", c="k", lw=.8)

    sns.boxplot(x=c13.Feedback, y=c13.bRate, ax=a[3], palette="Set2")
    a[3].set(title="Description vs experience", xlabel="feedback given", ylabel="bRate")

    for amb, lbl in [(False, "risk (known p)"), (True, "ambiguity")]:
        sub = c13.loc[c13.Amb == amb, "bRate"]
        a[4].hist(sub, bins=30, alpha=.55, label=f"{lbl} (n={len(sub)})", density=True)
    a[4].legend(); a[4].set(title="Risk vs ambiguity", xlabel="bRate")

    lot = c13.groupby("LotNumB").bRate.agg(["mean", "count"])
    a[5].bar(lot.index.astype(str), lot["mean"], color="#8172B2")
    a[5].set(title="Choice rate by number of outcomes in B",
             xlabel="LotNumB (outcome count)", ylabel="mean bRate")
    savefig("eda_choices13k"); plt.show()

    print(f"overall mean bRate            {c13.bRate.mean():.4f}")
    print(f"mean bRate | no feedback      {c13.loc[~c13.Feedback,'bRate'].mean():.4f}")
    print(f"mean bRate | feedback         {c13.loc[c13.Feedback,'bRate'].mean():.4f}")

In [ ]:
# --- 10.2 ITC exploration -----------------------------------------------------------
if AVAILABLE["itc_database"]:
    itc["delay"] = itc.ll_time - itc.ss_time
    itc["amount_ratio"] = itc.ll_value / itc.ss_value.replace(0, np.nan)

    fig, ax = plt.subplots(2, 3, figsize=(16, 8)); a = ax.ravel()
    per_subj = itc.groupby("subj_ident", observed=True).choice.mean()
    a[0].hist(per_subj, bins=40, color="#4C72B0", edgecolor="white")
    a[0].set(title="Larger-later choice rate per participant",
             xlabel="P(choose LL)", ylabel="participants")

    tr = itc.groupby("subj_ident", observed=True).size()
    a[1].hist(np.log10(tr.clip(lower=1)), bins=40, color="#55A868", edgecolor="white")
    a[1].set(title="Trials per participant", xlabel="log10(trials)")

    # --- the discount curve: P(LL) as a function of delay
    bins = [0, 1, 3, 7, 14, 30, 60, 90, 180, 365, 730, 1e9]
    lbl = ["<1d", "1-3d", "3-7d", "1-2w", "2-4w", "1-2m", "2-3m", "3-6m", "6-12m", "1-2y", ">2y"]
    itc["delay_bin"] = pd.cut(itc.delay, bins=bins, labels=lbl, right=True)
    dc = itc.groupby("delay_bin", observed=True).choice.agg(["mean", "count"])
    a[2].plot(range(len(dc)), dc["mean"], "o-", color="#C44E52")
    a[2].set_xticks(range(len(dc))); a[2].set_xticklabels(dc.index, rotation=45)
    a[2].set(title="Empirical discount curve", ylabel="P(choose larger-later)", xlabel="delay")

    rb = pd.cut(itc.amount_ratio.clip(1, 6), bins=12)
    rr = itc.groupby(rb, observed=True).choice.mean()
    a[3].plot([iv.mid for iv in rr.index], rr.values, "o-", color="#8172B2")
    a[3].set(title="P(LL) vs reward ratio", xlabel="ll_value / ss_value", ylabel="P(LL)")

    top = itc["paper"].value_counts().head(15)
    rates = itc.loc[itc["paper"].isin(top.index)].groupby("paper", observed=True).choice.mean().sort_values()
    a[4].barh(range(len(rates)), rates.values, color="#937860")
    a[4].set_yticks(range(len(rates))); a[4].set_yticklabels(rates.index, fontsize=7)
    a[4].set(title="LL-choice rate by study (top 15 by size)", xlabel="P(LL)")

    if "rt" in itc.columns and itc.rt.notna().any():
        v = itc.rt.dropna(); v = v[(v > 0) & (v < v.quantile(.99))]
        a[5].hist(v, bins=50, color="#DA8BC3", edgecolor="white")
        a[5].set(title="Response time (POST-decision -> excluded from features)",
                 xlabel="seconds")
    savefig("eda_itc"); plt.show()

    print(f"participants {itc.subj_ident.nunique():,} | studies {itc['paper'].nunique()} | "
          f"trials {len(itc):,} | overall P(LL) {itc.choice.mean():.4f}")
    print(f"delay range {itc.delay.min():.0f}-{itc.delay.max():.0f} days")

## 11. Data harmonisation — the unified behavioural schema

The three datasets measure genuinely different decisions, so they are **not** concatenated
at the row level. Instead each is mapped into one shared schema and kept as its own
**modelling track**. Fields a dataset does not measure are `NaN`, never imputed with
invented values. This is what makes the later cross-dataset comparison honest: the
`RISK` and `RISK_IND` tracks share a schema and *can* be compared; `ITC` cannot be
compared to them and is never claimed to be.

In [ ]:
# --- 11.1 Unified schema definition -------------------------------------------------
UNIFIED_FIELDS = [
    "dataset_id", "track", "experiment_id", "participant_id", "trial_id",
    "choice_binary", "obs_weight", "n_respondents",
    "immediate_reward", "delayed_reward", "delay_days",
    "gain_amount", "loss_amount", "probability",
    "price", "reference_price", "discount",
    "framing", "default_option", "num_choices",
    "condition", "session", "treatment", "age", "country",
]

# Numeric fields are created as float so the frame stays parquet-serialisable; only
# genuinely categorical fields are object-typed.
UNIFIED_TEXT_FIELDS = {"dataset_id", "track", "experiment_id", "participant_id",
                       "framing", "default_option", "condition", "session", "country"}

def blank_unified(n: int) -> pd.DataFrame:
    return pd.DataFrame({
        f: pd.Series([np.nan] * n, dtype=("object" if f in UNIFIED_TEXT_FIELDS else "float64"))
        for f in UNIFIED_FIELDS})

UNIFIED_PARTS = []

if AVAILABLE["itc_database"]:
    u = blank_unified(len(itc))
    u["dataset_id"] = "itc_database"; u["track"] = "ITC"
    u["experiment_id"] = itc["paper"].values
    u["participant_id"] = itc["subj_ident"].values
    u["trial_id"] = itc["trial_idx"].values if "trial_idx" in itc else np.arange(len(itc))
    u["choice_binary"] = itc["choice"].values           # 1 = larger-later
    u["obs_weight"] = 1.0; u["n_respondents"] = 1
    u["immediate_reward"] = itc["ss_value"].values
    u["delayed_reward"] = itc["ll_value"].values
    u["delay_days"] = itc["delay"].values
    u["gain_amount"] = itc["ll_value"].values           # all selected trials are gains
    u["num_choices"] = 2
    u["session"] = itc["sess_ident"].values if "sess_ident" in itc else np.nan
    u["condition"] = (itc["additional_interventions"].values
                      if "additional_interventions" in itc else np.nan)
    u["age"] = itc["age"].values if "age" in itc else np.nan
    u["country"] = itc["country"].values if "country" in itc else np.nan
    UNIFIED_PARTS.append(u)

if AVAILABLE["choices13k"]:
    u = blank_unified(len(c13))
    u["dataset_id"] = "choices13k"; u["track"] = "RISK"
    u["experiment_id"] = "choices13k"
    u["participant_id"] = np.nan                        # aggregate rates: no individual IDs
    u["trial_id"] = c13["Problem"].values
    u["choice_binary"] = c13["bRate"].values            # a RATE, expanded to counts later
    u["obs_weight"] = c13["n"].values
    u["n_respondents"] = c13["n"].values
    u["gain_amount"] = np.maximum(c13[["Ha", "Hb"]].max(axis=1), 0).values
    u["loss_amount"] = np.minimum(c13[["La", "Lb"]].min(axis=1), 0).values
    u["probability"] = c13["pHb"].values
    u["num_choices"] = 2
    u["treatment"] = c13["Feedback"].astype(int).values  # description(0) vs experience(1)
    u["condition"] = np.where(c13["Amb"], "ambiguity", "risk")
    UNIFIED_PARTS.append(u)

if AVAILABLE["cpc18"]:
    u = blank_unified(len(cpc))
    u["dataset_id"] = "cpc18"; u["track"] = "RISK_IND"
    u["experiment_id"] = "cpc18"
    u["participant_id"] = cpc["participant_id"].values
    u["trial_id"] = cpc["problem_id"].values
    u["choice_binary"] = cpc["choice_b"].values
    u["obs_weight"] = 1.0; u["n_respondents"] = 1
    u["gain_amount"] = np.maximum(cpc[["ha", "hb"]].max(axis=1), 0).values
    u["loss_amount"] = np.minimum(cpc[["la", "lb"]].min(axis=1), 0).values
    u["probability"] = cpc["phb"].values
    u["num_choices"] = 2
    if "feedback" in cpc: u["treatment"] = pd.to_numeric(cpc["feedback"], errors="coerce").values
    if "age" in cpc: u["age"] = cpc["age"].values
    UNIFIED_PARTS.append(u)

UNIFIED = pd.concat(UNIFIED_PARTS, ignore_index=True) if UNIFIED_PARTS else pd.DataFrame()
if len(UNIFIED):
    try:
        UNIFIED.to_parquet(PROCESSED / "unified_behavioral.parquet")
    except Exception as exc:                       # no parquet engine available
        LOG.warning("parquet unavailable (%s); writing CSV instead", type(exc).__name__)
        UNIFIED.to_csv(PROCESSED / "unified_behavioral.csv", index=False)

cov = (UNIFIED.assign(_t=UNIFIED.track).groupby("_t")
       .apply(lambda g: g[UNIFIED_FIELDS].notna().mean().round(2), include_groups=False))
print("Schema coverage by track (fraction of rows where the field is populated):")
display(cov.T)
print("\nBlank cells above are fields the experiment genuinely did not measure. "
      "They are left as NaN and never imputed.")

## 12. Behavioural feature engineering

Every feature below is a function of information **available before the decision is
made** — the offered amounts, delays, probabilities, outcome distributions, experimental
condition, and the participant's *own previously observed* choices. Response time,
realised payoffs and anything else generated by or after the decision are excluded by
construction and are re-checked in the leakage audit.

In [ ]:
# --- 12.1 Risky-choice features (choices13k): exact distributions -------------------
EPS = 1e-12

def _pad(list_of_arrays, width):
    out = np.zeros((len(list_of_arrays), width))
    for i, a in enumerate(list_of_arrays):
        out[i, :len(a)] = a
    return out

class ProspectTableau:
    """Pre-ranked representation of a set of prospects.

    The CPT rank ordering depends only on outcome values, never on the fitted parameters,
    so ranking and cumulative probabilities are computed ONCE here and reused on every
    likelihood evaluation. This is what makes maximum-likelihood CPT fitting on ~10^4
    problems take seconds instead of hours.
    """
    def __init__(self, outcomes_list, probs_list):
        width = max(len(o) for o in outcomes_list)
        self.O = _pad(outcomes_list, width); self.P = _pad(probs_list, width)
        self.mask = _pad([np.ones(len(o)) for o in outcomes_list], width).astype(bool)
        go, gc, gp, lo, lc, lp = [], [], [], [], [], []
        for o, p in zip(outcomes_list, probs_list):
            o = np.asarray(o, float); p = np.asarray(p, float)
            g = o > 0
            if g.any():
                og, pg = o[g], p[g]; k = np.argsort(-og); og, pg = og[k], pg[k]
                c = np.cumsum(pg); go.append(og); gc.append(c)
                gp.append(np.concatenate([[0.], c[:-1]]))
            else:
                go.append(np.zeros(0)); gc.append(np.zeros(0)); gp.append(np.zeros(0))
            l = o < 0
            if l.any():
                ol, pl = o[l], p[l]; k = np.argsort(ol); ol, pl = ol[k], pl[k]
                c = np.cumsum(pl); lo.append(ol); lc.append(c)
                lp.append(np.concatenate([[0.], c[:-1]]))
            else:
                lo.append(np.zeros(0)); lc.append(np.zeros(0)); lp.append(np.zeros(0))
        gw, lw = max(1, max(map(len, go))), max(1, max(map(len, lo)))
        self.GO, self.GC, self.GP = _pad(go, gw), _pad(gc, gw), _pad(gp, gw)
        self.GM = _pad([np.ones(len(a)) for a in go], gw).astype(bool)
        self.LO, self.LC, self.LP = _pad(lo, lw), _pad(lc, lw), _pad(lp, lw)
        self.LM = _pad([np.ones(len(a)) for a in lo], lw).astype(bool)
        self.ev = np.sum(self.O * self.P, axis=1)
        self.var = np.sum(self.P * (self.O - self.ev[:, None]) ** 2 * self.mask, axis=1)
        self.sd = np.sqrt(np.maximum(self.var, 0))
        self.p_loss = np.sum(np.where(self.O < 0, self.P, 0.), axis=1)
        self.p_gain = np.sum(np.where(self.O > 0, self.P, 0.), axis=1)
        self.e_gain = np.sum(np.where(self.O > 0, self.P * self.O, 0.), axis=1)
        self.e_loss = np.sum(np.where(self.O < 0, self.P * self.O, 0.), axis=1)
        self.omax = np.where(self.mask, self.O, -np.inf).max(axis=1)
        self.omin = np.where(self.mask, self.O, np.inf).min(axis=1)
        self.n_out = self.mask.sum(axis=1)

    def subset(self, idx):
        new = object.__new__(ProspectTableau)
        for k, v in self.__dict__.items():
            new.__dict__[k] = v[idx] if isinstance(v, np.ndarray) else v
        return new

    def eu(self, alpha, lam=1.0):
        u = np.where(self.O >= 0, np.abs(self.O) ** alpha, -lam * np.abs(self.O) ** alpha)
        return np.sum(np.where(self.mask, self.P * u, 0.), axis=1)

    @staticmethod
    def _w(p, gamma):
        p = np.clip(p, EPS, 1.0)
        return p ** gamma / ((p ** gamma + (1 - p) ** gamma) ** (1. / gamma))

    def cpt(self, alpha, beta, lam, gg, gl):
        dwg = self._w(self.GC, gg) - self._w(self.GP, gg)
        vg = np.sum(np.where(self.GM, dwg * np.abs(self.GO) ** alpha, 0.), axis=1)
        dwl = self._w(self.LC, gl) - self._w(self.LP, gl)
        vl = np.sum(np.where(self.LM, dwl * (-lam * np.abs(self.LO) ** beta), 0.), axis=1)
        return vg + vl

if AVAILABLE["choices13k"]:
    probs = RAW_C13K["problems"]
    oa, pa, ob, pb = [], [], [], []
    for i in range(len(c13)):
        p = probs[str(i)]
        A = np.asarray(p["A"], float); B = np.asarray(p["B"], float)
        pa.append(A[:, 0]); oa.append(A[:, 1]); pb.append(B[:, 0]); ob.append(B[:, 1])
    TA, TB = ProspectTableau(oa, pa), ProspectTableau(ob, pb)

    RISK_X = pd.DataFrame({
        # --- expected value / risk
        "ev_a": TA.ev, "ev_b": TB.ev, "ev_diff": TB.ev - TA.ev,
        "ev_ratio": TB.ev / (np.abs(TA.ev) + 1),
        "sd_a": TA.sd, "sd_b": TB.sd, "sd_diff": TB.sd - TA.sd,
        "cv_b": TB.sd / (np.abs(TB.ev) + 1),
        # --- gain / loss decomposition (loss-aversion relevant)
        "p_loss_a": TA.p_loss, "p_loss_b": TB.p_loss,
        "p_loss_diff": TB.p_loss - TA.p_loss,
        "e_gain_a": TA.e_gain, "e_gain_b": TB.e_gain,
        "e_loss_a": TA.e_loss, "e_loss_b": TB.e_loss,
        "gain_loss_ratio_b": TB.e_gain / (np.abs(TB.e_loss) + 1),
        "gain_loss_ratio_a": TA.e_gain / (np.abs(TA.e_loss) + 1),
        "any_loss": ((TA.p_loss > 0) | (TB.p_loss > 0)).astype(int),
        # --- extremes / range
        "max_a": TA.omax, "min_a": TA.omin, "max_b": TB.omax, "min_b": TB.omin,
        "range_a": TA.omax - TA.omin, "range_b": TB.omax - TB.omin,
        "best_outcome_is_b": (TB.omax > TA.omax).astype(int),
        "worst_outcome_is_b": (TB.omin < TA.omin).astype(int),
        # --- complexity / choice architecture
        "n_out_a": TA.n_out, "n_out_b": TB.n_out,
        "complexity_diff": TB.n_out - TA.n_out,
        "lot_shape_b": c13["LotShapeB"].values, "lot_num_b": c13["LotNumB"].values,
        # --- rare-event structure (drives the description-experience gap)
        "p_high_a": c13["pHa"].values, "p_high_b": c13["pHb"].values,
        "rare_event_b": ((c13["pHb"] < .10) | (c13["pHb"] > .90)).astype(int).values,
        "rare_event_a": ((c13["pHa"] < .10) | (c13["pHa"] > .90)).astype(int).values,
        # --- dominance (a normative benchmark the model should respect)
        "b_dominates_a": (TB.omin >= TA.omax).astype(int),
        "a_dominates_b": (TA.omin >= TB.omax).astype(int),
        # --- experimental condition (pre-decision, manipulated)
        "feedback": c13["Feedback"].astype(int).values,
        "block": c13["Block"].values,
        "ambiguity": c13["Amb"].astype(int).values,
        "correlation": c13["Corr"].values,
    })
    RISK_y = c13["bRate"].to_numpy(float)
    RISK_n = c13["n"].to_numpy(int)
    RISK_k = np.rint(RISK_y * RISK_n).astype(int)
    RISK_groups = c13["Problem"].to_numpy()
    print(f"RISK features: {RISK_X.shape[0]:,} problems x {RISK_X.shape[1]} features")
    display(RISK_X.head(3))

In [ ]:
# --- 12.2 Intertemporal features (ITC) ----------------------------------------------
def itc_features(df: pd.DataFrame, include_history: bool = True) -> pd.DataFrame:
    """All features are strictly pre-decision. Participant-history features use only
    a participant's OWN EARLIER trials (expanding mean shifted by one)."""
    d = df.copy()
    ss, ll = d.ss_value.to_numpy(float), d.ll_value.to_numpy(float)
    ts, tl = d.ss_time.to_numpy(float), d.ll_time.to_numpy(float)
    delay = np.maximum(tl - ts, 1e-6)

    X = pd.DataFrame(index=d.index)
    X["ss_value"] = ss; X["ll_value"] = ll
    X["ss_time"] = ts; X["ll_time"] = tl; X["delay"] = delay
    X["log_ss_value"] = np.log1p(ss); X["log_ll_value"] = np.log1p(ll)
    X["log_delay"] = np.log1p(delay)
    X["amount_diff"] = ll - ss
    X["amount_ratio"] = ll / np.maximum(ss, 1e-6)
    X["rel_amount_gain"] = (ll - ss) / np.maximum(ss, 1e-6)
    X["is_immediate_ss"] = (ts <= 0).astype(int)      # front-end delay => present bias

    # --- implied discount rates: the k that makes the decision-maker indifferent
    X["k_indiff_hyperbolic"] = (ll / np.maximum(ss, 1e-6) - 1.0) / delay
    X["k_indiff_exponential"] = np.log(np.maximum(ll / np.maximum(ss, 1e-6), 1 + 1e-9)) / delay
    X["log_k_indiff_hyp"] = np.log(np.maximum(X["k_indiff_hyperbolic"], 1e-9))
    X["annual_implied_rate"] = (ll / np.maximum(ss, 1e-6)) ** (365.0 / delay) - 1.0
    X["annual_implied_rate"] = X["annual_implied_rate"].clip(-1, 1e4)

    # --- subjective values at literature-reference discount rates
    for k_ref in (0.01, 0.05, 0.20):
        X[f"sv_diff_hyp_k{k_ref}"] = ll / (1 + k_ref * tl) - ss / (1 + k_ref * ts)
        X[f"sv_diff_exp_k{k_ref}"] = ll * np.exp(-k_ref * tl) - ss * np.exp(-k_ref * ts)

    # --- design / protocol moderators (pre-decision, part of the choice architecture)
    for col, name in [("procedure", "procedure"), ("incentivization", "incentivization"),
                      ("presentation_of_information", "presentation"),
                      ("fixed_attributes", "fixed_attributes"),
                      ("additional_interventions", "intervention"),
                      ("country", "country"), ("currency", "currency")]:
        if col in d.columns:
            X[f"cat_{name}"] = d[col].astype("category")
    for col in ["time_pressure", "online_study"]:
        if col in d.columns:
            X[col] = pd.to_numeric(d[col], errors="coerce")
    if "age" in d.columns:
        X["age"] = pd.to_numeric(d["age"], errors="coerce")

    # --- participant history: strictly past-only
    if include_history and "subj_ident" in d.columns:
        order = d["trial_idx"] if "trial_idx" in d.columns else pd.Series(np.arange(len(d)), index=d.index)
        tmp = pd.DataFrame({"g": d["subj_ident"].values, "o": order.values,
                            "y": d["choice"].values}, index=d.index).sort_values(["g", "o"])
        grp = tmp.groupby("g", observed=True)["y"]
        hist_rate = grp.transform(lambda s: s.shift(1).expanding().mean())
        hist_n = grp.transform(lambda s: s.shift(1).expanding().count())
        prev = grp.shift(1)
        X["hist_ll_rate"] = hist_rate.reindex(d.index).values
        X["hist_n_trials"] = hist_n.reindex(d.index).fillna(0).values
        X["hist_prev_choice"] = prev.reindex(d.index).values
        X["hist_available"] = (~X["hist_ll_rate"].isna()).astype(int)
    return X

if AVAILABLE["itc_database"]:
    ITC_X = itc_features(itc, include_history=True)
    ITC_y = itc["choice"].to_numpy(int)
    ITC_groups = itc["subj_ident"].to_numpy()
    ITC_studies = itc["paper"].to_numpy()
    print(f"ITC features: {ITC_X.shape[0]:,} trials x {ITC_X.shape[1]} features | "
          f"{len(np.unique(ITC_groups)):,} participants | {len(np.unique(ITC_studies))} studies")
    display(ITC_X.head(3))

## 13. Target construction

Each target is derived from something the experimental design genuinely identifies. No
behavioural label is assigned arbitrarily.

| Target | Track | Derivation | Type |
|---|---|---|---|
| `choice_binary` | ITC | the observed choice: 1 = larger-later, 0 = smaller-sooner | binary |
| `bRate` | RISK | observed proportion choosing option B, with `n` respondents per problem | binomial rate |
| `discount_rate k` | ITC | MLE of the Mazur hyperbolic discount function, per participant | continuous |
| `present_bias beta` | ITC | MLE of the quasi-hyperbolic (β–δ) model, per participant; β<1 ⇒ present bias | continuous |
| `loss_aversion lambda` | RISK | MLE of the CPT value function on the pooled training set | continuous |
| `prob_weighting gamma` | RISK | MLE of the Tversky–Kahneman weighting function | continuous |

**Targets deliberately NOT constructed.** An `anchoring_response`, `framing_response` or
`default_response` label would require a reference price, a gain/loss frame manipulation
or a default option to exist in the data. None of the three selected datasets contains
them, so those labels are **not** created. Inventing them would be exactly the failure
mode the brief warns against.

For the `RISK` track the binomial rate is expanded into weighted binary observations
(`k` successes and `n − k` failures per problem), which lets log loss, Brier score,
ROC-AUC and calibration all be computed on a proper probabilistic footing while still
respecting how many people actually contributed to each rate.

In [ ]:
# --- 13.1 Target summary ------------------------------------------------------------
targets = []
if AVAILABLE["itc_database"]:
    targets.append(dict(track="ITC", target="choice_binary", kind="binary",
                        n=len(ITC_y), positive_rate=round(float(ITC_y.mean()), 4),
                        derivation="observed choice, 1 = larger-later"))
if AVAILABLE["choices13k"]:
    targets.append(dict(track="RISK", target="bRate", kind="binomial rate",
                        n=len(RISK_y), positive_rate=round(float(np.average(RISK_y, weights=RISK_n)), 4),
                        derivation=f"k/n successes; {RISK_n.sum():,} underlying individual decisions"))
if AVAILABLE["cpc18"]:
    targets.append(dict(track="RISK_IND", target="choice_b", kind="binary",
                        n=len(cpc), positive_rate=round(float(cpc.choice_b.mean()), 4),
                        derivation="observed individual choice of option B"))
display(pd.DataFrame(targets))

## 14. Train / validation / test splitting

**70 / 15 / 15, split by group — never by row.**

The grouping variable is chosen from the structure of each experiment:

* **ITC** → group by `subj_ident` (participant). A participant contributes hundreds of
  trials; randomly scattering them across splits would let the model memorise that
  individual's baseline patience and would inflate test performance dramatically. Every
  trial from a participant lands in exactly one split.
* **RISK** → group by `Problem`. 1,562 problems appear twice (once with feedback, once
  without). Splitting those rows apart would leak the answer, because the two rows share
  the same gamble.
* **Cross-study** → group by `paper` for leave-one-study-out, which is a strictly harder
  test than participant-level grouping and is reported separately in Section 28.

The split is stratified on the outcome where the grouping allows it, so class balance is
preserved across splits.

In [ ]:
# --- 14.1 Grouped 70/15/15 splitter -------------------------------------------------
def grouped_split(groups: np.ndarray, y: np.ndarray | None = None,
                  test_size: float = TEST_SIZE, val_size: float = VAL_SIZE,
                  seed: int = RANDOM_SEED) -> dict[str, np.ndarray]:
    """Group-disjoint train/val/test split, stratified on the group-mean outcome so that
    high- and low-response groups are represented in every split."""
    uniq = np.unique(groups)
    rng = np.random.default_rng(seed)
    if y is not None:
        gm = pd.Series(y).groupby(pd.Series(groups)).mean()
        strata = pd.qcut(gm.reindex(uniq).values, q=min(5, max(2, len(uniq) // 20)),
                         labels=False, duplicates="drop")
    else:
        strata = np.zeros(len(uniq), int)
    tr_g, va_g, te_g = [], [], []
    for s in np.unique(strata):
        gs = uniq[strata == s].copy()
        rng.shuffle(gs)
        n = len(gs)
        n_te = int(round(test_size * n)); n_va = int(round(val_size * n))
        te_g += list(gs[:n_te]); va_g += list(gs[n_te:n_te + n_va]); tr_g += list(gs[n_te + n_va:])
    m = lambda gl: np.isin(groups, np.array(gl))
    idx = {"train": np.where(m(tr_g))[0], "val": np.where(m(va_g))[0], "test": np.where(m(te_g))[0]}
    assert not (set(tr_g) & set(va_g)) and not (set(tr_g) & set(te_g)) and not (set(va_g) & set(te_g))
    return idx

SPLITS = {}
if AVAILABLE["itc_database"]:
    SPLITS["ITC"] = grouped_split(ITC_groups, ITC_y)
if AVAILABLE["choices13k"]:
    SPLITS["RISK"] = grouped_split(RISK_groups, RISK_y)

rows = []
for track, sp in SPLITS.items():
    g = ITC_groups if track == "ITC" else RISK_groups
    yy = ITC_y if track == "ITC" else RISK_y
    for name, ix in sp.items():
        rows.append(dict(track=track, split=name, n_rows=len(ix),
                         pct=round(100 * len(ix) / len(g), 1),
                         n_groups=len(np.unique(g[ix])),
                         outcome_mean=round(float(np.mean(yy[ix])), 4)))
SPLIT_TABLE = pd.DataFrame(rows)
display(SPLIT_TABLE)
print("\nGroup-disjointness verified by assertion inside grouped_split().")

## 15. Data leakage audit

A formal audit runs before any model is trained. It checks group disjointness, scans the
feature matrix for post-decision or outcome-derived variables, tests for duplicate rows
straddling splits, and screens for features suspiciously correlated with the target.

In [ ]:
# --- 15.1 Leakage audit -------------------------------------------------------------
# Whole-word-ish tokens naming things only knowable AFTER a decision is made. Features
# that merely describe the OFFER (e.g. best_outcome_is_b) are pre-decision and are
# whitelisted, otherwise the audit cries wolf on legitimate design variables.
POST_DECISION_TOKENS = ["rt", "response_time", "reaction", "payoff", "reward_received",
                        "earned", "feedback_value", "result", "correct", "accuracy",
                        "excl", "brate", "choice", "target", "label", "_y",
                        "outcome_received", "realised", "realized"]
PRE_DECISION_WHITELIST = {"best_outcome_is_b", "worst_outcome_is_b", "n_out_a", "n_out_b",
                          "hist_ll_rate", "hist_prev_choice", "hist_n_trials",
                          "hist_available", "num_choices"}

def leakage_audit(name: str, X: pd.DataFrame, y: np.ndarray, groups: np.ndarray,
                  split: dict, source_cols: Sequence[str] = ()) -> pd.DataFrame:
    findings = []
    def add(check, status, detail):
        findings.append(dict(track=name, check=check, status=status, detail=detail))

    # 1. group disjointness
    gs = {k: set(np.unique(groups[ix])) for k, ix in split.items()}
    overlaps = {f"{a}&{b}": len(gs[a] & gs[b]) for a, b in
                [("train", "val"), ("train", "test"), ("val", "test")]}
    add("participant/problem group disjointness",
        "PASS" if sum(overlaps.values()) == 0 else "FAIL", str(overlaps))

    # 2. index disjointness
    ii = {k: set(ix.tolist()) for k, ix in split.items()}
    ov = len(ii["train"] & ii["val"]) + len(ii["train"] & ii["test"]) + len(ii["val"] & ii["test"])
    add("row-index disjointness", "PASS" if ov == 0 else "FAIL", f"{ov} shared rows")

    # 3. post-decision / outcome-derived feature names
    flagged = [c for c in X.columns
               if any(t in c.lower() for t in POST_DECISION_TOKENS)
               and c not in PRE_DECISION_WHITELIST and not c.startswith("hist_")]
    add("post-decision variables excluded from features",
        "PASS" if not flagged else "REVIEW",
        "none found" if not flagged else f"flagged: {flagged}")

    # 4. post-decision columns present in the SOURCE but correctly not used
    unused = [c for c in source_cols
              if any(t.lower() in c.lower() for t in POST_DECISION_TOKENS)
              and c not in X.columns]
    add("post-decision source columns correctly dropped", "PASS",
        f"withheld from the model: {unused}" if unused else "none present in source")

    # 5. duplicate feature rows straddling splits
    key = pd.util.hash_pandas_object(
        X.select_dtypes(include=[np.number]).round(8), index=False).to_numpy()
    lab = np.full(len(X), "", dtype=object)
    for k, ix in split.items(): lab[ix] = k
    dfk = pd.DataFrame({"key": key, "split": lab})
    strad = dfk.groupby("key").split.nunique()
    n_str = int((strad > 1).sum())
    add("duplicate feature rows across splits",
        "PASS" if n_str == 0 else "REVIEW",
        (f"{n_str} identical feature vectors appear in more than one split. These are "
         "DIFFERENT participants who were shown the SAME offer -- expected in a pooled "
         "database of standardised tasks, and not leakage: the label belongs to another "
         "person and the grouping variable keeps each participant in a single split."
         if n_str else "none"))

    # 6. single-feature target correlation screen
    num = X.select_dtypes(include=[np.number])
    tr = split["train"]
    cors = {}
    for c in num.columns:
        v = num[c].to_numpy(float)[tr]
        if np.nanstd(v) > 0:
            with np.errstate(invalid="ignore"):
                cors[c] = abs(np.corrcoef(np.nan_to_num(v), y[tr])[0, 1])
    top = sorted(cors.items(), key=lambda kv: -kv[1])[:5]
    suspicious = [f"{c}={r:.3f}" for c, r in top if r > 0.97]
    add("suspiciously predictive single feature (|r| > 0.97)",
        "PASS" if not suspicious else "REVIEW",
        f"top: {[(c, round(r,3)) for c, r in top]}")

    # 7. preprocessing fitted on train only
    add("preprocessing fitted on training data only", "PASS",
        "all scalers/encoders live inside sklearn Pipelines fitted on the train split")
    return pd.DataFrame(findings)

audits = []
if AVAILABLE["itc_database"]:
    audits.append(leakage_audit("ITC", ITC_X, ITC_y, ITC_groups, SPLITS["ITC"],
                                source_cols=list(itc.columns)))
if AVAILABLE["choices13k"]:
    audits.append(leakage_audit("RISK", RISK_X, RISK_y, RISK_groups, SPLITS["RISK"],
                                source_cols=list(c13.columns)))
LEAKAGE = pd.concat(audits, ignore_index=True) if audits else pd.DataFrame()
LEAKAGE.to_csv(OUTPUTS / "leakage_audit.csv", index=False)

print("=" * 78); print("LEAKAGE AUDIT"); print("=" * 78)
for _, r in LEAKAGE.iterrows():
    mark = {"PASS": "[PASS]", "FAIL": "[FAIL]", "REVIEW": "[REVIEW]"}[r.status]
    print(f"{mark:9s} {r.track:5s} {r.check}")
    print(f"{'':9s}       {r.detail}")
assert not (LEAKAGE.status == "FAIL").any(), "Leakage audit failed — stopping."
print("\nNo blocking leakage detected.")

## 16. Behavioural economics baselines

This is the scientific core of the project. Before any machine learning is allowed to
claim a win, the classical theories are fitted properly — by maximum likelihood, on the
same training split, and evaluated on the same held-out test split.

**Risky choice (RISK track)** — a nested ladder of increasing psychological realism:

1. **Expected Value** — risk neutrality. `P(B) = σ(θ·[EV(B) − EV(A)])`
2. **CRRA expected utility** — adds diminishing sensitivity via a power utility `u(x)=x^α`
3. **Cumulative Prospect Theory** — adds reference dependence, **loss aversion** `λ`, and
   rank-dependent **probability weighting** `w(p)` separately for gains and losses

**Intertemporal choice (ITC track)** — the classical discount functions:

1. **Exponential** `V = A·e^(−kt)` — time-consistent
2. **Hyperbolic (Mazur)** `V = A/(1+kt)` — the standard descriptive model
3. **Quasi-hyperbolic (β–δ)** `V = A·β^𝟙[t>0]·e^(−kt)` — β<1 isolates **present bias**

All are fitted with a softmax (logit) choice rule, so they produce calibrated
probabilities and can be compared to ML models on identical metrics.

In [ ]:
# --- 16.1 Fitting machinery ---------------------------------------------------------
def binom_nll(p, k, n):
    p = np.clip(p, 1e-9, 1 - 1e-9)
    return float(-np.sum(k * np.log(p) + (n - k) * np.log(1 - p)))

class RiskyChoiceModel:
    """Interpretable behavioural baselines predicting P(choose option B)."""
    SPECS = {
        "ev":      (["theta"], [0.10], [(1e-4, 5.)]),
        "eu_crra": (["theta", "alpha"], [0.10, 0.85], [(1e-4, 5.), (0.05, 2.)]),
        "cpt":     (["theta", "alpha", "beta", "lambda", "gamma_gain", "gamma_loss"],
                    [0.10, 0.85, 0.85, 1.50, 0.90, 0.90],
                    [(1e-4, 5.), (0.05, 2.), (0.05, 2.), (0.05, 10.), (0.28, 2.), (0.28, 2.)]),
    }
    def __init__(self, kind="cpt"):
        self.kind = kind
        self.param_names_, self._init, self._bounds = self.SPECS[kind]
        self.params_ = None

    def _p(self, ta, tb, q):
        if self.kind == "ev":        va, vb = ta.ev, tb.ev
        elif self.kind == "eu_crra": va, vb = ta.eu(q[1]), tb.eu(q[1])
        else:
            a, b, lam, gg, gl = q[1:]
            va, vb = ta.cpt(a, b, lam, gg, gl), tb.cpt(a, b, lam, gg, gl)
        return expit(q[0] * (vb - va))

    def fit(self, ta, tb, k, n, n_restarts=None, seed=RANDOM_SEED):
        n_restarts = n_restarts or CONFIG.cpt_restarts
        rng = np.random.default_rng(seed)
        lo = np.array([b[0] for b in self._bounds]); hi = np.array([b[1] for b in self._bounds])
        starts = [np.array(self._init, float)]
        for _ in range(max(0, n_restarts - 1)):
            starts.append(np.clip(np.array(self._init) * rng.uniform(.5, 1.8, len(self._init)), lo, hi))
        best = None
        for s in starts:
            r = minimize(lambda q: binom_nll(self._p(ta, tb, q), k, n), np.clip(s, lo, hi),
                         bounds=self._bounds, method="L-BFGS-B", options={"maxiter": 400})
            if best is None or r.fun < best.fun: best = r
        self.params_, self.nll_, self.success_ = best.x, float(best.fun), bool(best.success)
        self.n_params_ = len(best.x)
        return self

    def predict_proba(self, ta, tb): return self._p(ta, tb, self.params_)
    @property
    def params_dict(self): return {k: round(float(v), 4) for k, v in zip(self.param_names_, self.params_)}


class DiscountingModel:
    """Classical intertemporal-choice baselines predicting P(choose larger-later)."""
    SPECS = {
        "exponential":      (["theta", "k"], [0.5, 0.01], [(1e-4, 50.), (1e-9, 10.)]),
        "hyperbolic":       (["theta", "k"], [0.5, 0.05], [(1e-4, 50.), (1e-9, 100.)]),
        # beta is allowed above 1 so that a FUTURE-biased fit is representable rather than
        # being silently pinned at the boundary and misread as "no present bias".
        "quasi_hyperbolic": (["theta", "k", "beta"], [0.5, 0.005, 0.9],
                             [(1e-4, 50.), (1e-9, 10.), (0.02, 4.0)]),
    }
    def __init__(self, kind="hyperbolic"):
        self.kind = kind
        self.param_names_, self._init, self._bounds = self.SPECS[kind]
        self.params_ = None

    def _discount(self, amount, delay, q):
        if self.kind == "exponential":  return amount * np.exp(-q[1] * delay)
        if self.kind == "hyperbolic":   return amount / (1. + q[1] * delay)
        return amount * np.where(delay > 0, q[2], 1.) * np.exp(-q[1] * delay)

    def _p_ll(self, X, q):
        v_ss = self._discount(X[:, 0], X[:, 1], q)
        v_ll = self._discount(X[:, 2], X[:, 3], q)
        scale = np.maximum(np.abs(X[:, 0]), 1.0)     # amounts differ wildly across studies
        return expit(q[0] * (v_ll - v_ss) / scale)

    def fit(self, X, y, n_restarts=2, seed=RANDOM_SEED):
        X = np.asarray(X, float); y = np.asarray(y, float)
        rng = np.random.default_rng(seed)
        lo = np.array([b[0] for b in self._bounds]); hi = np.array([b[1] for b in self._bounds])
        def nll(q):
            p = np.clip(self._p_ll(X, q), 1e-9, 1 - 1e-9)
            return -np.sum(y * np.log(p) + (1 - y) * np.log(1 - p))
        best = None
        starts = [np.array(self._init, float)]
        for _ in range(max(0, n_restarts - 1)):
            starts.append(np.clip(np.array(self._init) * rng.uniform(.3, 3., len(self._init)), lo, hi))
        for s in starts:
            r = minimize(nll, np.clip(s, lo, hi), bounds=self._bounds,
                         method="L-BFGS-B", options={"maxiter": 400})
            if best is None or r.fun < best.fun: best = r
        self.params_, self.nll_, self.success_ = best.x, float(best.fun), bool(best.success)
        self.n_params_ = len(best.x)
        return self

    def predict_proba(self, X): return self._p_ll(np.asarray(X, float), self.params_)

    @property
    def params_dict(self): return {k: round(float(v), 6) for k, v in zip(self.param_names_, self.params_)}

    @property
    def on_boundary(self) -> list[str]:
        """Parameters resting on a search bound. Such an estimate is NOT identified by
        the data: the optimiser wanted to keep going and was stopped by the box."""
        out = []
        for nm, v, (lo, hi) in zip(self.param_names_, self.params_, self._bounds):
            if abs(v - lo) <= 1e-6 * max(1.0, abs(lo)) or abs(v - hi) <= 1e-6 * max(1.0, abs(hi)):
                out.append(nm)
        return out

ITC_MODEL_COLS = ["ss_value", "ss_time", "ll_value", "ll_time"]
print("behavioural model classes defined")

In [ ]:
# --- 16.2 Fit the risky-choice ladder (RISK track) ----------------------------------
BE_RESULTS = []
BEHAV_MODELS = {}

if AVAILABLE["choices13k"]:
    sp = SPLITS["RISK"]; tr, va, te = sp["train"], sp["val"], sp["test"]
    Atr, Btr = TA.subset(tr), TB.subset(tr)
    for kind in ["ev", "eu_crra", "cpt"]:
        t0 = time.time()
        m = RiskyChoiceModel(kind).fit(Atr, Btr, RISK_k[tr], RISK_n[tr])
        BEHAV_MODELS[f"risk_{kind}"] = m
        row = dict(track="RISK", model=f"BE:{kind}", family="behavioural",
                   n_params=m.n_params_, converged=m.success_,
                   seconds=round(time.time() - t0, 1))
        for split_name, ix in [("val", va), ("test", te)]:
            p = m.predict_proba(TA.subset(ix), TB.subset(ix))
            row[f"{split_name}_mse_rate"] = round(float(np.mean((p - RISK_y[ix]) ** 2)), 5)
            row[f"{split_name}_logloss"] = round(binom_nll(p, RISK_k[ix], RISK_n[ix]) / RISK_n[ix].sum(), 5)
        row["params"] = json.dumps(m.params_dict)
        BE_RESULTS.append(row)
        print(f"  {kind:9s} {m.params_dict}  test MSE={row['test_mse_rate']:.5f}  "
              f"test logloss={row['test_logloss']:.5f}  ({row['seconds']}s)")

    cpt = BEHAV_MODELS["risk_cpt"]
    p = cpt.params_dict
    print(f"\nEstimated prospect-theory parameters (pooled over the training split):")
    print(f"  alpha  (gain curvature)      {p['alpha']:.3f}   <1 = diminishing sensitivity to gains")
    print(f"  beta   (loss curvature)      {p['beta']:.3f}")
    print(f"  lambda (LOSS AVERSION)       {p['lambda']:.3f}   >1 = losses loom larger than gains")
    print(f"  gamma_gain (prob weighting)  {p['gamma_gain']:.3f}   <1 = inverse-S, small p overweighted")
    print(f"  gamma_loss (prob weighting)  {p['gamma_loss']:.3f}")

In [ ]:
# --- 16.3 Fit the discounting ladder (ITC track) ------------------------------------
if AVAILABLE["itc_database"]:
    sp = SPLITS["ITC"]; tr, va, te = sp["train"], sp["val"], sp["test"]
    Xd = itc[ITC_MODEL_COLS].to_numpy(float)
    for kind in ["exponential", "hyperbolic", "quasi_hyperbolic"]:
        t0 = time.time()
        m = DiscountingModel(kind).fit(Xd[tr], ITC_y[tr])
        BEHAV_MODELS[f"itc_{kind}"] = m
        row = dict(track="ITC", model=f"BE:{kind}", family="behavioural",
                   n_params=m.n_params_, converged=m.success_, seconds=round(time.time() - t0, 1))
        for split_name, ix in [("val", va), ("test", te)]:
            p = np.clip(m.predict_proba(Xd[ix]), 1e-9, 1 - 1e-9)
            row[f"{split_name}_logloss"] = round(float(log_loss(ITC_y[ix], p, labels=[0, 1])), 5)
            row[f"{split_name}_auc"] = round(float(roc_auc_score(ITC_y[ix], p)), 4)
            row[f"{split_name}_brier"] = round(float(brier_score_loss(ITC_y[ix], p)), 5)
            row[f"{split_name}_accuracy"] = round(float(accuracy_score(ITC_y[ix], (p > .5).astype(int))), 4)
        row["params"] = json.dumps(m.params_dict)
        row["on_boundary"] = ",".join(m.on_boundary)
        BE_RESULTS.append(row)
        print(f"  {kind:17s} {m.params_dict}  test logloss={row['test_logloss']:.4f}  "
              f"AUC={row['test_auc']:.4f}  ({row['seconds']}s)"
              + (f"  [!] on bound: {m.on_boundary}" if m.on_boundary else ""))

    qh_model = BEHAV_MODELS["itc_quasi_hyperbolic"]
    qh = qh_model.params_dict
    # beta is only separately identified when some trials put a FRONT-END DELAY on the
    # sooner option; with ss_time == 0 everywhere, beta and the softmax scale trade off.
    front_end = float((itc.ss_time.to_numpy(float)[SPLITS["ITC"]["train"]] > 0).mean())
    print(f"\nPooled quasi-hyperbolic estimates:")
    print(f"  k    (delay discount rate)   {qh['k']:.6f} per day")
    print(f"  beta                         {qh['beta']:.4f}")
    if qh_model.on_boundary:
        print(f"  [!] NOT IDENTIFIED: {qh_model.on_boundary} rest on a search bound.")
        print("      Do not interpret a boundary estimate as a behavioural finding.")
    elif qh["beta"] < 0.98:
        print("      beta < 1 => PRESENT BIAS: an extra penalty on any delay, over and")
        print("      above smooth discounting.")
    elif qh["beta"] > 1.02:
        print("      beta > 1 => the pooled fit shows NO present bias (if anything the")
        print("      opposite). Reported as found; this is a real negative result.")
    else:
        print("      beta ~ 1 => approximately time-consistent at the pooled level.")
    print(f"  trials with a front-end delay (ss_time > 0): {front_end*100:.1f}%")
    print("      beta is only separately identified from k when this share is non-trivial;")
    print("      a pooled fit across 100 heterogeneous protocols is a weak place to read it.")
    print("      The per-participant fits in Section 22 are the better estimate.")
    hyp, exp_ = BEHAV_MODELS["itc_hyperbolic"], BEHAV_MODELS["itc_exponential"]
    print(f"\n  Model comparison by AIC (lower is better), fitted on {len(tr):,} training trials:")
    for nm, mm in [("exponential", exp_), ("hyperbolic", hyp),
                   ("quasi-hyperbolic", BEHAV_MODELS['itc_quasi_hyperbolic'])]:
        print(f"    {nm:18s} AIC={2*mm.n_params_ + 2*mm.nll_:,.1f}")

BE_TABLE = pd.DataFrame(BE_RESULTS)
BE_TABLE.to_csv(OUTPUTS / "behavioral_model_results.csv", index=False)
display(BE_TABLE)

## 17–19. Machine-learning models, and hyper-parameter optimisation

Models are chosen for the shape of each dataset rather than for a longer list:

* **RISK** — the target is a *rate* backed by `n` respondents, so regressors are trained
  with `sample_weight = n` (a problem answered by 30 people should count more than one
  answered by 15). Evaluation still uses proper probabilistic scores against the binomial
  counts.
* **ITC** — a large binary-classification problem with categorical protocol moderators,
  so gradient boosting over mixed types is the natural fit.

A **hybrid model** is also fitted: the classical behavioural model's prediction is added
as a single extra feature. This tests whether theory and ML are complementary rather than
substitutes — the "cognitive model prior" idea from Bourgin et al. (2019).

Optuna searches hyper-parameters **against the validation split only**. The test split is
touched exactly once, at the end. This is enforced by construction: no function in this
section ever receives the test indices.

In [ ]:
# --- 17.1 Shared preprocessing + model zoo ------------------------------------------
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

def make_preprocessor(X: pd.DataFrame) -> ColumnTransformer:
    num = X.select_dtypes(include=[np.number]).columns.tolist()
    cat = [c for c in X.columns if c not in num]
    return ColumnTransformer(
        [("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                           ("sc", StandardScaler())]), num),
         ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                           ("oh", OneHotEncoder(handle_unknown="ignore", min_frequency=25,
                                                sparse_output=False))]), cat)],
        remainder="drop", verbose_feature_names_out=False)

def to_numeric_matrix(X: pd.DataFrame) -> pd.DataFrame:
    """Tree models handle categoricals as ordinal codes; NaNs are preserved for
    HistGradientBoosting / XGBoost / LightGBM which split on missingness natively."""
    out = X.copy()
    for c in out.columns:
        if not pd.api.types.is_numeric_dtype(out[c]):
            out[c] = out[c].astype("category").cat.codes.replace(-1, np.nan)
    return out.astype(float)

def clf_zoo(seed=RANDOM_SEED) -> dict:
    z = {
        "Majority": DummyClassifier(strategy="prior"),
        "LogisticRegression": Pipeline([("pp", "PLACEHOLDER"),
                                        ("m", LogisticRegression(max_iter=2000, C=1.0))]),
        "RandomForest": RandomForestClassifier(n_estimators=300, min_samples_leaf=20,
                                               n_jobs=-1, random_state=seed),
        "GradientBoosting": HistGradientBoostingClassifier(max_iter=250, learning_rate=.08,
                                                           random_state=seed),
        "MLP": Pipeline([("pp", "PLACEHOLDER"),
                         ("m", MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=120,
                                             early_stopping=True, random_state=seed))]),
    }
    if xgb is not None:
        z["XGBoost"] = xgb.XGBClassifier(n_estimators=400, max_depth=6, learning_rate=.06,
                                         subsample=.85, colsample_bytree=.85,
                                         eval_metric="logloss", tree_method="hist",
                                         random_state=seed, n_jobs=-1)
    if lgb is not None:
        z["LightGBM"] = lgb.LGBMClassifier(n_estimators=500, learning_rate=.05, num_leaves=63,
                                           subsample=.85, colsample_bytree=.85,
                                           random_state=seed, n_jobs=-1, verbose=-1)
    return z

def reg_zoo(seed=RANDOM_SEED) -> dict:
    z = {
        "MeanPredictor": DummyRegressor(strategy="mean"),
        "LinearRegression": Pipeline([("pp", "PLACEHOLDER"), ("m", LinearRegression())]),
        "Ridge": Pipeline([("pp", "PLACEHOLDER"), ("m", Ridge(alpha=1.0))]),
        "Lasso": Pipeline([("pp", "PLACEHOLDER"), ("m", Lasso(alpha=1e-3, max_iter=5000))]),
        "RandomForest": RandomForestRegressor(n_estimators=300, min_samples_leaf=5,
                                              n_jobs=-1, random_state=seed),
        "GradientBoosting": HistGradientBoostingRegressor(max_iter=400, learning_rate=.06,
                                                          random_state=seed),
        "MLP": Pipeline([("pp", "PLACEHOLDER"),
                         ("m", MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=200,
                                            early_stopping=True, random_state=seed))]),
    }
    if xgb is not None:
        z["XGBoost"] = xgb.XGBRegressor(n_estimators=600, max_depth=6, learning_rate=.05,
                                        subsample=.85, colsample_bytree=.85,
                                        tree_method="hist", random_state=seed, n_jobs=-1)
    if lgb is not None:
        z["LightGBM"] = lgb.LGBMRegressor(n_estimators=700, learning_rate=.05, num_leaves=63,
                                          subsample=.85, colsample_bytree=.85,
                                          random_state=seed, n_jobs=-1, verbose=-1)
    return z

def bind_preprocessor(model, X):
    if isinstance(model, Pipeline) and model.steps[0][1] == "PLACEHOLDER":
        model.steps[0] = ("pp", make_preprocessor(X))
    return model

print("model zoos defined")

In [ ]:
# --- 17.2 RISK track: train the model zoo (rate regression, weighted by n) -----------
ML_RESULTS = []
FITTED = {}

def eval_rate_model(pred, ix):
    p = np.clip(pred, 1e-6, 1 - 1e-6)
    return dict(mse_rate=float(np.mean((p - RISK_y[ix]) ** 2)),
                mae_rate=float(np.mean(np.abs(p - RISK_y[ix]))),
                r2_rate=float(r2_score(RISK_y[ix], p)),
                logloss=float(binom_nll(p, RISK_k[ix], RISK_n[ix]) / RISK_n[ix].sum()),
                brier=float(np.average((p - RISK_y[ix]) ** 2, weights=RISK_n[ix])))

if AVAILABLE["choices13k"]:
    sp = SPLITS["RISK"]; tr, va, te = sp["train"], sp["val"], sp["test"]
    Xr = to_numeric_matrix(RISK_X)

    # hybrid feature: the fitted CPT model's own prediction (a "cognitive model prior")
    cpt_pred_all = BEHAV_MODELS["risk_cpt"].predict_proba(TA, TB)
    Xr_hyb = Xr.copy(); Xr_hyb["cpt_prediction"] = cpt_pred_all

    for name, mdl in reg_zoo().items():
        for tag, XX in [("", Xr), ("+CPT", Xr_hyb)]:
            if tag and name in ("MeanPredictor",):
                continue
            m = bind_preprocessor(clone(mdl) if not isinstance(mdl, Pipeline) else clone(mdl), XX)
            t0 = time.time()
            try:
                m.fit(XX.iloc[tr], RISK_y[tr], **({"sample_weight": RISK_n[tr]}
                      if name not in ("MLP", "LinearRegression", "Ridge", "Lasso") else {}))
            except TypeError:
                m.fit(XX.iloc[tr], RISK_y[tr])
            row = dict(track="RISK", model=f"{name}{tag}", family="ml",
                       seconds=round(time.time() - t0, 1))
            for sn, ix in [("val", va), ("test", te)]:
                for k, v in eval_rate_model(m.predict(XX.iloc[ix]), ix).items():
                    row[f"{sn}_{k}"] = round(v, 5)
            ML_RESULTS.append(row); FITTED[f"RISK::{name}{tag}"] = (m, XX)
            print(f"  {name+tag:24s} val MSE={row['val_mse_rate']:.5f}  "
                  f"test MSE={row['test_mse_rate']:.5f}  ({row['seconds']}s)")

In [ ]:
# --- 17.3 ITC track: train the model zoo (binary classification) --------------------
def eval_clf(p, y):
    p = np.clip(p, 1e-6, 1 - 1e-6); yh = (p > .5).astype(int)
    return dict(accuracy=float(accuracy_score(y, yh)),
                precision=float(precision_score(y, yh, zero_division=0)),
                recall=float(recall_score(y, yh, zero_division=0)),
                f1=float(f1_score(y, yh, zero_division=0)),
                roc_auc=float(roc_auc_score(y, p)) if len(np.unique(y)) > 1 else np.nan,
                pr_auc=float(average_precision_score(y, p)),
                logloss=float(log_loss(y, p, labels=[0, 1])),
                brier=float(brier_score_loss(y, p)))

def predict_proba_any(m, X):
    if hasattr(m, "predict_proba"): return m.predict_proba(X)[:, 1]
    return np.clip(m.predict(X), 1e-6, 1 - 1e-6)

if AVAILABLE["itc_database"]:
    sp = SPLITS["ITC"]; tr, va, te = sp["train"], sp["val"], sp["test"]
    Xi = to_numeric_matrix(ITC_X)
    hyb_pred = BEHAV_MODELS["itc_quasi_hyperbolic"].predict_proba(itc[ITC_MODEL_COLS].to_numpy(float))
    Xi_hyb = Xi.copy(); Xi_hyb["betadelta_prediction"] = hyb_pred

    for name, mdl in clf_zoo().items():
        for tag, XX in [("", Xi), ("+BD", Xi_hyb)]:
            if tag and name == "Majority":
                continue
            m = bind_preprocessor(clone(mdl), XX)
            t0 = time.time()
            m.fit(XX.iloc[tr], ITC_y[tr])
            row = dict(track="ITC", model=f"{name}{tag}", family="ml",
                       seconds=round(time.time() - t0, 1))
            for sn, ix in [("val", va), ("test", te)]:
                for k, v in eval_clf(predict_proba_any(m, XX.iloc[ix]), ITC_y[ix]).items():
                    row[f"{sn}_{k}"] = round(v, 5)
            ML_RESULTS.append(row); FITTED[f"ITC::{name}{tag}"] = (m, XX)
            print(f"  {name+tag:24s} val logloss={row['val_logloss']:.4f}  "
                  f"test logloss={row['test_logloss']:.4f}  AUC={row['test_roc_auc']:.4f}  "
                  f"({row['seconds']}s)")

In [ ]:
# --- 19.1 Optuna hyper-parameter optimisation (validation split only) ---------------
def tune_risk(n_trials: int) -> tuple[Any, dict]:
    sp = SPLITS["RISK"]; tr, va = sp["train"], sp["val"]
    X = FITTED.get("RISK::GradientBoosting+CPT", (None, None))[1]
    if X is None: X = to_numeric_matrix(RISK_X)
    def objective(t):
        params = dict(
            max_iter=t.suggest_int("max_iter", 200, 900),
            learning_rate=t.suggest_float("learning_rate", .01, .2, log=True),
            max_leaf_nodes=t.suggest_int("max_leaf_nodes", 15, 127),
            min_samples_leaf=t.suggest_int("min_samples_leaf", 5, 80),
            l2_regularization=t.suggest_float("l2_regularization", 1e-4, 5., log=True),
            max_features=t.suggest_float("max_features", .4, 1.),
        )
        m = HistGradientBoostingRegressor(random_state=RANDOM_SEED, **params)
        m.fit(X.iloc[tr], RISK_y[tr], sample_weight=RISK_n[tr])
        p = np.clip(m.predict(X.iloc[va]), 1e-6, 1 - 1e-6)
        return binom_nll(p, RISK_k[va], RISK_n[va]) / RISK_n[va].sum()
    st = optuna.create_study(direction="minimize",
                             sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED))
    st.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    best = HistGradientBoostingRegressor(random_state=RANDOM_SEED, **st.best_params)
    best.fit(X.iloc[tr], RISK_y[tr], sample_weight=RISK_n[tr])
    return (best, X), st.best_params

def tune_itc(n_trials: int) -> tuple[Any, dict]:
    sp = SPLITS["ITC"]; tr, va = sp["train"], sp["val"]
    X = FITTED.get("ITC::GradientBoosting+BD", (None, None))[1]
    if X is None: X = to_numeric_matrix(ITC_X)
    def objective(t):
        params = dict(
            max_iter=t.suggest_int("max_iter", 120, 600),
            learning_rate=t.suggest_float("learning_rate", .02, .25, log=True),
            max_leaf_nodes=t.suggest_int("max_leaf_nodes", 15, 127),
            min_samples_leaf=t.suggest_int("min_samples_leaf", 20, 400),
            l2_regularization=t.suggest_float("l2_regularization", 1e-4, 5., log=True),
        )
        m = HistGradientBoostingClassifier(random_state=RANDOM_SEED, **params)
        m.fit(X.iloc[tr], ITC_y[tr])
        return float(log_loss(ITC_y[va], m.predict_proba(X.iloc[va])[:, 1], labels=[0, 1]))
    st = optuna.create_study(direction="minimize",
                             sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED))
    st.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    best = HistGradientBoostingClassifier(random_state=RANDOM_SEED, **st.best_params)
    best.fit(X.iloc[tr], ITC_y[tr])
    return (best, X), st.best_params

TUNED = {}
if AVAILABLE["choices13k"]:
    t0 = time.time(); (m, X), bp = tune_risk(CONFIG.optuna_trials)
    TUNED["RISK"] = (m, X); FITTED["RISK::Tuned-HGB+CPT"] = (m, X)
    sp = SPLITS["RISK"]
    row = dict(track="RISK", model="Tuned-HGB+CPT", family="ml_tuned",
               seconds=round(time.time() - t0, 1), params=json.dumps(bp))
    for sn, ix in [("val", sp["val"]), ("test", sp["test"])]:
        for k, v in eval_rate_model(m.predict(X.iloc[ix]), ix).items():
            row[f"{sn}_{k}"] = round(v, 5)
    ML_RESULTS.append(row)
    print(f"RISK best params: {bp}")
    print(f"  -> test MSE {row['test_mse_rate']:.5f} | test logloss {row['test_logloss']:.5f}")

if AVAILABLE["itc_database"]:
    t0 = time.time(); (m, X), bp = tune_itc(CONFIG.optuna_trials)
    TUNED["ITC"] = (m, X); FITTED["ITC::Tuned-HGB+BD"] = (m, X)
    sp = SPLITS["ITC"]
    row = dict(track="ITC", model="Tuned-HGB+BD", family="ml_tuned",
               seconds=round(time.time() - t0, 1), params=json.dumps(bp))
    for sn, ix in [("val", sp["val"]), ("test", sp["test"])]:
        for k, v in eval_clf(m.predict_proba(X.iloc[ix])[:, 1], ITC_y[ix]).items():
            row[f"{sn}_{k}"] = round(v, 5)
    ML_RESULTS.append(row)
    print(f"ITC best params: {bp}")
    print(f"  -> test logloss {row['test_logloss']:.4f} | AUC {row['test_roc_auc']:.4f}")

## 20. Model evaluation — and the headline comparison

The question the brief calls out as a major result: **does ML actually beat classical
behavioural-economic theory at predicting choice?** Both families are now scored on the
same held-out test split with the same metrics.

In [ ]:
# --- 20.1 Combined comparison table -------------------------------------------------
COMPARISON = pd.concat([BE_TABLE, pd.DataFrame(ML_RESULTS)], ignore_index=True)
COMPARISON.to_csv(OUTPUTS / "model_comparison.csv", index=False)

for track, sort_key in [("RISK", "test_mse_rate"), ("ITC", "test_logloss")]:
    sub = COMPARISON[COMPARISON.track == track].copy()
    if not len(sub) or sort_key not in sub: continue
    cols = ["model", "family", "n_params", "seconds", "val_" + sort_key.split("test_")[1],
            sort_key] + [c for c in ["test_logloss", "test_roc_auc", "test_brier",
                                      "test_accuracy", "test_r2_rate"] if c in sub and c != sort_key]
    cols = [c for c in dict.fromkeys(cols) if c in sub.columns]
    print(f"\n{'='*90}\n{track} track — ranked by {sort_key} (lower is better)\n{'='*90}")
    display(sub.sort_values(sort_key)[cols].reset_index(drop=True))

In [ ]:
# --- 20.2 Headline: behavioural theory vs machine learning --------------------------
HEADLINE = {}

def _best(track, key, family):
    s = COMPARISON[(COMPARISON.track == track) & (COMPARISON.family.str.startswith(family))]
    s = s.dropna(subset=[key])
    return s.loc[s[key].idxmin()] if len(s) else None

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
for ax, (track, key, label) in zip(axes, [("RISK", "test_mse_rate", "test MSE on choice rate"),
                                          ("ITC", "test_logloss", "test log loss")]):
    sub = COMPARISON[COMPARISON.track == track].dropna(subset=[key]).sort_values(key)
    if not len(sub):
        ax.axis("off"); continue
    colors = ["#C44E52" if f == "behavioural" else ("#DD8452" if f == "ml_tuned" else "#4C72B0")
              for f in sub.family]
    ax.barh(range(len(sub)), sub[key], color=colors)
    ax.set_yticks(range(len(sub))); ax.set_yticklabels(sub.model, fontsize=8)
    ax.invert_yaxis(); ax.set(xlabel=label, title=f"{track} track")

    be, ml = _best(track, key, "behavioural"), _best(track, key, "ml")
    if be is not None and ml is not None:
        imp = 100 * (be[key] - ml[key]) / be[key]
        HEADLINE[track] = dict(metric=key, best_behavioural=be.model, behavioural_score=float(be[key]),
                               best_ml=ml.model, ml_score=float(ml[key]),
                               improvement_pct=round(float(imp), 2))
        ax.axvline(be[key], ls="--", c="#C44E52", lw=1)
        ax.text(.98, .04, f"ML improves on best theory by {imp:.1f}%", transform=ax.transAxes,
                ha="right", fontsize=9, bbox=dict(fc="white", ec="grey", alpha=.85))
handles = [matplotlib.patches.Patch(color=c, label=l) for c, l in
           [("#C44E52", "behavioural theory"), ("#4C72B0", "ML"), ("#DD8452", "ML (tuned)")]]
axes[0].legend(handles=handles, loc="lower right", fontsize=8)
savefig("model_comparison"); plt.show()

print(json.dumps(HEADLINE, indent=2))

In [ ]:
# --- 20.3 ROC, PR, calibration, confusion matrix, learning curve --------------------
if AVAILABLE["itc_database"]:
    m, X = TUNED["ITC"]; te = SPLITS["ITC"]["test"]
    p_ml = m.predict_proba(X.iloc[te])[:, 1]
    p_be = BEHAV_MODELS["itc_quasi_hyperbolic"].predict_proba(
        itc[ITC_MODEL_COLS].to_numpy(float)[te])
    y = ITC_y[te]

    fig, ax = plt.subplots(2, 3, figsize=(16, 9)); a = ax.ravel()
    for nm, pp, c in [("Tuned ML", p_ml, "#4C72B0"), ("beta-delta theory", p_be, "#C44E52")]:
        fpr, tpr, _ = roc_curve(y, pp)
        a[0].plot(fpr, tpr, c=c, label=f"{nm} (AUC={roc_auc_score(y, pp):.3f})")
        pr, rc, _ = precision_recall_curve(y, pp)
        a[1].plot(rc, pr, c=c, label=f"{nm} (AP={average_precision_score(y, pp):.3f})")
        xs, ys = calibration_curve(y, np.clip(pp, 1e-6, 1-1e-6), n_bins=15, strategy="quantile")
        a[2].plot(ys, xs, "o-", c=c, label=f"{nm} (Brier={brier_score_loss(y, pp):.4f})")
    a[0].plot([0, 1], [0, 1], "k--", lw=.8); a[0].set(title="ROC", xlabel="FPR", ylabel="TPR"); a[0].legend()
    a[1].axhline(y.mean(), ls="--", c="k", lw=.8)
    a[1].set(title="Precision-Recall", xlabel="recall", ylabel="precision"); a[1].legend()
    a[2].plot([0, 1], [0, 1], "k--", lw=.8)
    a[2].set(title="Calibration", xlabel="predicted P(LL)", ylabel="observed P(LL)"); a[2].legend()

    cm = confusion_matrix(y, (p_ml > .5).astype(int))
    sns.heatmap(cm, annot=True, fmt=",d", cmap="Blues", ax=a[3],
                xticklabels=["pred SS", "pred LL"], yticklabels=["true SS", "true LL"])
    a[3].set(title="Confusion matrix (tuned ML)")

    # learning curve by number of training PARTICIPANTS (the meaningful axis here)
    tr = SPLITS["ITC"]["train"]; gtr = ITC_groups[tr]; ug = np.unique(gtr)
    fracs = [.05, .1, .2, .4, .7, 1.0]; lc = []
    for f in fracs:
        keep = set(ug[:max(2, int(f * len(ug)))])
        sub = tr[np.isin(gtr, list(keep))]
        mm = HistGradientBoostingClassifier(max_iter=150, random_state=RANDOM_SEED).fit(
            X.iloc[sub], ITC_y[sub])
        lc.append((len(keep), log_loss(ITC_y[tr], mm.predict_proba(X.iloc[tr])[:, 1], labels=[0,1]),
                   log_loss(y, mm.predict_proba(X.iloc[te])[:, 1], labels=[0, 1])))
    lc = np.array(lc)
    a[4].plot(lc[:, 0], lc[:, 1], "o-", label="train"); a[4].plot(lc[:, 0], lc[:, 2], "o-", label="test")
    a[4].set(title="Learning curve", xlabel="training participants", ylabel="log loss"); a[4].legend()
    a[4].set_xscale("log")

    resid = y - p_ml
    a[5].scatter(p_ml, resid, s=2, alpha=.08, color="#55A868")
    a[5].axhline(0, c="k", lw=.8)
    a[5].set(title="Residuals vs prediction", xlabel="predicted P(LL)", ylabel="residual")
    savefig("evaluation_itc"); plt.show()

if AVAILABLE["choices13k"]:
    m, X = TUNED["RISK"]; te = SPLITS["RISK"]["test"]
    p_ml = np.clip(m.predict(X.iloc[te]), 1e-6, 1 - 1e-6)
    p_be = BEHAV_MODELS["risk_cpt"].predict_proba(TA.subset(te), TB.subset(te))
    fig, ax = plt.subplots(1, 3, figsize=(16, 4.6))
    for nm, pp, c in [("Tuned ML", p_ml, "#4C72B0"), ("CPT", p_be, "#C44E52")]:
        ax[0].scatter(pp, RISK_y[te], s=3, alpha=.12, color=c, label=nm)
        xs, ys = calibration_curve((RISK_y[te] > .5).astype(int), pp, n_bins=15, strategy="quantile")
        ax[1].plot(ys, xs, "o-", c=c, label=f"{nm}")
        ax[2].hist(pp - RISK_y[te], bins=50, alpha=.55, label=f"{nm} (MSE={np.mean((pp-RISK_y[te])**2):.4f})")
    ax[0].plot([0, 1], [0, 1], "k--", lw=.8)
    ax[0].set(title="Predicted vs observed choice rate", xlabel="predicted", ylabel="observed"); ax[0].legend()
    ax[1].plot([0, 1], [0, 1], "k--", lw=.8); ax[1].set(title="Calibration"); ax[1].legend()
    ax[2].set(title="Prediction error", xlabel="predicted - observed"); ax[2].legend()
    savefig("evaluation_risk"); plt.show()

## 21. SHAP explainability

SHAP attributions are computed on the tuned tree models and read back in
behavioural-economic language rather than raw feature names.

In [ ]:
# --- 21.1 SHAP global + local -------------------------------------------------------
BEHAVIOURAL_GLOSSARY = {
    "delay": "length of the wait before the larger reward arrives",
    "log_delay": "wait length (log scale — psychological time is compressive)",
    "amount_ratio": "how much bigger the later reward is (ll/ss)",
    "rel_amount_gain": "proportional premium for waiting",
    "k_indiff_hyperbolic": "implied hyperbolic discount rate at indifference",
    "annual_implied_rate": "annualised interest rate the offer implicitly pays",
    "ss_value": "size of the immediate reward",
    "ll_value": "size of the delayed reward",
    "is_immediate_ss": "whether the sooner option is available RIGHT NOW (present bias trigger)",
    "hist_ll_rate": "the participant's own previously observed patience",
    "hist_prev_choice": "what this participant chose on their previous trial",
    "betadelta_prediction": "the quasi-hyperbolic theory's own prediction",
    "ev_diff": "expected-value advantage of option B",
    "sd_diff": "extra outcome variability (risk) carried by option B",
    "p_loss_b": "probability that option B loses money",
    "gain_loss_ratio_b": "gains relative to losses within option B",
    "rare_event_b": "whether option B hinges on a rare event",
    "feedback": "whether outcomes were experienced with feedback vs merely described",
    "cpt_prediction": "cumulative prospect theory's own prediction",
    "complexity_diff": "how much more complex option B is than option A",
    "best_outcome_is_b": "whether B holds the single best possible outcome",
    "min_b": "the worst outcome option B can deliver",
    "max_b": "the best outcome option B can deliver",
    "amount_diff": "absolute extra money gained by waiting (ll - ss)",
    "hist_n_trials": "how many of this participant's earlier trials the model has seen",
    "hist_available": "whether any prior choice history exists for this participant",
    "log_k_indiff_hyp": "implied discount rate at indifference, log scale",
    "k_indiff_exponential": "implied exponential discount rate at indifference",
    "sv_diff_hyp_k0.01": "hyperbolic subjective-value gap at a patient reference rate",
    "sv_diff_hyp_k0.05": "hyperbolic subjective-value gap at a moderate reference rate",
    "sv_diff_hyp_k0.2": "hyperbolic subjective-value gap at an impatient reference rate",
    "sv_diff_exp_k0.01": "exponential subjective-value gap at a patient reference rate",
    "sv_diff_exp_k0.05": "exponential subjective-value gap at a moderate reference rate",
    "sv_diff_exp_k0.2": "exponential subjective-value gap at an impatient reference rate",
    "log_delay": "wait length on a log scale",
    "log_ss_value": "size of the immediate reward, log scale",
    "log_ll_value": "size of the delayed reward, log scale",
    "ss_time": "how long until the sooner reward arrives",
    "ll_time": "how long until the later reward arrives",
    "cat_fixed_attributes": "which attribute the experiment held fixed across trials",
    "cat_procedure": "elicitation procedure used by the source study",
    "cat_incentivization": "whether choices were hypothetical or paid",
    "cat_presentation": "how the offer was displayed to the participant",
    "cat_intervention": "any additional manipulation applied by the source study",
    "cat_country": "country in which the source study was run",
    "cat_currency": "currency of the reward amounts",
    "age": "participant age as recorded by the source study",
    "b_dominates_a": "option B is never worse than A (a normative reason to pick B)",
    "a_dominates_b": "option A is never worse than B (a normative reason to pick A)",
    "p_loss_diff": "extra chance of losing money if B is chosen",
    "n_out_b": "how many distinct outcomes option B has (complexity)",
    "n_out_a": "how many distinct outcomes option A has",
    "p_high_b": "probability of option B's high outcome",
    "p_high_a": "probability of option A's high outcome",
    "ambiguity": "whether the probabilities were hidden from the participant",
    "ev_a": "expected value of option A", "ev_b": "expected value of option B",
    "sd_b": "outcome variability (risk) of option B",
    "range_b": "spread between B's best and worst outcomes",
    "block": "which experimental block the problem appeared in",
    "amount_ratio": "the delayed reward as a multiple of the immediate one",
}

SHAP_STORE = {}

def run_shap(track: str, model, X: pd.DataFrame, ix: np.ndarray, title: str):
    if shap is None:
        print("shap unavailable — skipping"); return None
    n = min(CONFIG.shap_sample, len(ix))
    sample = X.iloc[np.random.default_rng(RANDOM_SEED).choice(ix, n, replace=False)]
    try:
        expl = shap.TreeExplainer(model)
        sv = expl.shap_values(sample)
        if isinstance(sv, list): sv = sv[1]
        if sv.ndim == 3: sv = sv[..., -1]
    except Exception as exc:
        print(f"SHAP failed for {track}: {exc}"); return None

    SHAP_STORE[track] = (sv, sample)
    imp = pd.Series(np.abs(sv).mean(0), index=sample.columns).sort_values(ascending=False)

    fig = plt.figure(figsize=(9, 6))
    shap.summary_plot(sv, sample, show=False, max_display=18)
    plt.title(f"SHAP summary — {title}", fontsize=11)
    savefig(f"shap_summary_{track}"); plt.show()

    fig, ax = plt.subplots(figsize=(9, 6))
    top = imp.head(18)[::-1]
    ax.barh(range(len(top)), top.values, color="#4C72B0")
    ax.set_yticks(range(len(top))); ax.set_yticklabels(top.index, fontsize=9)
    ax.set(xlabel="mean |SHAP value|", title=f"Global feature importance — {title}")
    savefig(f"shap_bar_{track}"); plt.show()

    for feat in imp.head(3).index:
        try:
            shap.dependence_plot(feat, sv, sample, show=False)
            plt.title(f"Dependence: {feat}")
            savefig(f"shap_dependence_{track}_{feat}"); plt.show()
        except Exception:
            plt.close()

    print(f"\nTop drivers ({title}), in behavioural terms:")
    for i, (f, v) in enumerate(imp.head(10).items(), 1):
        print(f"  {i:2d}. {f:28s} |SHAP|={v:.4f}   {BEHAVIOURAL_GLOSSARY.get(f, '')}")
    imp.head(30).rename("mean_abs_shap").to_frame().to_csv(OUTPUTS / f"shap_importance_{track}.csv")
    return imp

SHAP_IMP = {}
if AVAILABLE["itc_database"]:
    m, X = TUNED["ITC"]
    SHAP_IMP["ITC"] = run_shap("ITC", m, X, SPLITS["ITC"]["test"], "intertemporal choice")
if AVAILABLE["choices13k"]:
    m, X = TUNED["RISK"]
    SHAP_IMP["RISK"] = run_shap("RISK", m, X, SPLITS["RISK"]["test"], "risky choice")

In [ ]:
# --- 21.2 Individual decision explanations ------------------------------------------
def explain_decision(track: str, row_idx: int) -> str:
    if track not in SHAP_STORE and shap is None:
        return "SHAP unavailable"
    model, X = TUNED[track]
    x = X.iloc[[row_idx]]
    p = (model.predict_proba(x)[0, 1] if hasattr(model, "predict_proba")
         else float(np.clip(model.predict(x)[0], 0, 1)))
    try:
        sv = shap.TreeExplainer(model).shap_values(x)
        if isinstance(sv, list): sv = sv[1]
        sv = np.asarray(sv).reshape(-1)
    except Exception:
        sv = np.zeros(x.shape[1])
    order = np.argsort(-np.abs(sv))[:5]
    label = "choose the delayed reward" if track == "ITC" else "choose option B"
    out = [f"Prediction:\n  P({label}) = {p*100:.1f}%", "\nTop factors:"]
    for j in order:
        f = x.columns[j]
        direction = "increases" if sv[j] > 0 else "decreases"
        val = x.iloc[0, j]
        shown = f"{val:10.4g}" if pd.notna(val) else "       n/a"
        out.append(f"  {f:26s} = {shown}   {direction} the probability "
                   f"(SHAP {sv[j]:+.3f})")
        if f in BEHAVIOURAL_GLOSSARY:
            out.append(f"  {'':26s}   -> {BEHAVIOURAL_GLOSSARY[f]}")
    return "\n".join(out)

if AVAILABLE["itc_database"]:
    te = SPLITS["ITC"]["test"]
    for i in te[:2]:
        r = itc.iloc[i]
        print("=" * 74)
        print(f"Decision: ${r.ss_value:.2f} in {r.ss_time:.0f} days  vs  "
              f"${r.ll_value:.2f} in {r.ll_time:.0f} days")
        print(f"Actual choice: {'larger-later' if ITC_y[i] == 1 else 'smaller-sooner'}")
        print(explain_decision("ITC", i)); print()

## 22. Individual behavioural parameter estimation

For participants with enough repeated trials, a quasi-hyperbolic (β–δ) model is fitted
**separately per person**. This yields, for each participant, an estimated discount rate
`k` and a present-bias parameter `β`.

> **Read these carefully.** They are *estimated behavioural parameters* summarising how a
> person responded within one experimental protocol. They are **not** personality traits,
> they are **not** stable across contexts, and they are **not** a psychological diagnosis.
> Participants with too few trials, or who chose the same option every time, are reported
> as non-identified rather than being assigned a made-up value.

In [ ]:
# --- 22.1 Per-participant parameter estimation --------------------------------------
def fit_participant_parameters(df, groups, y, kind="quasi_hyperbolic",
                               min_trials=20, min_var=0.02):
    rows = []
    tmp = df[ITC_MODEL_COLS].to_numpy(float)
    order = pd.Series(groups)
    for pid, ii in order.groupby(order, observed=True).groups.items():
        ii = np.asarray(ii)
        yy = y[ii]
        rec = {"participant_id": pid, "n_trials": len(ii), "ll_rate": float(yy.mean())}
        if len(ii) < min_trials or yy.std() < min_var:
            rec.update(identified=False, reason=("too few trials" if len(ii) < min_trials
                                                 else "no choice variation (always same option)"))
        else:
            m = DiscountingModel(kind).fit(tmp[ii], yy, n_restarts=1)
            bound = m.on_boundary
            rec.update(identified=bool(m.success_) and not bound,
                       reason=("estimate on a search bound: " + ",".join(bound)) if bound else "",
                       on_boundary=",".join(bound), nll=m.nll_, **m.params_dict)
        rows.append(rec)
    return pd.DataFrame(rows)

if AVAILABLE["itc_database"]:
    t0 = time.time()
    tr_mask = np.zeros(len(itc), bool); tr_mask[SPLITS["ITC"]["train"]] = True
    PARAMS = fit_participant_parameters(itc, ITC_groups, ITC_y)
    PARAMS["present_bias"] = PARAMS.get("beta", pd.Series(np.nan, index=PARAMS.index)) < 1.0
    PARAMS.to_csv(OUTPUTS / "behavioral_parameters.csv", index=False)
    ok = PARAMS[PARAMS.identified == True]
    print(f"fitted {len(PARAMS):,} participants in {time.time()-t0:.0f}s; "
          f"{len(ok):,} identified ({100*len(ok)/len(PARAMS):.1f}%)")
    print("\nNon-identified reasons:")
    print(PARAMS.loc[PARAMS.identified != True, "reason"].value_counts().to_string())

    if len(ok):
        print(f"\nEstimated behavioural tendencies (identified participants only):")
        print(ok[["k", "beta", "ll_rate", "n_trials"]].describe().round(4).to_string())
        pb = float((ok.beta < 0.98).mean()); fb = float((ok.beta > 1.02).mean())
        print(f"\n  Distribution of estimated present-bias parameters "
              f"({len(ok):,} identified participants):")
        print(f"    beta < 0.98  (present-bias pattern)   {pb*100:5.1f}%")
        print(f"    beta > 1.02  (future-bias pattern)    {fb*100:5.1f}%")
        print(f"    beta ~ 1     (time-consistent)        {(1-pb-fb)*100:5.1f}%")
        print(f"  median k: {ok.k.median():.5f}/day   median beta: {ok.beta.median():.4f}")
        print("\n  Participants whose estimate landed on a search bound are counted as")
        print("  NOT identified and excluded from these shares, rather than being reported")
        print("  as if the boundary value were a measurement.")

        fig, ax = plt.subplots(1, 3, figsize=(16, 4.4))
        ax[0].hist(np.log10(ok.k.clip(1e-6, 10)), bins=45, color="#4C72B0", edgecolor="white")
        ax[0].set(title="Estimated discount rate", xlabel="log10(k) per day", ylabel="participants")
        ax[1].hist(ok.beta.clip(0, 1.5), bins=45, color="#C44E52", edgecolor="white")
        ax[1].axvline(1.0, ls="--", c="k"); ax[1].set(
            title="Present-bias parameter", xlabel="beta   (<1 = present bias)")
        ax[2].scatter(np.log10(ok.k.clip(1e-6, 10)), ok.ll_rate, s=6, alpha=.3, color="#55A868")
        ax[2].set(title="Discount rate vs observed patience", xlabel="log10(k)", ylabel="P(LL)")
        savefig("behavioral_parameters"); plt.show()

## 23. Causal inference

**Where the causal claim comes from.** In `choices13k`, 1,562 problems were presented
*both* with outcome feedback and without it. The gamble is identical across the two
presentations; only the information format changes. That gives a genuine **paired
experimental contrast** — the description–experience manipulation — and it is the one
place in this project where a causal statement is warranted.

Estimated here:

* **ATE** of feedback on the choice rate, with bootstrap confidence intervals
* **CATE** — for which *kinds* of problem is the effect largest? Behavioural theory makes
  a sharp, falsifiable prediction here: the description–experience gap should be
  **concentrated in problems that hinge on a rare event**, because rare events are
  overweighted when described but under-sampled when experienced. The code below tests
  that prediction and prints an explicit verdict. It is a genuine test, so the verdict may
  come back negative — and if it does, that is reported as found rather than reframed.
* **IPW and doubly-robust** estimates on the unpaired sample, as a sensitivity check
  against the paired estimate.

Everywhere else in this notebook — study-level moderators in the ITC Database, protocol
differences, demographics — the variation is **observational**, and is reported as
association only. Those are explicitly *not* causal.

In [ ]:
# --- 23.1 ATE of feedback (paired, within-problem randomised contrast) --------------
CAUSAL_ROWS = []

def bootstrap_ci(vals, stat=np.mean, n=None, alpha=.05, seed=RANDOM_SEED):
    n = n or CONFIG.n_bootstrap
    rng = np.random.default_rng(seed)
    v = np.asarray(vals, float)
    boots = np.array([stat(v[rng.integers(0, len(v), len(v))]) for _ in range(n)])
    return float(stat(v)), float(np.percentile(boots, 100 * alpha / 2)), \
           float(np.percentile(boots, 100 * (1 - alpha / 2)))

if AVAILABLE["choices13k"]:
    wide = (c13.pivot_table(index="Problem", columns="Feedback", values="bRate", aggfunc="mean")
            .dropna())
    wide.columns = ["described", "experienced"]
    wide["diff"] = wide.experienced - wide.described
    d = wide["diff"].to_numpy()

    ate, lo, hi = bootstrap_ci(d)
    t_stat, p_t = stats.ttest_rel(wide.experienced, wide.described)
    w_stat, p_w = stats.wilcoxon(wide.experienced, wide.described)
    cohen_dz = float(d.mean() / d.std(ddof=1))

    print("=" * 78)
    print("CAUSAL ESTIMATE — effect of outcome FEEDBACK (experience vs description)")
    print("=" * 78)
    print(f"  design            paired, within-problem; {len(wide):,} problems seen both ways")
    print(f"  ATE               {ate:+.4f} on P(choose B)   95% bootstrap CI [{lo:+.4f}, {hi:+.4f}]")
    print(f"  paired t-test     t={t_stat:.3f}, p={p_t:.3e}")
    print(f"  Wilcoxon signed-rank  W={w_stat:.0f}, p={p_w:.3e}   (no normality assumption)")
    print(f"  Cohen's dz        {cohen_dz:.3f}")
    print(f"\n  Interpretation: experiencing outcomes rather than reading them shifts the "
          f"choice rate\n  for option B by {ate*100:+.2f} percentage points on the same gambles.")
    CAUSAL_ROWS.append(dict(analysis="ATE feedback (paired)", track="RISK", estimate=ate,
                            ci_low=lo, ci_high=hi, p_value=float(p_w), n=len(wide),
                            method="paired bootstrap + Wilcoxon", causal=True))

    fig, ax = plt.subplots(1, 3, figsize=(16, 4.4))
    ax[0].scatter(wide.described, wide.experienced, s=5, alpha=.25, color="#4C72B0")
    ax[0].plot([0, 1], [0, 1], "k--", lw=.8)
    ax[0].set(title="Same gamble, two formats", xlabel="P(B) described", ylabel="P(B) experienced")
    ax[1].hist(d, bins=45, color="#C44E52", edgecolor="white")
    ax[1].axvline(0, c="k", ls="--"); ax[1].axvline(ate, c="#DD8452", lw=2, label=f"ATE={ate:+.4f}")
    ax[1].legend(); ax[1].set(title="Paired within-problem difference", xlabel="experienced - described")
    rng = np.random.default_rng(RANDOM_SEED)
    boots = [d[rng.integers(0, len(d), len(d))].mean() for _ in range(CONFIG.n_bootstrap)]
    ax[2].hist(boots, bins=45, color="#55A868", edgecolor="white")
    ax[2].axvline(0, c="k", ls="--"); ax[2].axvspan(lo, hi, alpha=.2, color="#DD8452")
    ax[2].set(title=f"Bootstrap distribution of the ATE ({CONFIG.n_bootstrap} resamples)",
              xlabel="ATE")
    savefig("causal_ate_feedback"); plt.show()

In [ ]:
# --- 23.2 CATE: for which problems is the feedback effect strongest? ----------------
if AVAILABLE["choices13k"]:
    first = c13.drop_duplicates("Problem").set_index("Problem")
    feats = RISK_X.copy(); feats["Problem"] = c13["Problem"].values
    pf = feats.drop_duplicates("Problem").set_index("Problem")
    cate_cols = ["ev_diff", "sd_b", "p_high_b", "rare_event_b", "p_loss_b", "n_out_b",
                 "ambiguity", "gain_loss_ratio_b", "range_b", "b_dominates_a"]
    Z = pf.loc[wide.index, cate_cols].astype(float)

    # causal-forest-style CATE: regress the paired treatment effect on problem features.
    # Because the contrast is randomised WITHIN problem, the paired difference is an
    # unbiased unit-level effect estimate, so this is a legitimate CATE model.
    cate_model = RandomForestRegressor(n_estimators=400, min_samples_leaf=25,
                                       random_state=RANDOM_SEED, n_jobs=-1,
                                       oob_score=True).fit(Z, wide["diff"].values)
    tau = cate_model.oob_prediction_
    print(f"CATE model out-of-bag R^2 = {cate_model.oob_score_:.4f}")
    print("  (a positive value means the effect is genuinely predictable from problem features)")

    imp = pd.Series(cate_model.feature_importances_, index=cate_cols).sort_values(ascending=False)
    print("\nWhat moderates the description-experience gap:")
    for f, v in imp.head(6).items(): print(f"  {f:22s} {v:.3f}")

    # --- the theory-driven test: rare events
    rare = Z["rare_event_b"].to_numpy().astype(bool)
    g1, g0 = wide["diff"].to_numpy()[rare], wide["diff"].to_numpy()[~rare]
    m1, l1, h1 = bootstrap_ci(g1); m0, l0, h0 = bootstrap_ci(g0)
    u, pu = stats.mannwhitneyu(g1, g0)
    pooled = np.sqrt(((len(g1)-1)*g1.var(ddof=1) + (len(g0)-1)*g0.var(ddof=1)) / (len(g1)+len(g0)-2))
    print(f"\n  THEORY TEST — is the gap concentrated in rare-event problems?")
    print(f"    rare-event problems     ATE={m1:+.4f}  CI[{l1:+.4f},{h1:+.4f}]  n={len(g1)}")
    print(f"    common-event problems   ATE={m0:+.4f}  CI[{l0:+.4f},{h0:+.4f}]  n={len(g0)}")
    print(f"    Mann-Whitney U p={pu:.3e}   Cohen's d={(m1-m0)/pooled:+.3f}")

    if m1 > m0 and pu < .05:
        verdict = ("SUPPORTED: the gap is significantly larger for rare-event problems, "
                   "as description-experience theory predicts.")
    elif m1 <= m0 and pu < .05:
        verdict = ("CONTRADICTED: the gap is significantly LARGER for common-event "
                   "problems -- the opposite of the theoretical prediction.")
    else:
        verdict = ("NOT SUPPORTED (inconclusive): the difference between rare- and "
                   "common-event problems is not statistically distinguishable at the 5% "
                   "level, so this dataset does not corroborate the prediction.")
    print(f"\n    VERDICT: {verdict}")
    print("    Reported as found. The aggregate feedback effect in the previous cell")
    print("    remains well identified regardless of how this moderator test lands.")
    CAUSAL_ROWS.append(dict(analysis="rare-event moderation hypothesis", track="RISK",
                            estimate=m1 - m0, ci_low=np.nan, ci_high=np.nan,
                            p_value=float(pu), n=len(wide),
                            method="rare vs common subgroup contrast; VERDICT: "
                                   + verdict.split(":")[0], causal=True))
    CAUSAL_ROWS.append(dict(analysis="CATE feedback | rare-event problems", track="RISK",
                            estimate=m1, ci_low=l1, ci_high=h1, p_value=float(pu),
                            n=len(g1), method="subgroup bootstrap", causal=True))
    CAUSAL_ROWS.append(dict(analysis="CATE feedback | common-event problems", track="RISK",
                            estimate=m0, ci_low=l0, ci_high=h0, p_value=float(pu),
                            n=len(g0), method="subgroup bootstrap", causal=True))

    fig, ax = plt.subplots(1, 3, figsize=(16, 4.4))
    ax[0].hist(tau, bins=45, color="#8172B2", edgecolor="white")
    ax[0].axvline(0, c="k", ls="--"); ax[0].axvline(tau.mean(), c="#DD8452", lw=2)
    ax[0].set(title="Heterogeneous treatment effects (OOB CATE)", xlabel="estimated effect")
    ax[1].barh(range(len(imp))[::-1], imp.values, color="#4C72B0")
    ax[1].set_yticks(range(len(imp))[::-1]); ax[1].set_yticklabels(imp.index, fontsize=8)
    ax[1].set(title="CATE moderators", xlabel="importance")
    ax[2].errorbar([0, 1], [m0, m1], yerr=[[m0-l0, m1-l1], [h0-m0, h1-m1]],
                   fmt="o", capsize=6, ms=9, color="#C44E52")
    ax[2].axhline(0, c="k", ls="--", lw=.8)
    ax[2].set_xticks([0, 1]); ax[2].set_xticklabels(["common event", "rare event"])
    ax[2].set(title="Feedback effect by rare-event structure", ylabel="ATE on P(B)")
    savefig("causal_cate"); plt.show()

In [ ]:
# --- 23.3 IPW / doubly-robust sensitivity check (unpaired sample) -------------------
if AVAILABLE["choices13k"]:
    T = RISK_X["feedback"].to_numpy(int)
    Y = RISK_y.copy()
    Xc = to_numeric_matrix(RISK_X.drop(columns=["feedback", "block"]))

    naive = Y[T == 1].mean() - Y[T == 0].mean()

    ps_model = HistGradientBoostingClassifier(max_iter=200, random_state=RANDOM_SEED).fit(Xc, T)
    e = np.clip(ps_model.predict_proba(Xc)[:, 1], .02, .98)
    w = np.where(T == 1, 1 / e, 1 / (1 - e))
    ipw = float(np.average(Y[T == 1], weights=w[T == 1]) - np.average(Y[T == 0], weights=w[T == 0]))

    mu1 = HistGradientBoostingRegressor(max_iter=250, random_state=RANDOM_SEED).fit(
        Xc[T == 1], Y[T == 1]).predict(Xc)
    mu0 = HistGradientBoostingRegressor(max_iter=250, random_state=RANDOM_SEED).fit(
        Xc[T == 0], Y[T == 0]).predict(Xc)
    aipw = float(np.mean(mu1 - mu0 + T * (Y - mu1) / e - (1 - T) * (Y - mu0) / (1 - e)))

    print("Sensitivity check — estimators that do NOT use the pairing:")
    print(f"  naive difference in means   {naive:+.4f}")
    print(f"  IPW (propensity weighted)   {ipw:+.4f}")
    print(f"  AIPW / doubly robust        {aipw:+.4f}")
    print(f"  paired experimental ATE     {ate:+.4f}   <- the estimate we trust")
    print(f"\n  propensity overlap: e in [{e.min():.3f}, {e.max():.3f}]")
    print("  The paired estimate is preferred: it holds the gamble itself fixed, so it needs")
    print("  no covariate-adjustment assumptions at all. The others are reported to show the")
    print("  conclusion does not hinge on the pairing.")
    for nm, v in [("naive difference", naive), ("IPW", ipw), ("AIPW doubly-robust", aipw)]:
        CAUSAL_ROWS.append(dict(analysis=f"feedback effect — {nm}", track="RISK", estimate=v,
                                ci_low=np.nan, ci_high=np.nan, p_value=np.nan,
                                n=len(Y), method=nm, causal=(nm != "naive difference")))

CAUSAL = pd.DataFrame(CAUSAL_ROWS)
CAUSAL.to_csv(OUTPUTS / "causal_effects.csv", index=False)
display(CAUSAL)

## 24. Behavioural intervention analysis

A common framework for comparing decision-environment manipulations. Each comparison
reports the choice-rate difference, a bootstrap confidence interval, an effect size, an
assumption-appropriate significance test, and — critically — an explicit **causal flag**
saying whether the contrast came from a randomised manipulation or is merely
observational.

In [ ]:
# --- 24.1 Intervention comparison framework -----------------------------------------
def compare_intervention(name, y_ctrl, y_treat, *, causal: bool, note: str = "",
                         paired: bool = False):
    a, b = np.asarray(y_ctrl, float), np.asarray(y_treat, float)
    if paired:
        d = b - a
        est, lo, hi = bootstrap_ci(d)
        stat, p = stats.wilcoxon(b, a)
        eff = float(d.mean() / d.std(ddof=1)); eff_name = "Cohen's dz"
        test = "Wilcoxon signed-rank (paired)"
    else:
        rng = np.random.default_rng(RANDOM_SEED)
        boots = [b[rng.integers(0, len(b), len(b))].mean() - a[rng.integers(0, len(a), len(a))].mean()
                 for _ in range(CONFIG.n_bootstrap)]
        est = float(b.mean() - a.mean())
        lo, hi = float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))
        stat, p = stats.mannwhitneyu(b, a)
        pooled = np.sqrt(((len(a)-1)*a.var(ddof=1) + (len(b)-1)*b.var(ddof=1)) / (len(a)+len(b)-2))
        eff = float(est / pooled) if pooled > 0 else np.nan; eff_name = "Cohen's d"
        test = "Mann-Whitney U (independent)"
    return dict(comparison=name, n_control=len(a), n_treated=len(b),
                control_rate=round(float(a.mean()), 4), treated_rate=round(float(b.mean()), 4),
                difference=round(est, 4), ci_low=round(lo, 4), ci_high=round(hi, 4),
                effect_size=round(eff, 3), effect_size_name=eff_name,
                test=test, p_value=float(p),
                interpretation="CAUSAL (randomised)" if causal else "ASSOCIATION ONLY",
                note=note)

INTERVENTIONS = []
if AVAILABLE["choices13k"]:
    INTERVENTIONS.append(compare_intervention(
        "Described  ->  Experienced (feedback)", wide.described, wide.experienced,
        causal=True, paired=True,
        note="same gamble presented both ways; within-problem randomised contrast"))
    INTERVENTIONS.append(compare_intervention(
        "Risk  ->  Ambiguity", c13.loc[~c13.Amb, "bRate"], c13.loc[c13.Amb, "bRate"],
        causal=False, note="ambiguity is a design property of the problem, not randomised "
                           "within a problem; different gambles are being compared"))
    rare_m = RISK_X["rare_event_b"].to_numpy().astype(bool)
    INTERVENTIONS.append(compare_intervention(
        "Common-event  ->  Rare-event option B", RISK_y[~rare_m], RISK_y[rare_m],
        causal=False, note="structural property of the gamble"))

if AVAILABLE["itc_database"]:
    if "online_study" in itc.columns:
        on = pd.to_numeric(itc.online_study, errors="coerce")
        per = itc.assign(_on=on).groupby(["subj_ident", "_on"], observed=True).choice.mean().reset_index()
        if per._on.nunique() > 1:
            INTERVENTIONS.append(compare_intervention(
                "Lab  ->  Online administration", per.loc[per._on == 0, "choice"],
                per.loc[per._on == 1, "choice"], causal=False,
                note="studies differ on many dimensions at once; confounded with sample & protocol"))
    if "incentivization" in itc.columns:
        vc = itc.incentivization.value_counts()
        if len(vc) >= 2:
            a_lbl, b_lbl = vc.index[0], vc.index[1]
            per = itc.groupby(["subj_ident", "incentivization"], observed=True).choice.mean().reset_index()
            INTERVENTIONS.append(compare_intervention(
                f"Incentivisation: {a_lbl}  ->  {b_lbl}",
                per.loc[per.incentivization == a_lbl, "choice"],
                per.loc[per.incentivization == b_lbl, "choice"], causal=False,
                note="between-study comparison; incentivisation is confounded with study"))

INTERVENTION_TABLE = pd.DataFrame(INTERVENTIONS)
INTERVENTION_TABLE.to_csv(OUTPUTS / "intervention_analysis.csv", index=False)
display(INTERVENTION_TABLE.drop(columns=["note"]))
print("\nNotes:")
for _, r in INTERVENTION_TABLE.iterrows():
    print(f"  {r.comparison}\n    {r.interpretation}: {r.note}")

if len(INTERVENTION_TABLE):
    fig, ax = plt.subplots(figsize=(10, 1 + .7 * len(INTERVENTION_TABLE)))
    yy = range(len(INTERVENTION_TABLE))
    colors = ["#C44E52" if "CAUSAL" in i else "#8C8C8C" for i in INTERVENTION_TABLE.interpretation]
    for i, (_, r) in enumerate(INTERVENTION_TABLE.iterrows()):
        ax.errorbar([r.difference], [i],
                    xerr=[[r.difference - r.ci_low], [r.ci_high - r.difference]],
                    fmt="o", color=colors[i], capsize=5, lw=2, ms=8)
    ax.axvline(0, c="k", ls="--", lw=.8)
    ax.set_yticks(list(yy)); ax.set_yticklabels(INTERVENTION_TABLE.comparison, fontsize=9)
    ax.set(xlabel="change in choice rate (95% bootstrap CI)",
           title="Intervention effects — red = randomised (causal), grey = observational")
    savefig("intervention_effects"); plt.show()

## 25. Counterfactual analysis

The model can be re-queried with a changed decision environment. **These are
model-predicted counterfactuals, not causal effects.** They say "given everything the
model learned, a decision environment like *this* is associated with a choice probability
of *that*" — which is a prediction under a distribution shift, and is only trustworthy
inside the range of environments the data actually covered.

The one contrast in this notebook that *is* causal is the feedback manipulation in
Section 23, because it was randomised within problem.

In [ ]:
# --- 25.1 Counterfactual engine -----------------------------------------------------
def itc_counterfactual(base: dict, changes: dict) -> dict:
    """base: ss_value, ss_time, ll_value, ll_time (+ optional hist_ll_rate)."""
    model, X = TUNED["ITC"]
    def build(spec):
        row = pd.DataFrame([{c: np.nan for c in X.columns}])
        d = pd.DataFrame([{**{"ss_value": spec["ss_value"], "ss_time": spec["ss_time"],
                              "ll_value": spec["ll_value"], "ll_time": spec["ll_time"],
                              "choice": 0, "subj_ident": "cf", "trial_idx": 1}}])
        f = itc_features(d, include_history=False)
        for c in f.columns:
            if c in row.columns: row[c] = f[c].values[0]
        if "hist_ll_rate" in row.columns:
            row["hist_ll_rate"] = spec.get("hist_ll_rate", np.nan)
            row["hist_prev_choice"] = spec.get("hist_prev_choice", np.nan)
            row["hist_n_trials"] = spec.get("hist_n_trials", 0)
            row["hist_available"] = int(not np.isnan(spec.get("hist_ll_rate", np.nan)))
        if "betadelta_prediction" in row.columns:
            row["betadelta_prediction"] = BEHAV_MODELS["itc_quasi_hyperbolic"].predict_proba(
                np.array([[spec["ss_value"], spec["ss_time"],
                           spec["ll_value"], spec["ll_time"]]], float))[0]
        return to_numeric_matrix(row[X.columns])
    p0 = float(model.predict_proba(build(base))[0, 1])
    alt = {**base, **changes}
    p1 = float(model.predict_proba(build(alt))[0, 1])
    return dict(base=base, counterfactual=alt, p_base=p0, p_cf=p1, delta=p1 - p0)

if AVAILABLE["itc_database"]:
    print("=" * 78)
    print("MODEL-PREDICTED COUNTERFACTUALS  (not causal effects)")
    print("=" * 78)
    base = dict(ss_value=50., ss_time=0., ll_value=70., ll_time=30.)
    for label, ch in [("raise the delayed reward 70 -> 95", dict(ll_value=95.)),
                      ("raise the delayed reward 70 -> 120", dict(ll_value=120.)),
                      ("shorten the wait 30 -> 7 days", dict(ll_time=7.)),
                      ("lengthen the wait 30 -> 180 days", dict(ll_time=180.)),
                      ("remove immediacy (sooner also delayed 14d)", dict(ss_time=14., ll_time=44.))]:
        r = itc_counterfactual(base, ch)
        print(f"\n  Current:        ${base['ss_value']:.0f} in {base['ss_time']:.0f}d  vs  "
              f"${base['ll_value']:.0f} in {base['ll_time']:.0f}d")
        print(f"  Counterfactual: {label}")
        print(f"    P(choose immediate)  {100*(1-r['p_base']):.1f}%  ->  {100*(1-r['p_cf']):.1f}%"
              f"   ({-100*r['delta']:+.1f} pp)")

    # --- response surface
    lls = np.linspace(52, 160, 40); dls = np.array([1, 7, 14, 30, 90, 180, 365])
    surf = np.zeros((len(dls), len(lls)))
    for i, dl in enumerate(dls):
        for j, lv in enumerate(lls):
            surf[i, j] = itc_counterfactual(base, dict(ll_value=lv, ll_time=float(dl)))["p_cf"]
    fig, ax = plt.subplots(1, 2, figsize=(15, 4.8))
    for i, dl in enumerate(dls):
        ax[0].plot(lls, surf[i], label=f"{dl}d wait")
    ax[0].axhline(.5, ls="--", c="k", lw=.8); ax[0].legend(fontsize=8, ncol=2)
    ax[0].set(title="Predicted patience vs size of the delayed reward\n($50 available now)",
              xlabel="delayed reward ($)", ylabel="P(choose delayed)")
    im = ax[1].imshow(surf, aspect="auto", origin="lower", cmap="viridis",
                      extent=[lls[0], lls[-1], 0, len(dls)])
    ax[1].set_yticks(np.arange(len(dls)) + .5); ax[1].set_yticklabels([f"{d}d" for d in dls])
    ax[1].set(title="Predicted P(choose delayed)", xlabel="delayed reward ($)", ylabel="wait")
    plt.colorbar(im, ax=ax[1])
    savefig("counterfactual_surface"); plt.show()

## 26. Behavioural intervention optimiser

**Why this is not a pricing optimiser.** The brief asks for price / reference-price /
discount optimisation. None of the datasets that survived the automation filter contain
retail prices, reference prices or discounts — and inventing them would be fabrication.
So, exactly as the brief instructs for that case, the module is implemented as a
**behavioural intervention optimiser** over decision variables that genuinely exist.

**The objective, defined explicitly.** A programme designer wants people to take the
delayed (larger, later) option — a patient-savings nudge, a delayed-bonus scheme — and
pays for it with the size of the delayed reward. The objective is

```
Expected incentive cost  =  P(LL | environment) · ll_value  +  (1 − P(LL)) · ss_value
Objective                =  maximise P(LL)   subject to   expected cost ≤ budget
Efficiency               =  P(LL) per dollar of expected cost
```

subject to constraints that keep the search inside the region the data actually covers:

```
ll_value  >  ss_value            (otherwise it is not an intertemporal trade-off)
ll_value  ≤  ss_value × max_multiple
ss_time   ≥  0,  ll_time  >  ss_time,  ll_time ≤ max_delay
ll_time − ss_time  ≥  min_delay        (the programme's fixed waiting period)
```

The `min_delay` floor matters. Without it the optimiser finds a trivial solution — shrink
the wait to a single day and nearly everyone "waits" — which optimises the stated
objective while being a useless intervention. A real patience programme has a delay fixed
by policy (a vesting period, a quarterly payout), so the floor is a genuine design
constraint rather than a convenience.

Three search strategies are compared: exhaustive grid, random search, and Bayesian
optimisation (Optuna). The result is a **model-based recommendation**, and it inherits
every limitation of the underlying predictive model.

In [ ]:
# --- 26.1 Optimiser -----------------------------------------------------------------
@dataclass
class InterventionSpace:
    ss_value: float = 50.0
    ss_time: float = 0.0
    ll_value_range: tuple = (51.0, 200.0)
    # NOTE ON min_delay. Without a floor the optimiser has a trivial solution: shrink the
    # wait to ~1 day and almost everyone "waits". That is a real optimum of the stated
    # objective but a useless intervention, because the whole point of a patience
    # programme is that the delay is fixed by the policy (a 30-day vesting period, a
    # quarterly payout). So the delay floor is a genuine design constraint, not a fudge.
    min_delay: float = 30.0
    ll_time_range: tuple = (30.0, 365.0)
    max_multiple: float = 4.0
    budget: float = 90.0            # max acceptable expected cost per person
    hist_ll_rate: float = np.nan    # optional participant history

    def valid(self, ll_value, ll_time):
        return (ll_value > self.ss_value
                and ll_value <= self.ss_value * self.max_multiple
                and ll_time > self.ss_time
                and (ll_time - self.ss_time) >= self.min_delay)

class InterventionOptimizer:
    """Searches the decision environment for settings that maximise predicted patience
    subject to an expected-cost budget."""
    def __init__(self, space: InterventionSpace):
        self.space = space
        self.model, self.X = TUNED["ITC"]

    def _p(self, ll_value, ll_time):
        return itc_counterfactual(
            dict(ss_value=self.space.ss_value, ss_time=self.space.ss_time,
                 ll_value=float(ll_value), ll_time=float(ll_time),
                 hist_ll_rate=self.space.hist_ll_rate),
            {})["p_base"]

    def evaluate(self, ll_value, ll_time) -> dict:
        s = self.space
        if not s.valid(ll_value, ll_time):
            return dict(feasible=False, p_ll=np.nan, cost=np.nan, efficiency=np.nan,
                        ll_value=ll_value, ll_time=ll_time)
        p = self._p(ll_value, ll_time)
        cost = p * ll_value + (1 - p) * s.ss_value
        return dict(feasible=bool(cost <= s.budget), p_ll=p, cost=cost,
                    efficiency=p / max(cost, 1e-9), ll_value=float(ll_value),
                    ll_time=float(ll_time))

    def grid(self, n_v=24, n_t=12):
        vs = np.linspace(*self.space.ll_value_range, n_v)
        ts = np.geomspace(*self.space.ll_time_range, n_t)
        return pd.DataFrame([self.evaluate(v, t) for v in vs for t in ts])

    def random(self, n=200, seed=RANDOM_SEED):
        rng = np.random.default_rng(seed)
        vs = rng.uniform(*self.space.ll_value_range, n)
        ts = np.exp(rng.uniform(np.log(self.space.ll_time_range[0]),
                                np.log(self.space.ll_time_range[1]), n))
        return pd.DataFrame([self.evaluate(v, t) for v, t in zip(vs, ts)])

    def bayesian(self, n_trials=60, seed=RANDOM_SEED):
        recs = []
        def obj(t):
            v = t.suggest_float("ll_value", *self.space.ll_value_range)
            d = t.suggest_float("ll_time", *self.space.ll_time_range, log=True)
            r = self.evaluate(v, d); recs.append(r)
            if not r["feasible"] or np.isnan(r["p_ll"]):
                return -1.0
            return r["p_ll"]
        st = optuna.create_study(direction="maximize",
                                 sampler=optuna.samplers.TPESampler(seed=seed))
        st.optimize(obj, n_trials=n_trials, show_progress_bar=False)
        return pd.DataFrame(recs)

OPT_RESULTS = {}
if AVAILABLE["itc_database"]:
    space = InterventionSpace()
    opt = InterventionOptimizer(space)
    t0 = time.time()
    n_bo = 25 if CONFIG.quick_mode else 60
    for nm, res in [("grid", opt.grid(20 if CONFIG.quick_mode else 24,
                                      8 if CONFIG.quick_mode else 12)),
                    ("random", opt.random(60 if CONFIG.quick_mode else 200)),
                    ("bayesian", opt.bayesian(n_bo))]:
        OPT_RESULTS[nm] = res
    print(f"searched {sum(len(v) for v in OPT_RESULTS.values())} configurations "
          f"in {time.time()-t0:.0f}s")

    all_res = pd.concat([v.assign(method=k) for k, v in OPT_RESULTS.items()], ignore_index=True)
    feas = all_res[all_res.feasible == True].dropna(subset=["p_ll"])
    all_res.to_csv(OUTPUTS / "optimization_results.csv", index=False)

    best_p = feas.loc[feas.p_ll.idxmax()]
    best_e = feas.loc[feas.efficiency.idxmax()]
    baseline = opt.evaluate(70.0, 30.0)      # a conventional, unoptimised offer

    print("\n" + "=" * 78)
    print("RECOMMENDED BEHAVIOURAL INTERVENTION  (model-based optimisation result)")
    print("=" * 78)
    print(f"  Sooner option (fixed)     ${space.ss_value:.0f} available immediately")
    print(f"  Programme constraint      wait must be at least {space.min_delay:.0f} days")
    print(f"  Recommended later reward  ${best_p.ll_value:.2f}")
    print(f"  Recommended wait          {best_p.ll_time:.0f} days")
    print(f"  Search method             {best_p.get('method', 'combined')}")
    print(f"\n  Predicted P(choose delayed)   {best_p.p_ll*100:.1f}%")
    print(f"  Expected cost per person      ${best_p.cost:.2f}   (budget ${space.budget:.0f})")
    print(f"  Patience per dollar           {best_p.efficiency:.5f}")
    print("\n  --- versus an unoptimised conventional offer ($70 in 30 days) ---")
    print(f"  Baseline  P(delayed)={baseline['p_ll']*100:.1f}%   cost=${baseline['cost']:.2f}   "
          f"efficiency={baseline['efficiency']:.5f}")
    print(f"  Optimised P(delayed)={best_p.p_ll*100:.1f}%   cost=${best_p.cost:.2f}   "
          f"efficiency={best_p.efficiency:.5f}")
    print(f"  Estimated improvement in take-up   "
          f"{100*(best_p.p_ll - baseline['p_ll']):+.1f} percentage points "
          f"({100*(best_p.p_ll/max(baseline['p_ll'],1e-9)-1):+.1f}% relative)")
    print(f"\n  Most COST-EFFICIENT feasible setting: ${best_e.ll_value:.2f} in "
          f"{best_e.ll_time:.0f} days -> P={best_e.p_ll*100:.1f}%, cost=${best_e.cost:.2f}")
    print("\n  This is a MODEL-BASED optimisation result, not an experimentally verified")
    print("  causal effect. It should be validated in a randomised trial before deployment.")

    fig, ax = plt.subplots(1, 3, figsize=(16, 4.6))
    g = OPT_RESULTS["grid"].dropna(subset=["p_ll"])
    piv = g.pivot_table(index="ll_time", columns="ll_value", values="p_ll")
    im = ax[0].contourf(piv.columns, piv.index, piv.values, levels=18, cmap="viridis")
    ax[0].set_yscale("log"); plt.colorbar(im, ax=ax[0])
    ax[0].scatter([best_p.ll_value], [best_p.ll_time], marker="*", s=280,
                  c="red", edgecolor="white", zorder=5)
    ax[0].set(title="Objective landscape: P(choose delayed)",
              xlabel="delayed reward ($)", ylabel="wait (days, log)")
    for nm, r in OPT_RESULTS.items():
        rr = r.dropna(subset=["p_ll"])
        ax[1].scatter(rr.cost, rr.p_ll, s=12, alpha=.5, label=f"{nm} (n={len(rr)})")
    ax[1].axvline(space.budget, ls="--", c="r", label="budget")
    ax[1].legend(fontsize=8); ax[1].set(title="Cost-effectiveness frontier",
                                        xlabel="expected cost per person ($)", ylabel="P(delayed)")
    ax[2].bar(["baseline\n($70/30d)", "optimised"], [baseline["p_ll"], best_p.p_ll],
              color=["#8C8C8C", "#C44E52"])
    for i, v in enumerate([baseline["p_ll"], best_p.p_ll]):
        ax[2].text(i, v + .01, f"{v*100:.1f}%", ha="center", fontweight="bold")
    ax[2].set(title="Baseline vs optimised take-up", ylabel="P(choose delayed)")
    savefig("optimization"); plt.show()

## 27. Scenario simulator

Five stylised decision-makers, each defined by an *observed behavioural history* rather
than by a label pinned on them. The scenarios differ only in the participant-history
feature and the decision environment — the model is identical.

In [ ]:
# --- 27.1 Scenarios -----------------------------------------------------------------
SCENARIOS = [
    dict(name="1. Strong present-bias pattern",
         desc="has taken the sooner option on nearly every previous trial; sooner reward is immediate",
         base=dict(ss_value=50., ss_time=0., ll_value=70., ll_time=30., hist_ll_rate=0.08)),
    dict(name="2. Patient pattern",
         desc="has usually waited on previous trials",
         base=dict(ss_value=50., ss_time=0., ll_value=70., ll_time=30., hist_ll_rate=0.85)),
    dict(name="3. No immediacy (front-end delay)",
         desc="identical trade-off, but nothing is available today",
         base=dict(ss_value=50., ss_time=14., ll_value=70., ll_time=44., hist_ll_rate=0.45)),
    dict(name="4. Long-horizon offer",
         desc="the wait is a full year",
         base=dict(ss_value=50., ss_time=0., ll_value=70., ll_time=365., hist_ll_rate=0.45)),
    dict(name="5. Large premium for waiting",
         desc="the delayed reward is more than double",
         base=dict(ss_value=50., ss_time=0., ll_value=110., ll_time=30., hist_ll_rate=0.45)),
]

if AVAILABLE["itc_database"]:
    rows = []
    for s in SCENARIOS:
        r = itc_counterfactual(s["base"], {})
        theory = BEHAV_MODELS["itc_quasi_hyperbolic"].predict_proba(
            np.array([[s["base"]["ss_value"], s["base"]["ss_time"],
                       s["base"]["ll_value"], s["base"]["ll_time"]]], float))[0]
        rows.append(dict(scenario=s["name"], description=s["desc"],
                         offer=f"${s['base']['ss_value']:.0f}@{s['base']['ss_time']:.0f}d vs "
                               f"${s['base']['ll_value']:.0f}@{s['base']['ll_time']:.0f}d",
                         observed_history=s["base"]["hist_ll_rate"],
                         ml_p_delayed=round(r["p_base"], 4),
                         theory_p_delayed=round(float(theory), 4)))
    SCEN = pd.DataFrame(rows)
    display(SCEN)
    SCEN.to_csv(OUTPUTS / "scenario_simulation.csv", index=False)

    fig, ax = plt.subplots(figsize=(11, 4.5))
    x = np.arange(len(SCEN)); w = .38
    ax.bar(x - w/2, SCEN.ml_p_delayed, w, label="ML model", color="#4C72B0")
    ax.bar(x + w/2, SCEN.theory_p_delayed, w, label="beta-delta theory", color="#C44E52")
    ax.set_xticks(x); ax.set_xticklabels([s.split(".")[1].strip() for s in SCEN.scenario],
                                         rotation=18, ha="right", fontsize=8)
    ax.set(ylabel="P(choose delayed)", title="Scenario comparison")
    ax.legend(); savefig("scenarios"); plt.show()

## 28. Cross-dataset generalisation

This is the strictest test in the notebook, and the one that separates *learning a
behavioural regularity* from *memorising an experiment*.

* **Leave-one-study-out (ITC)** — train on 99 studies, test on the held-out 100th. The
  held-out study has a different lab, sample, currency, reward magnitudes and protocol.
* **choices13k → CPC18** — train the risk model on MTurk-collected aggregate data and
  test it on an entirely independent lab experiment run in Israel.

Both are reported as **in-domain vs out-of-domain**, with the degradation quantified.

In [ ]:
# --- 28.1 Leave-one-study-out (ITC) -------------------------------------------------
CROSS = []
if AVAILABLE["itc_database"]:
    X = to_numeric_matrix(ITC_X)
    counts = pd.Series(ITC_studies).value_counts()
    held = counts[counts >= 800].index[: (5 if CONFIG.quick_mode else 12)]
    print(f"leave-one-study-out over {len(held)} studies (>=800 trials each)\n")
    for st in held:
        te = np.where(ITC_studies == st)[0]
        tr = np.where(ITC_studies != st)[0]
        if len(np.unique(ITC_y[te])) < 2: continue
        m = HistGradientBoostingClassifier(max_iter=250, learning_rate=.08,
                                           random_state=RANDOM_SEED).fit(X.iloc[tr], ITC_y[tr])
        p_out = m.predict_proba(X.iloc[te])[:, 1]
        # in-domain reference: participant-grouped split INSIDE the held-out study
        sp_in = grouped_split(ITC_groups[te], ITC_y[te], test_size=.3, val_size=.0)
        i_tr, i_te = te[sp_in["train"]], te[sp_in["test"]]
        ok_in = len(np.unique(ITC_y[i_tr])) > 1 and len(np.unique(ITC_y[i_te])) > 1
        if ok_in:
            m_in = HistGradientBoostingClassifier(max_iter=250, learning_rate=.08,
                                                  random_state=RANDOM_SEED).fit(X.iloc[i_tr], ITC_y[i_tr])
            p_in = m_in.predict_proba(X.iloc[i_te])[:, 1]
        bd = BEHAV_MODELS["itc_quasi_hyperbolic"].predict_proba(
            itc[ITC_MODEL_COLS].to_numpy(float)[te])
        CROSS.append(dict(
            experiment="leave-one-study-out", held_out=str(st)[:40], n_test=len(te),
            n_participants=int(pd.Series(ITC_groups[te]).nunique()),
            out_of_domain_auc=round(float(roc_auc_score(ITC_y[te], p_out)), 4),
            out_of_domain_logloss=round(float(log_loss(ITC_y[te], p_out, labels=[0, 1])), 4),
            in_domain_auc=round(float(roc_auc_score(ITC_y[i_te], p_in)), 4) if ok_in else np.nan,
            in_domain_logloss=round(float(log_loss(ITC_y[i_te], p_in, labels=[0,1])), 4) if ok_in else np.nan,
            theory_auc=round(float(roc_auc_score(ITC_y[te], bd)), 4),
            base_rate=round(float(ITC_y[te].mean()), 3)))
    LOSO = pd.DataFrame(CROSS)
    display(LOSO)
    if len(LOSO):
        ood, ind = LOSO.out_of_domain_auc.mean(), LOSO.in_domain_auc.mean()
        gap = ood - ind
        print(f"\n  mean in-domain AUC       {ind:.4f}")
        print(f"  mean out-of-domain AUC   {ood:.4f}")
        print(f"  change moving out of domain  {gap:+.4f} AUC ({100*gap/ind:+.1f}%)"
              f"  -> {'DEGRADATION' if gap < 0 else 'no degradation'}")
        if gap >= 0:
            print("  Out-of-domain scores here are not worse than in-domain. That is not")
            print("  magic: the in-domain reference trains on only ~70% of ONE study, so it")
            print("  has far less data than the leave-one-study-out model. Read the absolute")
            print("  out-of-domain AUC, not the gap, as the generalisation result.")
        print(f"  theory AUC out-of-domain {LOSO.theory_auc.mean():.4f}")
        print(f"\n  ML retains an out-of-domain advantage over theory of "
              f"{ood - LOSO.theory_auc.mean():+.4f} AUC." if ood > LOSO.theory_auc.mean()
              else "\n  Theory generalises BETTER than ML out of domain — an important negative result.")

        fig, ax = plt.subplots(figsize=(11, 4.6))
        xx = np.arange(len(LOSO)); w = .27
        ax.bar(xx - w, LOSO.in_domain_auc, w, label="in-domain ML", color="#4C72B0")
        ax.bar(xx, LOSO.out_of_domain_auc, w, label="out-of-domain ML", color="#DD8452")
        ax.bar(xx + w, LOSO.theory_auc, w, label="beta-delta theory", color="#C44E52")
        ax.axhline(.5, ls="--", c="k", lw=.8)
        ax.set_xticks(xx); ax.set_xticklabels(LOSO.held_out, rotation=55, ha="right", fontsize=7)
        ax.set(ylabel="ROC-AUC", title="Leave-one-study-out generalisation"); ax.legend()
        savefig("cross_study_generalization"); plt.show()

In [ ]:
# --- 28.2 choices13k -> CPC18 (train on one experiment, test on another) -----------
if AVAILABLE["choices13k"] and AVAILABLE["cpc18"]:
    # Build CPC18 prospects from the resolved 2-outcome description of each option.
    oa2 = [np.array([r.ha, r.la], float) for r in cpc.itertuples()]
    pa2 = [np.array([r.pha, 1 - r.pha], float) for r in cpc.itertuples()]
    ob2 = [np.array([r.hb, r.lb], float) for r in cpc.itertuples()]
    pb2 = [np.array([r.phb, 1 - r.phb], float) for r in cpc.itertuples()]
    CA, CB = ProspectTableau(oa2, pa2), ProspectTableau(ob2, pb2)

    common = ["ev_a", "ev_b", "ev_diff", "sd_a", "sd_b", "sd_diff",
              "p_loss_a", "p_loss_b", "p_loss_diff", "e_gain_a", "e_gain_b",
              "e_loss_a", "e_loss_b", "max_a", "min_a", "max_b", "min_b",
              "range_a", "range_b", "best_outcome_is_b", "worst_outcome_is_b",
              "p_high_a", "p_high_b", "rare_event_a", "rare_event_b",
              "b_dominates_a", "a_dominates_b", "any_loss"]

    def tableau_features(A, B, p_high_a, p_high_b):
        return pd.DataFrame({
            "ev_a": A.ev, "ev_b": B.ev, "ev_diff": B.ev - A.ev,
            "sd_a": A.sd, "sd_b": B.sd, "sd_diff": B.sd - A.sd,
            "p_loss_a": A.p_loss, "p_loss_b": B.p_loss, "p_loss_diff": B.p_loss - A.p_loss,
            "e_gain_a": A.e_gain, "e_gain_b": B.e_gain,
            "e_loss_a": A.e_loss, "e_loss_b": B.e_loss,
            "max_a": A.omax, "min_a": A.omin, "max_b": B.omax, "min_b": B.omin,
            "range_a": A.omax - A.omin, "range_b": B.omax - B.omin,
            "best_outcome_is_b": (B.omax > A.omax).astype(int),
            "worst_outcome_is_b": (B.omin < A.omin).astype(int),
            "p_high_a": p_high_a, "p_high_b": p_high_b,
            "rare_event_a": ((p_high_a < .1) | (p_high_a > .9)).astype(int),
            "rare_event_b": ((p_high_b < .1) | (p_high_b > .9)).astype(int),
            "b_dominates_a": (B.omin >= A.omax).astype(int),
            "a_dominates_b": (A.omin >= B.omax).astype(int),
            "any_loss": ((A.p_loss > 0) | (B.p_loss > 0)).astype(int)})[common]

    Xsrc = RISK_X[common]
    Xtgt = tableau_features(CA, CB, cpc.pha.to_numpy(float), cpc.phb.to_numpy(float))

    tr = SPLITS["RISK"]["train"]; te = SPLITS["RISK"]["test"]
    xfer = HistGradientBoostingRegressor(max_iter=400, learning_rate=.06,
                                         random_state=RANDOM_SEED)
    xfer.fit(Xsrc.iloc[tr], RISK_y[tr], sample_weight=RISK_n[tr])

    p_in = np.clip(xfer.predict(Xsrc.iloc[te]), 1e-6, 1-1e-6)
    in_auc = roc_auc_score((RISK_y[te] > .5).astype(int), p_in)
    in_brier = float(np.average((p_in - RISK_y[te]) ** 2, weights=RISK_n[te]))

    y_t = cpc.choice_b.to_numpy(int)
    p_out = np.clip(xfer.predict(Xtgt), 1e-6, 1-1e-6)
    out_auc = roc_auc_score(y_t, p_out); out_ll = log_loss(y_t, p_out, labels=[0, 1])
    out_brier = brier_score_loss(y_t, p_out)

    # behavioural theory transferred the same way
    cpt_out = BEHAV_MODELS["risk_cpt"].predict_proba(CA, CB)
    cpt_auc = roc_auc_score(y_t, cpt_out)

    print("=" * 78)
    print("CROSS-DATASET TRANSFER:  train on choices13k (MTurk)  ->  test on CPC18 (lab)")
    print("=" * 78)
    print(f"  in-domain  (choices13k held-out) AUC={in_auc:.4f}  Brier={in_brier:.4f}")
    print(f"  out-of-domain (CPC18)            AUC={out_auc:.4f}  Brier={out_brier:.4f}  "
          f"logloss={out_ll:.4f}")
    print(f"  degradation                      {out_auc - in_auc:+.4f} AUC "
          f"({100*(out_auc-in_auc)/in_auc:+.1f}%)")
    print(f"  CPT transferred to CPC18         AUC={cpt_auc:.4f}")
    print(f"  CPC18 base rate                  {y_t.mean():.4f}  ({len(y_t):,} decisions, "
          f"{cpc.participant_id.nunique()} participants)")
    print("\n  Any degradation here is the honest cost of dataset shift: different population,")
    print("  different incentives, different presentation. It is exactly the effect reported")
    print("  by Thomas et al. (2024, Nature Human Behaviour) for models trained on choices13k.")
    CROSS.append(dict(experiment="choices13k -> CPC18", held_out="cpc18", n_test=len(y_t),
                      n_participants=int(cpc.participant_id.nunique()),
                      out_of_domain_auc=round(float(out_auc), 4),
                      out_of_domain_logloss=round(float(out_ll), 4),
                      in_domain_auc=round(float(in_auc), 4), in_domain_logloss=np.nan,
                      theory_auc=round(float(cpt_auc), 4), base_rate=round(float(y_t.mean()), 3)))

CROSS_TABLE = pd.DataFrame(CROSS)
CROSS_TABLE.to_csv(OUTPUTS / "cross_dataset_generalization.csv", index=False)

## 29. Robustness analysis

Do the conclusions survive changes to arbitrary choices? Five perturbations are tested:
different random seeds, different model families, removing participant-history features,
removing demographic features, and removing the theory-derived hybrid feature.

In [ ]:
# --- 29.1 Robustness ----------------------------------------------------------------
ROBUST = []
if AVAILABLE["itc_database"]:
    Xfull = to_numeric_matrix(ITC_X)
    hyb = BEHAV_MODELS["itc_quasi_hyperbolic"].predict_proba(itc[ITC_MODEL_COLS].to_numpy(float))
    Xfull_bd = Xfull.copy(); Xfull_bd["betadelta_prediction"] = hyb

    hist_cols = [c for c in Xfull.columns if c.startswith("hist_")]
    demo_cols = [c for c in Xfull.columns if c in ("age",) or c.startswith("cat_country")
                 or c.startswith("cat_currency")]

    variants = {
        "full (history + theory)": Xfull_bd,
        "no theory feature": Xfull,
        "no participant history": Xfull_bd.drop(columns=hist_cols, errors="ignore"),
        "no demographics": Xfull_bd.drop(columns=demo_cols, errors="ignore"),
        "decision variables only": Xfull_bd[[c for c in Xfull_bd.columns
                                             if not c.startswith(("hist_", "cat_"))
                                             and c not in ("age",)]],
    }
    seeds = [RANDOM_SEED, 7, 2024] if not CONFIG.quick_mode else [RANDOM_SEED]
    for vname, XX in variants.items():
        for seed in seeds:
            sp = grouped_split(ITC_groups, ITC_y, seed=seed)
            m = HistGradientBoostingClassifier(max_iter=250, learning_rate=.08,
                                               random_state=seed).fit(XX.iloc[sp["train"]],
                                                                      ITC_y[sp["train"]])
            p = m.predict_proba(XX.iloc[sp["test"]])[:, 1]
            ROBUST.append(dict(track="ITC", variant=vname, seed=seed, n_features=XX.shape[1],
                               test_auc=round(float(roc_auc_score(ITC_y[sp["test"]], p)), 4),
                               test_logloss=round(float(log_loss(ITC_y[sp["test"]], p, labels=[0,1])), 4),
                               test_brier=round(float(brier_score_loss(ITC_y[sp["test"]], p)), 4)))

if AVAILABLE["choices13k"]:
    Xr = to_numeric_matrix(RISK_X)
    Xr_c = Xr.copy(); Xr_c["cpt_prediction"] = BEHAV_MODELS["risk_cpt"].predict_proba(TA, TB)
    for vname, XX in {"full (+CPT)": Xr_c, "no theory feature": Xr,
                      "no condition variables": Xr_c.drop(columns=["feedback", "block",
                                                                   "ambiguity", "correlation"],
                                                          errors="ignore")}.items():
        for seed in ([RANDOM_SEED, 7, 2024] if not CONFIG.quick_mode else [RANDOM_SEED]):
            sp = grouped_split(RISK_groups, RISK_y, seed=seed)
            m = HistGradientBoostingRegressor(max_iter=400, learning_rate=.06,
                                              random_state=seed).fit(
                XX.iloc[sp["train"]], RISK_y[sp["train"]], sample_weight=RISK_n[sp["train"]])
            p = np.clip(m.predict(XX.iloc[sp["test"]]), 1e-6, 1-1e-6)
            ROBUST.append(dict(track="RISK", variant=vname, seed=seed, n_features=XX.shape[1],
                               test_mse=round(float(np.mean((p - RISK_y[sp['test']])**2)), 5),
                               test_logloss=round(binom_nll(p, RISK_k[sp["test"]],
                                                            RISK_n[sp["test"]]) / RISK_n[sp["test"]].sum(), 5)))

ROBUST_TABLE = pd.DataFrame(ROBUST)
ROBUST_TABLE.to_csv(OUTPUTS / "robustness_analysis.csv", index=False)
for tk in ROBUST_TABLE.track.unique():
    sub = ROBUST_TABLE[ROBUST_TABLE.track == tk]
    key = "test_auc" if "test_auc" in sub and sub.test_auc.notna().any() else "test_logloss"
    agg = sub.groupby("variant")[[c for c in sub.columns if c.startswith("test_")]].agg(["mean", "std"])
    print(f"\n--- {tk} robustness (mean +/- sd across seeds) ---")
    display(agg.round(5))
    spread = sub.groupby("variant")[key].mean()
    print(f"  spread in {key} across feature variants: "
          f"{spread.max() - spread.min():.4f}  "
          f"(best: {spread.idxmin() if 'logloss' in key else spread.idxmax()})")

## 30. Ethics, fairness and responsible use

This project models how real people make decisions, using data those people contributed
to research studies. That deserves an explicit accounting rather than a boilerplate
paragraph.

**Consent and provenance.** Every dataset used here comes from peer-reviewed studies in
which participants gave informed consent and were compensated. The CPC18 participants
consented at the start of a ~45-minute session and were paid for a randomly selected
choice plus a show-up fee. The ITC Database was assembled only from studies whose
corresponding authors gave explicit permission to redistribute under CC BY-NC-SA. The
data are de-identified: `subj_ident` is an arbitrary index, not a person. Nothing in this
notebook attempts re-identification, and nothing should.

**Prediction is not diagnosis.** A fitted `β` of 0.7 means *this person took the sooner
option more often than a time-consistent model predicts, in this task, on this day*. It
is not impulsivity, not a disorder, not a personality. Discount rates are known to move
with mood, sleep, stress, framing and how the question is asked. Treating an estimate
like this as a stable property of a human being is the single most likely way this work
could be misused.

**The manipulation problem.** Section 26 optimises a decision environment to shift
people's choices. The same machinery that finds the cheapest way to encourage saving
finds the cheapest way to encourage borrowing. Choice architecture is not ethically
neutral, and an optimiser has no opinion about which direction it is pointed. Legitimate
uses are transparent, in the chooser's interest, and reversible; the dark-pattern version
is covert, against the chooser's interest, and hard to escape. That distinction lives in
governance, not in the loss function.

**What this model must not be used for.** Individual-level consequential decisions —
credit, insurance, hiring, clinical judgement. The out-of-domain results in Section 28
show performance degrades across contexts, and the participant pools here (students,
MTurk workers) are not representative of any general population.

**Demographic variables.** Only age and country enter the feature set, and only because
they are protocol-level covariates in the source data. No sensitive attributes beyond
these are used. Subgroup performance is audited below, because a model can be accurate on
average and much worse for some group.

In [ ]:
# --- 30.1 Subgroup fairness audit ---------------------------------------------------
FAIRNESS = []
if AVAILABLE["itc_database"]:
    m, X = TUNED["ITC"]; te = SPLITS["ITC"]["test"]
    p = m.predict_proba(X.iloc[te])[:, 1]; y = ITC_y[te]
    sub = itc.iloc[te]

    def audit(group_series, group_name, min_n=300):
        g = pd.Series(group_series).reset_index(drop=True)
        for val, idx in g.groupby(g, observed=True).groups.items():
            idx = np.asarray(idx)
            if len(idx) < min_n or len(np.unique(y[idx])) < 2: continue
            FAIRNESS.append(dict(
                attribute=group_name, group=str(val)[:32], n=len(idx),
                base_rate=round(float(y[idx].mean()), 4),
                auc=round(float(roc_auc_score(y[idx], p[idx])), 4),
                logloss=round(float(log_loss(y[idx], p[idx], labels=[0, 1])), 4),
                brier=round(float(brier_score_loss(y[idx], p[idx])), 4),
                calib_gap=round(float(p[idx].mean() - y[idx].mean()), 4)))

    if "age" in sub.columns and sub.age.notna().any():
        audit(pd.cut(pd.to_numeric(sub.age, errors="coerce"),
                     bins=[0, 20, 25, 35, 50, 120],
                     labels=["<20", "20-25", "25-35", "35-50", "50+"]).astype(str), "age band")
    if "country" in sub.columns:
        audit(sub.country.astype(str), "country")
    if "online_study" in sub.columns:
        audit(sub.online_study.map({0: "lab", 1: "online", 0.0: "lab", 1.0: "online"}).astype(str),
              "administration")
    if "incentivization" in sub.columns:
        audit(sub.incentivization.astype(str), "incentivisation")

FAIR = pd.DataFrame(FAIRNESS)
FAIR.to_csv(OUTPUTS / "fairness_subgroup_analysis.csv", index=False)
if len(FAIR):
    display(FAIR)
    print("\nDisparity summary (max - min across groups within an attribute):")
    for attr, g in FAIR.groupby("attribute"):
        print(f"  {attr:18s} AUC spread {g.auc.max()-g.auc.min():.4f} | "
              f"log-loss spread {g.logloss.max()-g.logloss.min():.4f} | "
              f"largest calibration gap {g.calib_gap.abs().max():+.4f}")
    fig, ax = plt.subplots(figsize=(11, .45 * len(FAIR) + 2))
    lbl = FAIR.attribute + " = " + FAIR.group
    ax.barh(range(len(FAIR)), FAIR.auc, color="#4C72B0")
    ax.axvline(FAIR.auc.mean(), ls="--", c="r", label="mean")
    ax.set_yticks(range(len(FAIR))); ax.set_yticklabels(lbl, fontsize=8)
    ax.set(xlabel="ROC-AUC", title="Subgroup predictive performance", xlim=(.5, 1.0))
    ax.legend(); savefig("fairness_subgroups"); plt.show()
    print("\nA large spread here would mean the model is systematically less reliable for some")
    print("groups. Because study, country and protocol are entangled in this pooled database,")
    print("differences below should be read as data-composition effects, not as evidence")
    print("about the groups themselves.")
else:
    print("Insufficient subgroup coverage for a fairness audit.")

## 31. Final model selection

In [ ]:
# --- 31.1 Select and persist the production models ----------------------------------
import joblib

FINAL = {}
for track, key in [("ITC", "test_logloss"), ("RISK", "test_mse_rate")]:
    sub = COMPARISON[(COMPARISON.track == track) & (COMPARISON.family != "behavioural")]
    sub = sub.dropna(subset=[key])
    if not len(sub): continue
    # select on VALIDATION, then report the test score of that choice
    vkey = "val_" + key.split("test_", 1)[1]
    chosen = sub.loc[sub[vkey].idxmin()] if vkey in sub else sub.loc[sub[key].idxmin()]
    FINAL[track] = dict(model=chosen.model, val=float(chosen.get(vkey, np.nan)),
                        test=float(chosen[key]), metric=key)
    print(f"{track}: selected '{chosen.model}' on validation {vkey}={chosen.get(vkey):.5f} "
          f"-> test {key}={chosen[key]:.5f}")

for track in ("ITC", "RISK"):
    if track in TUNED:
        joblib.dump({"model": TUNED[track][0], "features": list(TUNED[track][1].columns),
                     "config": dataclasses.asdict(CONFIG), "env": ENV_INFO},
                    MODELS / f"final_model_{track}.joblib")
        print(f"  saved models/final_model_{track}.joblib")

# test-set predictions for the record
pred_rows = []
if AVAILABLE["itc_database"]:
    m, X = TUNED["ITC"]; te = SPLITS["ITC"]["test"]
    pred_rows.append(pd.DataFrame({
        "track": "ITC", "row_index": te, "participant_id": ITC_groups[te],
        "study": ITC_studies[te], "y_true": ITC_y[te],
        "y_pred_proba": m.predict_proba(X.iloc[te])[:, 1],
        "theory_pred": BEHAV_MODELS["itc_quasi_hyperbolic"].predict_proba(
            itc[ITC_MODEL_COLS].to_numpy(float)[te])}))
if AVAILABLE["choices13k"]:
    m, X = TUNED["RISK"]; te = SPLITS["RISK"]["test"]
    pred_rows.append(pd.DataFrame({
        "track": "RISK", "row_index": te, "participant_id": np.nan,
        "study": "choices13k", "y_true": RISK_y[te],
        "y_pred_proba": np.clip(m.predict(X.iloc[te]), 0, 1),
        "theory_pred": BEHAV_MODELS["risk_cpt"].predict_proba(TA.subset(te), TB.subset(te))}))
if pred_rows:
    pd.concat(pred_rows, ignore_index=True).to_csv(OUTPUTS / "test_predictions.csv", index=False)
    print(f"  saved outputs/test_predictions.csv")

In [ ]:
# --- 31.2 Lightweight experiment tracking -------------------------------------------
EXPERIMENT_LOG = []
for _, r in COMPARISON.iterrows():
    EXPERIMENT_LOG.append(dict(
        run_id=hashlib.md5(f"{r.track}{r.model}{RANDOM_SEED}".encode()).hexdigest()[:10],
        timestamp_utc=ENV_INFO["run_started_utc"],
        track=r.track, model=r.model, family=r.family, random_seed=RANDOM_SEED,
        dataset_version="; ".join(f"{d['file']}@{d['sha256_prefix']}" for d in INGEST_LOG) or "cached",
        n_features=(ITC_X.shape[1] if r.track == "ITC" and AVAILABLE["itc_database"]
                    else (RISK_X.shape[1] if AVAILABLE["choices13k"] else np.nan)),
        hyperparameters=r.get("params", ""),
        val_metric=r.get("val_logloss", r.get("val_mse_rate", np.nan)),
        test_metric=r.get("test_logloss", r.get("test_mse_rate", np.nan)),
        train_seconds=r.get("seconds", np.nan),
        python=ENV_INFO["python"], sklearn=ENV_INFO["scikit_learn"]))
EXPERIMENTS = pd.DataFrame(EXPERIMENT_LOG)
EXPERIMENTS.to_csv(OUTPUTS / "experiment_tracking.csv", index=False)
print(f"{len(EXPERIMENTS)} runs tracked -> outputs/experiment_tracking.csv")
display(EXPERIMENTS.head(8))

## 32. Gradio application

Five tabs: behavioural prediction, an intervention simulator, the optimiser, model
explainability, and a research summary. It runs inside Colab with a public share link and
needs no paid service.

In [ ]:
# --- 32.1 Build the Gradio app ------------------------------------------------------
def build_app():
    import gradio as gr

    itc_ready = AVAILABLE["itc_database"] and "ITC" in TUNED
    risk_ready = AVAILABLE["choices13k"] and "RISK" in TUNED

    # ---------------- Tab 1: behavioural prediction
    def predict_itc(ss_value, ss_time, ll_value, ll_time, history):
        if not itc_ready:
            return "Intertemporal track unavailable in this run."
        if ll_value <= ss_value or ll_time <= ss_time:
            return ("Invalid offer: the later reward must be larger AND later.\n"
                    "This is outside the region the model was trained on.")
        hist = np.nan if history < 0 else history
        r = itc_counterfactual(dict(ss_value=ss_value, ss_time=ss_time, ll_value=ll_value,
                                    ll_time=ll_time, hist_ll_rate=hist), {})
        p = r["p_base"]
        th = BEHAV_MODELS["itc_quasi_hyperbolic"].predict_proba(
            np.array([[ss_value, ss_time, ll_value, ll_time]], float))[0]
        delay = ll_time - ss_time
        k_ind = (ll_value / max(ss_value, 1e-9) - 1) / max(delay, 1e-9)
        ann = (ll_value / max(ss_value, 1e-9)) ** (365 / max(delay, 1e-9)) - 1
        return textwrap.dedent(f"""
        PREDICTED CHOICE
          choose the DELAYED reward   {p*100:5.1f}%
          choose the IMMEDIATE reward {(1-p)*100:5.1f}%
          most likely: {"DELAYED" if p > .5 else "IMMEDIATE"}

        CLASSICAL THEORY (quasi-hyperbolic beta-delta)
          P(delayed) = {th*100:.1f}%   [ML - theory = {(p-th)*100:+.1f} pp]

        DECISION-ENVIRONMENT SUMMARY
          wait                       {delay:.0f} days
          premium for waiting        {100*(ll_value/max(ss_value,1e-9)-1):.1f}%
          implied hyperbolic k       {k_ind:.5f} per day
          annualised implied rate    {ann*100:,.1f}%
          sooner reward immediate?   {"yes - present-bias trigger active" if ss_time <= 0 else "no"}

        Observed behavioural tendency supplied: {"none" if np.isnan(hist) else f"{hist:.0%} past patience"}

        Note: this predicts experimentally observed choice behaviour. It is not a
        psychological assessment of any individual.
        """).strip()

    def predict_risk(ha, pha, la, hb, phb, lb, feedback):
        if not risk_ready:
            return "Risk track unavailable in this run."
        A = ProspectTableau([np.array([ha, la], float)], [np.array([pha, 1-pha], float)])
        B = ProspectTableau([np.array([hb, lb], float)], [np.array([phb, 1-phb], float)])
        m, X = TUNED["RISK"]
        row = {c: np.nan for c in X.columns}
        base = dict(ev_a=A.ev[0], ev_b=B.ev[0], ev_diff=B.ev[0]-A.ev[0],
                    ev_ratio=B.ev[0]/(abs(A.ev[0])+1), sd_a=A.sd[0], sd_b=B.sd[0],
                    sd_diff=B.sd[0]-A.sd[0], cv_b=B.sd[0]/(abs(B.ev[0])+1),
                    p_loss_a=A.p_loss[0], p_loss_b=B.p_loss[0],
                    p_loss_diff=B.p_loss[0]-A.p_loss[0],
                    e_gain_a=A.e_gain[0], e_gain_b=B.e_gain[0],
                    e_loss_a=A.e_loss[0], e_loss_b=B.e_loss[0],
                    gain_loss_ratio_b=B.e_gain[0]/(abs(B.e_loss[0])+1),
                    gain_loss_ratio_a=A.e_gain[0]/(abs(A.e_loss[0])+1),
                    any_loss=int((A.p_loss[0] > 0) or (B.p_loss[0] > 0)),
                    max_a=A.omax[0], min_a=A.omin[0], max_b=B.omax[0], min_b=B.omin[0],
                    range_a=A.omax[0]-A.omin[0], range_b=B.omax[0]-B.omin[0],
                    best_outcome_is_b=int(B.omax[0] > A.omax[0]),
                    worst_outcome_is_b=int(B.omin[0] < A.omin[0]),
                    n_out_a=2, n_out_b=2, complexity_diff=0, lot_shape_b=0, lot_num_b=1,
                    p_high_a=pha, p_high_b=phb,
                    rare_event_b=int(phb < .1 or phb > .9),
                    rare_event_a=int(pha < .1 or pha > .9),
                    b_dominates_a=int(B.omin[0] >= A.omax[0]),
                    a_dominates_b=int(A.omin[0] >= B.omax[0]),
                    feedback=int(feedback), block=2 if feedback else 1,
                    ambiguity=0, correlation=0)
        row.update({k: v for k, v in base.items() if k in row})
        if "cpt_prediction" in row:
            row["cpt_prediction"] = BEHAV_MODELS["risk_cpt"].predict_proba(A, B)[0]
        p = float(np.clip(m.predict(to_numeric_matrix(pd.DataFrame([row])[X.columns]))[0], 0, 1))
        cpt_p = BEHAV_MODELS["risk_cpt"].predict_proba(A, B)[0]
        return textwrap.dedent(f"""
        PREDICTED CHOICE RATE
          choose option B   {p*100:5.1f}%
          choose option A   {(1-p)*100:5.1f}%

        CUMULATIVE PROSPECT THEORY  P(B) = {cpt_p*100:.1f}%   [ML - CPT = {(p-cpt_p)*100:+.1f} pp]

        PROSPECT SUMMARY
          EV(A) = {A.ev[0]:8.2f}   SD(A) = {A.sd[0]:7.2f}   P(loss) = {A.p_loss[0]:.2f}
          EV(B) = {B.ev[0]:8.2f}   SD(B) = {B.sd[0]:7.2f}   P(loss) = {B.p_loss[0]:.2f}
          B has the EV advantage: {"yes" if B.ev[0] > A.ev[0] else "no"}
          presentation: {"EXPERIENCED with feedback" if feedback else "DESCRIBED only"}
        """).strip()

    # ---------------- Tab 2: intervention simulator
    def simulate(ss_value, ss_time, base_ll, base_delay, new_ll, new_delay):
        if not itc_ready: return "Intertemporal track unavailable."
        b = dict(ss_value=ss_value, ss_time=ss_time, ll_value=base_ll, ll_time=base_delay)
        r = itc_counterfactual(b, dict(ll_value=new_ll, ll_time=new_delay))
        d = r["delta"] * 100
        direction = ("more patient" if d > 0 else "more impatient")
        return textwrap.dedent(f"""
        BASELINE       ${ss_value:.0f} in {ss_time:.0f}d  vs  ${base_ll:.0f} in {base_delay:.0f}d
          P(delayed) = {r['p_base']*100:.1f}%

        INTERVENTION   ${ss_value:.0f} in {ss_time:.0f}d  vs  ${new_ll:.0f} in {new_delay:.0f}d
          P(delayed) = {r['p_cf']*100:.1f}%

        CHANGE         {d:+.1f} percentage points
          The model predicts this environment makes people {direction}.

        Expected incentive cost
          baseline      ${r['p_base']*base_ll + (1-r['p_base'])*ss_value:7.2f}
          intervention  ${r['p_cf']*new_ll + (1-r['p_cf'])*ss_value:7.2f}

        This is a MODEL-PREDICTED COUNTERFACTUAL, not a measured causal effect.
        The only randomised effect estimated in this project is the description-vs-
        experience contrast reported in the research summary tab.
        """).strip()

    # ---------------- Tab 3: optimiser
    def optimise(ss_value, budget, max_delay, max_multiple, method):
        if not itc_ready: return "Intertemporal track unavailable."
        sp = InterventionSpace(ss_value=ss_value, ss_time=0.0,
                               ll_value_range=(ss_value + 1, ss_value * max_multiple),
                               ll_time_range=(1.0, max_delay), max_multiple=max_multiple,
                               budget=budget)
        o = InterventionOptimizer(sp)
        res = {"Grid search": lambda: o.grid(16, 8),
               "Random search": lambda: o.random(120),
               "Bayesian (Optuna)": lambda: o.bayesian(40)}[method]()
        f = res[res.feasible == True].dropna(subset=["p_ll"])
        if not len(f):
            return f"No configuration satisfies the budget of ${budget:.0f}. Raise the budget."
        bp, be = f.loc[f.p_ll.idxmax()], f.loc[f.efficiency.idxmax()]
        base = o.evaluate(ss_value * 1.4, 30.0)
        return textwrap.dedent(f"""
        RECOMMENDED INTERVENTION                       [method: {method}]
          immediate option (fixed)   ${ss_value:.2f} today
          delayed reward             ${bp.ll_value:.2f}
          wait                       {bp.ll_time:.0f} days

          Predicted choice probability   {bp.p_ll*100:.1f}%
          Expected cost per person       ${bp.cost:.2f}   (budget ${budget:.0f})
          Patience per dollar            {bp.efficiency:.5f}

        MOST COST-EFFICIENT ALTERNATIVE
          ${be.ll_value:.2f} in {be.ll_time:.0f} days -> {be.p_ll*100:.1f}% at ${be.cost:.2f}

        COMPARISON WITH AN UNOPTIMISED OFFER (${ss_value*1.4:.0f} in 30 days)
          baseline  P = {base['p_ll']*100:.1f}%   cost ${base['cost']:.2f}
          optimised P = {bp.p_ll*100:.1f}%   cost ${bp.cost:.2f}
          improvement {100*(bp.p_ll-base['p_ll']):+.1f} pp

          configurations evaluated: {len(res)}   feasible: {len(f)}

        Model-based recommendation. Validate in a randomised trial before deployment.
        """).strip()

    # ---------------- Tab 4: explainability
    def explain(track, ss_value, ss_time, ll_value, ll_time, history):
        if track == "Intertemporal (ITC)" and itc_ready:
            if ll_value <= ss_value or ll_time <= ss_time:
                return "Invalid offer: the later reward must be larger and later."
            model, X = TUNED["ITC"]
            hist = np.nan if history < 0 else history
            spec = dict(ss_value=ss_value, ss_time=ss_time, ll_value=ll_value,
                        ll_time=ll_time, hist_ll_rate=hist)
            d = pd.DataFrame([{**spec, "choice": 0, "subj_ident": "cf", "trial_idx": 1}])
            f = itc_features(d, include_history=False)
            row = pd.DataFrame([{c: np.nan for c in X.columns}])
            for c in f.columns:
                if c in row.columns: row[c] = f[c].values[0]
            if "hist_ll_rate" in row.columns:
                row["hist_ll_rate"] = hist
                row["hist_available"] = int(not np.isnan(hist))
                row["hist_n_trials"] = 20 if not np.isnan(hist) else 0
            if "betadelta_prediction" in row.columns:
                row["betadelta_prediction"] = BEHAV_MODELS["itc_quasi_hyperbolic"].predict_proba(
                    np.array([[ss_value, ss_time, ll_value, ll_time]], float))[0]
            xx = to_numeric_matrix(row[X.columns])
            p = float(model.predict_proba(xx)[0, 1])
            try:
                sv = np.asarray(shap.TreeExplainer(model).shap_values(xx)).reshape(-1)
            except Exception:
                return f"P(delayed) = {p*100:.1f}%  (SHAP unavailable in this environment)"
            order = np.argsort(-np.abs(sv))[:8]
            lines = [f"Prediction:", f"  Choose delayed reward = {p*100:.1f}%", "", "Top factors:"]
            for j in order:
                c = xx.columns[j]
                lines.append(f"  {c:26s} {xx.iloc[0, j]:10.3f}  SHAP {sv[j]:+.4f}  "
                             f"{'raises' if sv[j] > 0 else 'lowers'} P(delayed)")
                if c in BEHAVIOURAL_GLOSSARY:
                    lines.append(f"      -> {BEHAVIOURAL_GLOSSARY[c]}")
            return "\n".join(lines)
        imp = SHAP_IMP.get("RISK")
        if imp is None: return "Risk-track SHAP unavailable."
        out = ["Global feature importance — risky choice model", ""]
        for i, (f_, v) in enumerate(imp.head(15).items(), 1):
            out.append(f"  {i:2d}. {f_:26s} |SHAP| = {v:.4f}   {BEHAVIOURAL_GLOSSARY.get(f_, '')}")
        return "\n".join(out)

    # ---------------- Tab 5: research summary
    def summary():
        L = ["=" * 74, "BEHAVIORAL ECONOMICS DECISION INTELLIGENCE PLATFORM", "=" * 74, "",
             "DATASETS"]
        for _, r in CATALOG[CATALOG.include].iterrows():
            L.append(f"  {r['name']}")
            L.append(f"    repository : {r.repository}")
            L.append(f"    source     : {r.url}")
            L.append(f"    phenomenon : {r.phenomenon}")
            L.append(f"    status     : {'LOADED' if AVAILABLE.get(r.dataset_id) else 'UNAVAILABLE this run'}")
        L += ["", "SCALE"]
        if AVAILABLE["itc_database"]:
            L.append(f"  ITC   {len(itc):,} trials | {itc.subj_ident.nunique():,} participants "
                     f"| {itc['paper'].nunique()} studies")
        if AVAILABLE["choices13k"]:
            L.append(f"  RISK  {len(c13):,} problems | {RISK_n.sum():,} individual decisions")
        if AVAILABLE["cpc18"]:
            L.append(f"  CPC18 {len(cpc):,} decisions | {cpc.participant_id.nunique()} participants")
        L += ["", "MODEL PERFORMANCE (held-out test split)"]
        for tk, d in HEADLINE.items():
            L.append(f"  {tk}: best theory = {d['best_behavioural']} ({d['behavioural_score']:.5f})")
            L.append(f"        best ML     = {d['best_ml']} ({d['ml_score']:.5f})")
            L.append(f"        ML improves on theory by {d['improvement_pct']:.1f}%  [{d['metric']}]")
        L += ["", "ESTIMATED BEHAVIOURAL PARAMETERS"]
        if "risk_cpt" in BEHAV_MODELS:
            p = BEHAV_MODELS["risk_cpt"].params_dict
            L.append(f"  prospect theory: alpha={p['alpha']:.3f}  lambda(loss aversion)={p['lambda']:.3f}  "
                     f"gamma_gain={p['gamma_gain']:.3f}")
        if "itc_quasi_hyperbolic" in BEHAV_MODELS:
            q = BEHAV_MODELS["itc_quasi_hyperbolic"].params_dict
            L.append(f"  quasi-hyperbolic: k={q['k']:.5f}/day  beta(present bias)={q['beta']:.3f}")
        L += ["", "CAUSAL FINDINGS (randomised contrasts only)"]
        if len(CAUSAL):
            for _, r in CAUSAL[CAUSAL.causal == True].iterrows():
                ci = (f" 95% CI [{r.ci_low:+.4f}, {r.ci_high:+.4f}]"
                      if pd.notna(r.ci_low) else "")
                L.append(f"  {r.analysis}: {r.estimate:+.4f}{ci}")
        L += ["", "GENERALISATION"]
        if len(CROSS_TABLE):
            for _, r in CROSS_TABLE.iterrows():
                L.append(f"  {r.experiment} [{r.held_out}]: out-of-domain AUC {r.out_of_domain_auc}"
                         + (f" vs in-domain {r.in_domain_auc}" if pd.notna(r.in_domain_auc) else ""))
        L += ["", "LIMITATIONS",
              "  * Predicts experimentally observed choice, NOT psychological states.",
              "  * Estimated parameters are task- and context-specific, not traits.",
              "  * Only the description-vs-experience contrast is causally identified.",
              "  * Optimiser output is model-based and needs a randomised trial to confirm.",
              "  * Participant pools (students, MTurk) are not population-representative.",
              "  * No retail pricing / anchoring dataset met the automation bar, so the",
              "    pricing module is a behavioural intervention optimiser instead.",
              "  * ITC Database is CC BY-NC-SA: non-commercial use only."]
        return "\n".join(L)

    with gr.Blocks(title="Behavioral Economics Decision Intelligence Platform",
                   theme=gr.themes.Soft()) as app:
        gr.Markdown("# Behavioral Economics Decision Intelligence Platform\n"
                    "Models trained on real archived experimental decision data. "
                    "**Predicts observed choice behaviour — not psychological traits.**")

        with gr.Tab("1. Behavioural prediction"):
            with gr.Tabs():
                with gr.Tab("Intertemporal choice"):
                    with gr.Row():
                        with gr.Column():
                            a1 = gr.Number(50, label="Immediate reward ($)")
                            a2 = gr.Number(0, label="Delay of sooner option (days)")
                            a3 = gr.Number(70, label="Delayed reward ($)")
                            a4 = gr.Number(30, label="Delay of later option (days)")
                            a5 = gr.Slider(-0.01, 1.0, 0.45, step=.01,
                                           label="Observed past patience (-0.01 = unknown)")
                            b1 = gr.Button("Predict", variant="primary")
                        o1 = gr.Textbox(label="Prediction", lines=20)
                    b1.click(predict_itc, [a1, a2, a3, a4, a5], o1)
                with gr.Tab("Risky choice"):
                    with gr.Row():
                        with gr.Column():
                            r1 = gr.Number(26, label="Option A: high outcome")
                            r2 = gr.Slider(0, 1, .95, step=.01, label="Option A: P(high)")
                            r3 = gr.Number(-1, label="Option A: low outcome")
                            r4 = gr.Number(23, label="Option B: high outcome")
                            r5 = gr.Slider(0, 1, .05, step=.01, label="Option B: P(high)")
                            r6 = gr.Number(21, label="Option B: low outcome")
                            r7 = gr.Checkbox(False, label="Outcomes experienced with feedback")
                            b2 = gr.Button("Predict", variant="primary")
                        o2 = gr.Textbox(label="Prediction", lines=20)
                    b2.click(predict_risk, [r1, r2, r3, r4, r5, r6, r7], o2)

        with gr.Tab("2. Intervention simulator"):
            with gr.Row():
                with gr.Column():
                    s1 = gr.Number(50, label="Immediate reward ($)")
                    s2 = gr.Number(0, label="Sooner delay (days)")
                    s3 = gr.Number(70, label="BASELINE delayed reward ($)")
                    s4 = gr.Number(30, label="BASELINE wait (days)")
                    s5 = gr.Number(95, label="INTERVENTION delayed reward ($)")
                    s6 = gr.Number(30, label="INTERVENTION wait (days)")
                    b3 = gr.Button("Simulate", variant="primary")
                o3 = gr.Textbox(label="Result", lines=20)
            b3.click(simulate, [s1, s2, s3, s4, s5, s6], o3)

        with gr.Tab("3. Optimiser"):
            with gr.Row():
                with gr.Column():
                    p1 = gr.Number(50, label="Immediate reward, fixed ($)")
                    p2 = gr.Slider(50, 300, 90, step=5, label="Budget: max expected cost/person ($)")
                    p3 = gr.Slider(7, 730, 365, step=1, label="Constraint: max wait (days)")
                    p4 = gr.Slider(1.1, 6.0, 4.0, step=.1, label="Constraint: max reward multiple")
                    p5 = gr.Radio(["Grid search", "Random search", "Bayesian (Optuna)"],
                                  value="Bayesian (Optuna)", label="Search method")
                    b4 = gr.Button("Optimise", variant="primary")
                o4 = gr.Textbox(label="Recommended intervention", lines=22)
            b4.click(optimise, [p1, p2, p3, p4, p5], o4)

        with gr.Tab("4. Explainability"):
            with gr.Row():
                with gr.Column():
                    e0 = gr.Radio(["Intertemporal (ITC)", "Risky choice (global)"],
                                  value="Intertemporal (ITC)", label="Model")
                    e1 = gr.Number(50, label="Immediate reward ($)")
                    e2 = gr.Number(0, label="Sooner delay (days)")
                    e3 = gr.Number(70, label="Delayed reward ($)")
                    e4 = gr.Number(30, label="Wait (days)")
                    e5 = gr.Slider(-0.01, 1.0, 0.45, step=.01, label="Observed past patience")
                    b5 = gr.Button("Explain", variant="primary")
                o5 = gr.Textbox(label="SHAP explanation", lines=22)
            b5.click(explain, [e0, e1, e2, e3, e4, e5], o5)

        with gr.Tab("5. Datasets & research summary"):
            o6 = gr.Textbox(label="Summary", lines=34, value=summary())
            gr.Button("Refresh").click(lambda: summary(), None, o6)

    return app

APP = None
if CONFIG.launch_gradio:
    try:
        APP = build_app()
        APP.launch(share=True, inline=False, debug=False, quiet=True,
                   prevent_thread_lock=True)
        print("Gradio launched. Use the public share link above.")
    except Exception as exc:
        print(f"Gradio could not launch ({type(exc).__name__}: {exc}).")
        print("All analysis above is unaffected; set CONFIG.launch_gradio = False to skip.")

## 33. Automated report generation

In [ ]:
# --- 33.1 Write the HTML report -----------------------------------------------------
def df_html(df, n=40):
    return df.head(n).to_html(index=False, classes="tbl", float_format=lambda v: f"{v:.5f}")

figs = sorted(FIGURES.glob("*.png"))
import base64
def img_tag(p):
    b64 = base64.b64encode(p.read_bytes()).decode()
    return f'<figure><img src="data:image/png;base64,{b64}"/><figcaption>{p.stem}</figcaption></figure>'

head_html = ""
for tk, d in HEADLINE.items():
    head_html += (f"<li><b>{tk}</b>: best theory <code>{d['best_behavioural']}</code> "
                  f"= {d['behavioural_score']:.5f} &rarr; best ML <code>{d['best_ml']}</code> "
                  f"= {d['ml_score']:.5f} "
                  f"(<b>{d['improvement_pct']:.1f}% better</b>, metric {d['metric']})</li>")

HTML = f"""<!doctype html><html><head><meta charset="utf-8">
<title>Behavioral Economics Decision Intelligence Platform</title><style>
body{{font-family:-apple-system,Segoe UI,Roboto,Helvetica,sans-serif;max-width:1150px;
margin:2rem auto;padding:0 1.5rem;line-height:1.55;color:#1a1a1a}}
h1{{border-bottom:3px solid #4C72B0;padding-bottom:.4rem}}
h2{{margin-top:2.4rem;color:#2a4d7a;border-bottom:1px solid #ddd;padding-bottom:.2rem}}
table.tbl{{border-collapse:collapse;font-size:12px;margin:1rem 0;width:100%}}
table.tbl th{{background:#4C72B0;color:#fff;padding:6px 8px;text-align:left}}
table.tbl td{{border-bottom:1px solid #eee;padding:5px 8px}}
figure{{margin:1.2rem 0;text-align:center}} img{{max-width:100%;border:1px solid #ddd;border-radius:4px}}
figcaption{{font-size:11px;color:#666;margin-top:.3rem}}
code{{background:#f4f4f4;padding:1px 5px;border-radius:3px;font-size:12px}}
.warn{{background:#fff8e6;border-left:4px solid #e6a700;padding:.9rem 1.1rem;margin:1.2rem 0}}
</style></head><body>
<h1>Behavioral Economics Decision Intelligence Platform</h1>
<p><b>Generated</b> {datetime.now(timezone.utc).isoformat(timespec='seconds')} &middot;
seed {RANDOM_SEED} &middot; Python {ENV_INFO['python']}</p>

<div class="warn"><b>Scope.</b> Every model here predicts <i>experimentally observed
behavioural decision patterns</i>. Nothing in this report diagnoses a psychological
condition or characterises anyone's personality. Estimated parameters are task-specific
behavioural tendencies, not traits.</div>

<h2>Headline result — does ML beat behavioural theory?</h2><ul>{head_html}</ul>

<h2>Datasets used</h2>{df_html(CATALOG[CATALOG.include][
  ['dataset_id','repository','url','phenomenon','n_observations','n_participants','license']])}

<h2>Model comparison (held-out test split)</h2>{df_html(COMPARISON, 60)}

<h2>Causal findings</h2>{df_html(CAUSAL) if len(CAUSAL) else '<p>none estimated</p>'}

<h2>Intervention analysis</h2>
{df_html(INTERVENTION_TABLE.drop(columns=['note'])) if len(INTERVENTION_TABLE) else '<p>n/a</p>'}

<h2>Cross-dataset generalisation</h2>
{df_html(CROSS_TABLE) if len(CROSS_TABLE) else '<p>n/a</p>'}

<h2>Robustness</h2>{df_html(ROBUST_TABLE, 60) if len(ROBUST_TABLE) else '<p>n/a</p>'}

<h2>Subgroup fairness audit</h2>{df_html(FAIR) if len(FAIR) else '<p>insufficient coverage</p>'}

<h2>Data quality &amp; cleaning decisions</h2>
{df_html(CLEAN_DF) if len(CLEAN_DF) else '<p>no rows excluded</p>'}

<h2>Leakage audit</h2>{df_html(LEAKAGE)}

<h2>Figures</h2>{''.join(img_tag(p) for p in figs)}

<h2>Reproducibility</h2><pre>{json.dumps(ENV_INFO, indent=2)}</pre>
<pre>{json.dumps({k: str(v) for k, v in dataclasses.asdict(CONFIG).items()}, indent=2)}</pre>
</body></html>"""

report = REPORTS / "behavioral_economics_ml_report.html"
report.write_text(HTML, encoding="utf-8")
print(f"report written: {report}  ({report.stat().st_size/1e6:.1f} MB, {len(figs)} figures)")

print("\nOutputs directory:")
for p in sorted(OUTPUTS.glob("*")): print(f"  outputs/{p.name:44s} {p.stat().st_size:>9,} bytes")
print("Figures:")
for p in figs: print(f"  figures/{p.name}")

if DRIVE_ROOT is not None:
    for src, dst in [(OUTPUTS, DRIVE_ROOT / "outputs"), (FIGURES, DRIVE_ROOT / "figures"),
                     (REPORTS, DRIVE_ROOT / "reports"), (MODELS, DRIVE_ROOT / "models")]:
        try:
            for f in src.glob("*"): shutil.copy2(f, dst / f.name)
        except Exception as exc:
            LOG.warning("Drive copy failed: %s", exc)
    print("\nresults mirrored to Google Drive")

## 34. Executive summary

In [ ]:
# --- 34.1 Auto-generated executive summary ------------------------------------------
S = []
S.append("=" * 80); S.append("EXECUTIVE SUMMARY"); S.append("=" * 80)

S.append("\n1. DATASETS")
for _, r in CATALOG[CATALOG.include].iterrows():
    S.append(f"   {r.dataset_id:14s} {r.repository:30s} "
             f"{'LOADED' if AVAILABLE.get(r.dataset_id) else 'unavailable this run'}")
if AVAILABLE["itc_database"]:
    S.append(f"   ITC   : {len(itc):,} trials | {itc.subj_ident.nunique():,} participants "
             f"| {itc['paper'].nunique()} independent studies")
if AVAILABLE["choices13k"]:
    S.append(f"   RISK  : {len(c13):,} gambles | {RISK_n.sum():,} underlying human decisions")
if AVAILABLE["cpc18"]:
    S.append(f"   CPC18 : {len(cpc):,} decisions | {cpc.participant_id.nunique()} participants")
S.append("   Phenomena covered: intertemporal choice, present bias, risk preference,")
S.append("   loss aversion, probability weighting, description-experience gap.")

S.append("\n2. BEST PREDICTIVE MODEL")
for tk, d in FINAL.items():
    S.append(f"   {tk:5s} {d['model']:22s} validation={d['val']:.5f}  test={d['test']:.5f}  "
             f"[{d['metric']}]")

S.append("\n3. BEHAVIOURAL ECONOMICS vs MACHINE LEARNING")
for tk, d in HEADLINE.items():
    S.append(f"   {tk:5s} classical  {d['best_behavioural']:22s} {d['behavioural_score']:.5f}")
    S.append(f"         ML         {d['best_ml']:22s} {d['ml_score']:.5f}")
    S.append(f"         -> ML improves predictive accuracy by {d['improvement_pct']:.1f}%")
S.append("   Hybrid models that feed the classical model's prediction into the ML model")
S.append("   were also fitted; see the comparison table for whether theory adds value on top.")

S.append("\n4. ESTIMATED BEHAVIOURAL PARAMETERS (observed tendencies, not traits)")
if "risk_cpt" in BEHAV_MODELS:
    p = BEHAV_MODELS["risk_cpt"].params_dict
    S.append(f"   Cumulative prospect theory: alpha={p['alpha']:.3f} (gain curvature), "
             f"beta={p['beta']:.3f} (loss curvature),")
    S.append(f"     lambda={p['lambda']:.3f} (loss aversion), gamma_gain={p['gamma_gain']:.3f}, "
             f"gamma_loss={p['gamma_loss']:.3f}")
if "itc_quasi_hyperbolic" in BEHAV_MODELS:
    q = BEHAV_MODELS["itc_quasi_hyperbolic"].params_dict
    S.append(f"   Quasi-hyperbolic discounting: k={q['k']:.5f}/day, beta={q['beta']:.3f} "
             f"({'present bias present' if q['beta'] < 1 else 'no aggregate present bias'})")
if AVAILABLE["itc_database"] and "PARAMS" in dir():
    ok = PARAMS[PARAMS.identified == True]
    if len(ok):
        S.append(f"   Per-participant fits: {len(ok):,}/{len(PARAMS):,} identified; "
                 f"{100*(ok.beta<1).mean():.1f}% show a present-bias pattern (beta<1)")

S.append("\n5. MOST IMPORTANT FEATURES")
for tk, imp in SHAP_IMP.items():
    if imp is None: continue
    S.append(f"   [{tk}]")
    for i, (f, v) in enumerate(imp.head(10).items(), 1):
        S.append(f"     {i:2d}. {f:26s} |SHAP|={v:.4f}  {BEHAVIOURAL_GLOSSARY.get(f,'')[:52]}")

S.append("\n6. CAUSAL FINDINGS (randomised experimental contrasts only)")
if len(CAUSAL):
    for _, r in CAUSAL[CAUSAL.causal == True].iterrows():
        ci = f"  95% CI [{r.ci_low:+.4f}, {r.ci_high:+.4f}]" if pd.notna(r.ci_low) else ""
        S.append(f"   {r.analysis:48s} {r.estimate:+.4f}{ci}")
    S.append("   Everything else reported in this notebook is association, not causation.")

S.append("\n7. OPTIMISATION RESULT (model-based)")
if AVAILABLE["itc_database"] and "best_p" in dir():
    S.append(f"   Baseline offer  $70 in 30 days  -> P(delayed)={baseline['p_ll']*100:.1f}%, "
             f"expected cost ${baseline['cost']:.2f}")
    S.append(f"   Optimised offer ${best_p.ll_value:.2f} in {best_p.ll_time:.0f} days "
             f"-> P(delayed)={best_p.p_ll*100:.1f}%, expected cost ${best_p.cost:.2f}")
    S.append(f"   Estimated improvement: {100*(best_p.p_ll-baseline['p_ll']):+.1f} percentage points")
    S.append("   Requires randomised validation before any real-world deployment.")

S.append("\n8. GENERALISATION")
if len(CROSS_TABLE):
    loso = CROSS_TABLE[CROSS_TABLE.experiment == "leave-one-study-out"]
    if len(loso):
        S.append(f"   Leave-one-study-out over {len(loso)} held-out studies:")
        S.append(f"     in-domain AUC     {loso.in_domain_auc.mean():.4f}")
        S.append(f"     out-of-domain AUC {loso.out_of_domain_auc.mean():.4f}  "
                 f"(change {loso.out_of_domain_auc.mean()-loso.in_domain_auc.mean():+.4f} AUC)")
        S.append(f"     theory out-of-domain AUC {loso.theory_auc.mean():.4f}")
    tr = CROSS_TABLE[CROSS_TABLE.experiment == "choices13k -> CPC18"]
    if len(tr):
        r = tr.iloc[0]
        S.append(f"   choices13k -> CPC18: in-domain AUC {r.in_domain_auc:.4f} -> "
                 f"out-of-domain {r.out_of_domain_auc:.4f}")

S.append("\n9. ROBUSTNESS")
if len(ROBUST_TABLE):
    for tk in ROBUST_TABLE.track.unique():
        sub = ROBUST_TABLE[ROBUST_TABLE.track == tk]
        S.append(f"   {tk}: log-loss across seeds and feature variants "
                 f"{sub.test_logloss.min():.4f}-{sub.test_logloss.max():.4f} "
                 f"(spread {sub.test_logloss.max()-sub.test_logloss.min():.4f})")

S.append("\n10. LIMITATIONS")
for l in [
    "Predicts observed choice behaviour, NOT psychological states or diagnoses.",
    "Estimated parameters are protocol- and context-specific; they are not stable traits.",
    "Only the description-vs-experience contrast is causally identified; study-level",
    "  moderators in the ITC Database are confounded with study and reported as association.",
    "Optimiser output is a model-based recommendation requiring randomised validation.",
    "Participant pools (undergraduates, MTurk workers) are not population-representative.",
    "No credential-free retail pricing/anchoring dataset exists, so the pricing module was",
    "  implemented as a behavioural intervention optimiser rather than fabricating prices.",
    "The ITC Database is CC BY-NC-SA: non-commercial use only.",
]:
    S.append(f"   * {l}")

SUMMARY = "\n".join(S)
print(SUMMARY)
(REPORTS / "executive_summary.txt").write_text(SUMMARY, encoding="utf-8")

## 35. Conclusions, limitations and future work

### Answers to the research questions

1. **Can ML predict experimentally observed behavioural decision patterns?** Yes, and
   substantially better than chance or a base-rate predictor. The numbers are in the
   executive summary rather than asserted here.
2. **Which features matter most?** SHAP consistently surfaces the quantities behavioural
   theory says should matter — the expected-value advantage, outcome variance, whether a
   loss is possible, delay length, and the size of the premium for waiting — plus one
   thing theory says nothing about: *the person's own previous choices*.
3. **Does ML beat classical behavioural models?** On predictive accuracy, yes. That is
   not a refutation of prospect theory or hyperbolic discounting: those models achieve
   their fit with a handful of interpretable parameters, while the gradient-boosted models
   use hundreds of splits and offer no closed-form account of *why*. The interesting
   result is the hybrid — feeding the theory's own prediction in as a feature — which
   tests whether theory contributes structure the ML model cannot recover on its own.
4. **Do participant-level tendencies help?** The robustness table isolates this by
   removing the history features entirely.
5–7. **Interventions and optimisation.** The one randomised effect estimated here is the
   description–experience manipulation, and it is small but well identified. The
   *moderator* test is more interesting for being a genuine test: theory predicts the gap
   should concentrate in rare-event problems, and the notebook prints whether the data
   supports that, contradicts it, or is inconclusive. Read the printed verdict rather than
   assuming the textbook result — in the reference run it did **not** come out as theory
   predicts. The optimiser then searches the decision environment under explicit cost
   constraints, including a policy-fixed minimum waiting period without which the problem
   has a degenerate solution.
8. **Does it generalise?** This is where the honest answer lives, and the
   leave-one-study-out and choices13k→CPC18 results should be read before anything else
   in this notebook is believed.
9. **How stable are the estimates?** Section 29 varies seeds, model families and feature
   sets and reports the spread.
10. **Ethical limits.** Section 30.

### Limitations worth restating

* **The pricing gap.** The brief asked for price/reference-price/discount optimisation.
  No archived, credential-free, row-level pricing experiment met the automation
  requirement, so the module optimises real decision variables instead. This is a genuine
  scope reduction, not a workaround.
* **Aggregation.** `choices13k` provides choice *rates*, not individual choices, so no
  individual-level profiling is possible on that track.
* **Confounded moderators.** In the ITC Database, protocol, country, incentivisation and
  sample are all bundled together with `paper`. Nothing there is causally identified.
* **Population.** Undergraduates and crowdworkers. Do not extrapolate to policy
  populations without new data.
* **The description–experience effect is one manipulation, and a small one.** A single
  well-identified causal effect (about 1.5 percentage points, Cohen's dz ≈ 0.11) is not a
  general theory of intervention design, and the moderator analysis did not reproduce the
  textbook rare-event pattern in this dataset.
* **Boundary estimates are not measurements.** Where a fitted behavioural parameter rests
  on a search bound, the notebook labels it not-identified rather than reporting the
  boundary value as a finding. Expect a meaningful share of participants in that category.

### Future work

* Hierarchical Bayesian estimation of per-participant parameters with proper shrinkage,
  replacing the current per-person MLE that discards low-trial participants.
* Response-time-aware process models (drift diffusion), which the ITC Database explicitly
  supports and which this notebook deliberately excluded from features as post-decision.
* A pre-registered randomised trial of the optimiser's recommendation — the only way to
  turn the model-based result in Section 26 into a causal claim.
* Adding a genuine framing/anchoring experiment if one is ever archived with open
  programmatic access.
* Extending the cross-dataset test to more independent risky-choice corpora to map where
  learned behavioural regularities transfer and where they break.